## 0. Cài thư viện

In [ ]:
%pip install -q \
    pandas==2.2.2 numpy==1.26.4 tqdm==4.66.4 pydantic==2.7.4 python-dotenv==1.0.1 \
    neo4j==5.19.0 qdrant-client==1.9.1 sentence-transformers==3.0.1 torch==2.3.1 transformers==4.44.2 \
    langchain==0.2.6 langchain-openai==0.1.13 langchain-anthropic==0.1.20 \
    python-igraph==0.11.6 FlagEmbedding==1.2.11
print("✅ Dependencies installed")

## 1. Imports, config và đường dẫn

In [4]:
from __future__ import annotations
import os, re, json, ast, math, hashlib, unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Literal
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from dotenv import load_dotenv
from pydantic import BaseModel, Field, ValidationError

load_dotenv(dotenv_path=Path.cwd() / ".env")

REPO_ROOT = Path.cwd()
DATA_ROOT = Path(os.getenv("DATA_ROOT", str(REPO_ROOT / "Utils")))

BEFOOD_RESTAURANTS_PATH = Path(os.getenv("BEFOOD_RESTAURANTS_PATH", str(DATA_ROOT / "befood_bachkhoa_restaurants.csv")))
BEFOOD_MENU_PATH = Path(os.getenv("BEFOOD_MENU_PATH", str(DATA_ROOT / "befood_bachkhoa_menu_items.csv")))
FOODY_PATH = Path(os.getenv("FOODY_PATH", str(DATA_ROOT / "foody_hust_places_from_store_csv.csv")))

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", os.getenv("NEO4J_USERNAME", "neo4j"))
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

QDRANT_HOST = os.getenv("QDRANT_HOST", "localhost")
QDRANT_PORT = int(os.getenv("QDRANT_PORT", 6333))
COLL_TEXT_UNIT = os.getenv("COLL_TEXT_UNIT", "graphrag_text_units_vietnamese")
COLL_RESTAURANT = os.getenv("COLL_RESTAURANT", "graphrag_restaurants_vietnamese")

EMBED_MODEL = os.getenv("EMBED_MODEL", "bkai-foundation-models/vietnamese-bi-encoder")
EMBED_PREFIX_QUERY = os.getenv("EMBED_PREFIX_QUERY", "")
EMBED_PREFIX_PASSAGE = os.getenv("EMBED_PREFIX_PASSAGE", "")
ASPECT_SENTIMENT_MODEL = os.getenv("ASPECT_SENTIMENT_MODEL", "wonrax/phobert-base-vietnamese-sentiment")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "openai").lower()
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None
ANTHROPIC_MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-4-20250514")

COMMUNITY_LEVEL = int(os.getenv("COMMUNITY_LEVEL", "0"))
RRF_K = int(os.getenv("RRF_K", "60"))
SIMILARITY_TOP_K = int(os.getenv("SIMILARITY_TOP_K", "8"))
SIMILARITY_MIN_SCORE = float(os.getenv("SIMILARITY_MIN_SCORE", "0.62"))

def env_float(name: str, default: Optional[float] = None) -> Optional[float]:
    raw = os.getenv(name)
    if raw is None or str(raw).strip() == "":
        return default
    try:
        return float(raw)
    except ValueError:
        return default

# Optional per-session user location. Set these from an app request, notebook cell, or .env.
USER_LAT = env_float("USER_LAT")
USER_LNG = env_float("USER_LNG")
MAX_DISTANCE_KM = env_float("MAX_DISTANCE_KM")
DISTANCE_WEIGHT = float(os.getenv("DISTANCE_WEIGHT", "0.20"))
DISTANCE_DECAY_KM = float(os.getenv("DISTANCE_DECAY_KM", "3.0"))
RECREATE_QDRANT = os.getenv("RECREATE_QDRANT", "true").lower() in {"1", "true", "yes", "y"}
RUN_COMMUNITY_REPORTS = os.getenv("RUN_COMMUNITY_REPORTS", "true").lower() in {"1", "true", "yes", "y"}
USE_LLM_GRAPH_EXTRACTION = os.getenv("USE_LLM_GRAPH_EXTRACTION", "false").lower() in {"1", "true", "yes", "y"}
ASPECT_SENTIMENT_BACKEND = os.getenv("ASPECT_SENTIMENT_BACKEND", "llm").lower()
LLM_ASPECT_MODEL_ID = os.getenv("LLM_ASPECT_MODEL_ID", OPENAI_MODEL)
LLM_GRAPH_EXTRACTION_MODEL_ID = os.getenv("LLM_GRAPH_EXTRACTION_MODEL_ID", OPENAI_MODEL)

print("Config loaded")
print("Repo root:", REPO_ROOT)
print("Data files:", BEFOOD_RESTAURANTS_PATH, BEFOOD_MENU_PATH, FOODY_PATH, sep="\n  ")
print("Neo4j:", NEO4J_URI, "user=", NEO4J_USER)
print("Qdrant:", f"{QDRANT_HOST}:{QDRANT_PORT}")
print("Embedding:", EMBED_MODEL)
print("Aspect sentiment:", ASPECT_SENTIMENT_MODEL)
print("User location:", USER_LAT, USER_LNG, "max_distance_km=", MAX_DISTANCE_KM)
print("Recreate Qdrant:", RECREATE_QDRANT, "run community reports:", RUN_COMMUNITY_REPORTS, "LLM graph extraction:", USE_LLM_GRAPH_EXTRACTION)

missing = [str(x) for x in [BEFOOD_RESTAURANTS_PATH, BEFOOD_MENU_PATH, FOODY_PATH] if not x.exists()]
if missing:
    raise FileNotFoundError("Missing required data file(s):\n" + "\n".join(missing))


Config loaded
Repo root: d:\restaurant-kg-recommender-master
Data files:
  Utils\befood_bachkhoa_restaurants.csv
  Utils\befood_bachkhoa_menu_items.csv
  Utils\foody_hust_places_from_store_csv.csv
Neo4j: bolt://localhost:7687 user= neo4j
Qdrant: localhost:6333
Embedding: bkai-foundation-models/vietnamese-bi-encoder
Aspect sentiment: wonrax/phobert-base-vietnamese-sentiment
User location: 21.005 105.843 max_distance_km= 3.0
Recreate Qdrant: True run community reports: True LLM graph extraction: True


In [5]:
# Define get_llm function manually
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

def get_llm():
    if LLM_PROVIDER == "anthropic":
        if not ANTHROPIC_API_KEY:
            raise RuntimeError("LLM_PROVIDER=anthropic but ANTHROPIC_API_KEY is missing.")
        return ChatAnthropic(model=ANTHROPIC_MODEL, api_key=ANTHROPIC_API_KEY, temperature=0)
    if LLM_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("LLM_PROVIDER=openai but OPENAI_API_KEY is missing.")
        return ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, temperature=0)
    raise ValueError(f"Unsupported LLM_PROVIDER={LLM_PROVIDER}. Use 'openai' or 'anthropic'.")

# Test get_llm function
try:
    llm = get_llm()
    print("✅ LLM initialized successfully")
    print(f"LLM type: {type(llm)}")
except Exception as e:
    print(f"❌ Error initializing LLM: {e}")

✅ LLM initialized successfully
LLM type: <class 'langchain_openai.chat_models.base.ChatOpenAI'>


In [6]:
from openai import OpenAI

client = OpenAI(
     api_key = OPENAI_API_KEY,

    base_url=OPENAI_BASE_URL
    
)

try:
    response = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {
                "role": "user",
                "content": "Xin chào"
            }
        ]
    )

    print("✅ API hoạt động")
    print(response.choices[0].message.content)

except Exception as e:
    print("❌ API lỗi")
    print(type(e).__name__)
    print(e)

✅ API hoạt động
❌ API lỗi
AttributeError
'str' object has no attribute 'choices'


## 2. Inspect dữ liệu thật

In [7]:
raw_befood = pd.read_csv(BEFOOD_RESTAURANTS_PATH)
raw_menu = pd.read_csv(BEFOOD_MENU_PATH)
raw_foody = pd.read_csv(FOODY_PATH)

display(raw_befood.head(2))
display(raw_menu.head(2))
display(raw_foody.head(2))

print("Shapes:")
print("  raw_befood:", raw_befood.shape)
print("  raw_menu  :", raw_menu.shape)
print("  raw_foody :", raw_foody.shape)


,restaurant_id,restaurant_name,source,matched_terms_text,latitude,longitude,distance_m,distance_km,address,rating,review_count,price_min,price_max,categories_text,opening_hours,delivery_time,image_url,menu_count,comment_count,comments_list
0,25639,Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn,befood,bún chả | cơm | phở,21.009489,105.827682,1921.57,1.922,"101A5 ngõ 167 Tây Sơn, Quang Trung, Đống Đa, H...",4.6,562,15000.0,490009.0,COMBO | BÚN MIẾN PHỞ | CƠM GÀ | ĐỒ UỐNG | Cơm ...,15:30-20:00,30.0,https://media.be.com.vn/bizops/image/1cd2e1e9-...,47,4,"[""ngon đấy, Ngon xỉu"", ""đặt phở cho bún?"", ""bú..."
1,97742,"NaBi Bún Chả Quạt, Cơm Tấm & Cơm Văn Phòng - T...",befood,bánh cuốn | bún chả | cơm,21.010160,105.826378,2071.82,2.072,"55C Ngách 25 Ngõ 167 Tây Sơn, Phường Quang T...",4.3,11,58000.0,95000.0,cơm vị nhà nấu - thay đổi theo ngày | Bún Chả ...,17:00-20:00,30.0,https://media.be.com.vn/bizops/image/662cdec9-...,14,7,"[""cơm quá tệ, Không hợp khẩu vị, Đóng gói chưa..."


,restaurant_id,restaurant_name,category_id,category_name,restaurant_item_id,item_name,item_details,price,old_price,order_count,like_count,dislike_count,category_position,item_position,item_image
0,25639,Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn,114762,COMBO,867932,Phở trộn bò và gà - 1 Coca cola,NaN,49000,49000,100,2,0,1,1,https://media.be.com.vn/bizops/image/3d6efda3-...
1,25639,Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn,114762,COMBO,867933,Miến trộn bò và gà - 1 coca cola,NaN,49000,49000,85,1,0,1,2,https://media.be.com.vn/bizops/image/69b90d3b-...


,input_store_id,input_store_name,input_address,matched_foody_url,crawl_status,error,restaurant_id,name,url,title_page,...,categories,cuisines,audiences,wifi,opening_hours,rating_quality,rating_position,rating_service,rating_price,rating_space
0,126162,Tiệm Trà Cô Chiêu - Milk Tea & Ăn Vặt - Lê Tha...,"76 Lê Thanh Nghị, Phường Bạch Mai, Hà Nội",NaN,foody_url_not_found,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,126679,Mì Cay Sassin 7 Cấp Độ Hàn Quốc - Bách Khoa,"104K9 Nguyễn Hiền, Phường Bạch Mai, Hà Nội",NaN,foody_url_not_found,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Shapes:
  raw_befood: (200, 20)
  raw_menu  : (3245, 15)
  raw_foody : (49, 37)


In [8]:
def _is_nan(x: Any) -> bool:
    return pd.isna(x)

def normalize_text(x: Any) -> str:
    if x is None or _is_nan(x):
        return ""
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKC", x)
    return re.sub(r"\s+", " ", x)

def slugify_vn(x: Any) -> str:
    x = normalize_text(x)
    x = ''.join(c for c in unicodedata.normalize('NFD', x) if unicodedata.category(c) != 'Mn')
    x = x.replace("đ", "d")
    x = re.sub(r"[^a-z0-9]+", "-", x).strip("-")
    return x

def to_float(x: Any) -> Optional[float]:
    if x is None or _is_nan(x) or str(x).strip() == "":
        return None
    try:
        return float(x)
    except:
        s = str(x).replace(".", "").replace(",", ".")
        s = re.sub(r"[^0-9.\-]", "", s)
        try:
            return float(s)
        except:
            return None

def to_int(x: Any) -> Optional[int]:
    v = to_float(x)
    return None if v is None else int(v)

def parse_jsonish(x: Any) -> Any:
    if x is None or _is_nan(x):
        return None
    if isinstance(x, (list, dict)):
        return x
    s = str(x).strip()
    if not s:
        return None
    for fn in (json.loads, ast.literal_eval):
        try:
            return fn(s)
        except:
            pass
    return s

def split_semi(x: Any) -> List[str]:
    if x is None or _is_nan(x):
        return []
    if isinstance(x, list):
        return [str(i).strip() for i in x if str(i).strip()]
    parsed = parse_jsonish(x)
    if isinstance(parsed, list):
        return [str(i).strip() for i in parsed if str(i).strip()]
    s = str(x)
    parts = re.split(r"[;|,/]", s)
    return [p.strip() for p in parts if p and p.strip()]

def parse_price_band(text: Any) -> Optional[str]:
    if text is None or _is_nan(text):
        return None
    s = normalize_text(text)
    if not s:
        return None
    nums = re.findall(r"\d[\d\.]*", s)
    vals = []
    for n in nums:
        try:
            vals.append(int(float(n.replace(".", ""))))
        except:
            pass
    if not vals:
        return None
    hi = max(vals)
    if hi <= 50000:
        return "budget"
    if hi <= 120000:
        return "mid"
    return "premium"

def set_user_location(lat: Optional[float], lng: Optional[float], max_distance_km: Optional[float] = None):
    """Set current user location for distance-aware retrieval in this notebook session."""
    global USER_LAT, USER_LNG, MAX_DISTANCE_KM
    USER_LAT = to_float(lat)
    USER_LNG = to_float(lng)
    if max_distance_km is not None:
        MAX_DISTANCE_KM = to_float(max_distance_km)


def get_user_location(user_lat: Optional[float] = None, user_lng: Optional[float] = None) -> Tuple[Optional[float], Optional[float]]:
    lat = to_float(user_lat) if user_lat is not None else USER_LAT
    lng = to_float(user_lng) if user_lng is not None else USER_LNG
    return lat, lng


def haversine_km(lat1: Any, lng1: Any, lat2: Any, lng2: Any) -> Optional[float]:
    lat1, lng1, lat2, lng2 = map(to_float, [lat1, lng1, lat2, lng2])
    if None in (lat1, lng1, lat2, lng2):
        return None
    r = 6371.0088
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lng2 - lng1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return round(2 * r * math.asin(math.sqrt(a)), 3)


def distance_score(distance_km: Any, decay_km: float = DISTANCE_DECAY_KM) -> float:
    d = to_float(distance_km)
    if d is None:
        return 0.0
    return 1.0 / (1.0 + max(d, 0.0) / max(decay_km, 1e-9))


def add_user_distance_to_record(rec: dict, user_lat: Optional[float] = None, user_lng: Optional[float] = None) -> dict:
    lat, lng = get_user_location(user_lat, user_lng)
    if lat is None or lng is None:
        rec.setdefault("distance_km", None)
        rec.setdefault("distance_score", 0.0)
        return rec
    dist = haversine_km(lat, lng, rec.get("lat"), rec.get("lng"))
    rec["distance_km"] = dist
    rec["distance_score"] = distance_score(dist)
    return rec



## 3. Canonicalize schema và hợp nhất nguồn dữ liệu

In [42]:
HANOI_DISTRICTS = [
    "Ba Dinh", "Hoan Kiem", "Tay Ho", "Long Bien", "Cau Giay", "Dong Da", "Hai Ba Trung",
    "Hoang Mai", "Thanh Xuan", "Nam Tu Liem", "Bac Tu Liem", "Ha Dong", "Son Tay",
    "Ba Vi", "Chuong My", "Dan Phuong", "Dong Anh", "Gia Lam", "Hoai Duc", "Me Linh",
    "My Duc", "Phu Xuyen", "Phuc Tho", "Quoc Oai", "Soc Son", "Thach That", "Thanh Oai",
    "Thanh Tri", "Thuong Tin", "Ung Hoa",
]
URBAN_DISTRICTS = {"Ba Dinh", "Hoan Kiem", "Tay Ho", "Long Bien", "Cau Giay", "Dong Da", "Hai Ba Trung", "Hoang Mai", "Thanh Xuan", "Nam Tu Liem", "Bac Tu Liem", "Ha Dong"}
DISTRICT_ALIASES = {slugify_vn(x): (f"Quan {x}" if x in URBAN_DISTRICTS else x) for x in HANOI_DISTRICTS}

def infer_district(address: Any) -> Optional[str]:
    s = slugify_vn(address)
    for key, label in DISTRICT_ALIASES.items():
        if key and key in s:
            return label
    return None

def price_band_from_bounds(price_min: Any, price_max: Any) -> Optional[str]:
    vals = [to_float(price_min), to_float(price_max)]
    vals = [v for v in vals if v is not None]
    if not vals:
        return None
    hi = max(vals)
    if hi <= 50000:
        return "budget"
    if hi <= 120000:
        return "mid"
    return "premium"

def price_band_from_menu_prices(prices: Any) -> Optional[str]:
    vals = [to_float(x) for x in list(prices) if to_float(x) is not None and to_float(x) > 0]
    if not vals:
        return None
    median = float(np.median(vals))
    budget_ratio = sum(v <= 50000 for v in vals) / len(vals)
    premium_ratio = sum(v > 120000 for v in vals) / len(vals)
    # Median and item distribution are more robust than max price because menus often contain combo/outlier items.
    if median <= 50000 or budget_ratio >= 0.60:
        return "budget"
    if median <= 120000 and premium_ratio < 0.35:
        return "mid"
    return "premium"

def has_phrase_slug(s: str, phrase: str) -> bool:
    tokens = s.split("-")
    phrase_tokens = phrase.split("-")
    n = len(phrase_tokens)

    if n == 0:
        return False

    for i in range(len(tokens) - n + 1):
        if tokens[i:i+n] == phrase_tokens:
            return True

    return False

def normalize_dish_family(name: Any) -> Optional[str]:
    """Normalize exact menu item names into broader dish families.

    Examples: "C?m g? s?t chua ng?t" -> "c?m g?", "Ph? b? t?i" -> "ph?",
    "B?nh cu?n ch?" -> "b?nh cu?n". This is intentionally broader than exact MenuItem.
    """
    raw = normalize_text(name)
    s = slugify_vn(raw)
    if not s:
        return None
    rules = [
        ("com-tam", "cơm tấm"),
        ("com-ga", "cơm gà"),
        ("com-rang", "cơm rang"),
        ("bun-cha", "bún chả"),
        ("bun-bo", "bún bò"),
        ("bun-ca", "bún cá"),
        ("bun-rieu", "bún riêu"),
        ("banh-cuon", "bánh cuốn"),
        ("banh-mi", "bánh mì"),
        ("ga-ran", "gà rán"),
        ("mi-cay", "mì cay"),
        ("tra-sua", "trà sữa"),
        ("ca-phe", "cà phê"),
        ("cafe", "cà phê"),
        ("nuoc-ep", "nước ép"),
        ("pho", "phở"),
        ("com", "cơm"),
        ("bun", "bún"),
        ("ga", "gà"),
        ("mi", "mì"),
        ("mien", "miến"),
        ("chao", "cháo"),
        ("xoi", "xôi"),
        ("lau", "lẩu"),
        ("nuong", "nướng"),
    ]

    for phrase, family in rules:
        if has_phrase_slug(s, phrase):
            return family

    # Fallback: remove common modifiers and keep a compact 1-2 token family.
    cleaned = re.sub(
        r"(^|-)(combo|set|size|phan|them|coca|cola|pepsi|sprite|dasani|lon|chai|dac-biet|full|mix)($|-)",
        "-",
        s,
    )
    cleaned = re.sub(r"-+", "-", cleaned).strip("-")

    tokens = [t for t in cleaned.split("-") if t and not t.isdigit()]
    if not tokens:
        return None

    return " ".join(tokens[:2])

def canonicalize_befood_restaurants(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["store_id"] = df["restaurant_id"].astype("Int64").astype(str)
    out["store_key"] = out["store_id"]
    out["store_name"] = df["restaurant_name"].fillna("").astype(str)
    out["query_name"] = out["store_name"]
    out["address"] = df.get("address", pd.Series([None] * len(df)))
    out["query_address"] = out["address"]
    out["district"] = out["address"].apply(infer_district)
    out["city"] = "Ha Noi"
    out["lat"] = df.get("latitude", pd.Series([None] * len(df))).apply(to_float)
    out["lng"] = df.get("longitude", pd.Series([None] * len(df))).apply(to_float)
    out["gmaps_rating"] = df.get("rating", pd.Series([None] * len(df))).apply(to_float)
    out["gmaps_review_count"] = df.get("review_count", pd.Series([None] * len(df))).apply(to_int)
    out["price_min"] = df.get("price_min", pd.Series([None] * len(df))).apply(to_float)
    out["price_max"] = df.get("price_max", pd.Series([None] * len(df))).apply(to_float)
    out["price_band"] = [price_band_from_bounds(a, b) for a, b in zip(out["price_min"], out["price_max"])]
    out["categories"] = df.get("categories_text", pd.Series([None] * len(df))).apply(split_semi)
    out["matched_terms"] = df.get("matched_terms_text", pd.Series([None] * len(df))).apply(split_semi)
    out["opening_hours_raw"] = df.get("opening_hours")
    out["delivery_time"] = df.get("delivery_time", pd.Series([None] * len(df))).apply(to_float)
    out["image_url"] = df.get("image_url")
    out["menu_count"] = df.get("menu_count", pd.Series([None] * len(df))).apply(to_int)
    out["comment_count"] = df.get("comment_count", pd.Series([None] * len(df))).apply(to_int)
    out["source"] = df.get("source", pd.Series(["befood"] * len(df))).fillna("befood")
    out["atmosphere"] = [[] for _ in range(len(out))]
    out["crowd"] = [[] for _ in range(len(out))]
    out["name_norm"] = out["store_name"].apply(slugify_vn)
    out["addr_norm"] = out["address"].apply(slugify_vn)
    return out

def canonicalize_feedback_from_befood(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, r in df.iterrows():
        store_id = str(int(r["restaurant_id"])) if pd.notna(r.get("restaurant_id")) else ""
        store_name = str(r.get("restaurant_name") or "")
        comments = parse_jsonish(r.get("comments_list"))
        if isinstance(comments, str):
            comments = [comments]
        if not isinstance(comments, list):
            comments = []
        for i, comment in enumerate(comments):
            feedback = str(comment or "").strip()
            if not feedback:
                continue
            rows.append({
                "store_id": store_id,
                "store_key": store_id,
                "store_name": store_name,
                "rated_at": None,
                "rating": to_float(r.get("rating")) or 3.0,
                "feedback": feedback,
                "source": "befood_comment",
                "review_id": hashlib.md5(f"{store_id}|{i}|{feedback}".encode("utf-8")).hexdigest()[:16],
                "name_norm": slugify_vn(store_name),
            })
    return pd.DataFrame(rows, columns=["store_id", "store_key", "store_name", "rated_at", "rating", "feedback", "source", "review_id", "name_norm"])

def canonicalize_menu_items(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["store_id"] = df["restaurant_id"].astype("Int64").astype(str)
    out["store_key"] = out["store_id"]
    out["store_name"] = df["restaurant_name"].fillna("").astype(str)
    out["category_id"] = df.get("category_id", pd.Series([None] * len(df))).astype("Int64").astype(str)
    out["category_name"] = df.get("category_name", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()
    out["menu_item_id"] = df["restaurant_item_id"].astype("Int64").astype(str)
    out["item_name"] = df["item_name"].fillna("").astype(str).str.strip()
    out["item_details"] = df.get("item_details", pd.Series([""] * len(df))).fillna("").astype(str).str.strip()
    out["price"] = df.get("price", pd.Series([None] * len(df))).apply(to_float)
    out["old_price"] = df.get("old_price", pd.Series([None] * len(df))).apply(to_float)
    out["order_count"] = df.get("order_count", pd.Series([0] * len(df))).apply(to_int).fillna(0).astype(int)
    out["like_count"] = df.get("like_count", pd.Series([0] * len(df))).apply(to_int).fillna(0).astype(int)
    out["dislike_count"] = df.get("dislike_count", pd.Series([0] * len(df))).apply(to_int).fillna(0).astype(int)
    out["category_position"] = df.get("category_position", pd.Series([None] * len(df))).apply(to_int)
    out["item_position"] = df.get("item_position", pd.Series([None] * len(df))).apply(to_int)
    out["item_image"] = df.get("item_image")
    out["item_norm"] = out["item_name"].apply(slugify_vn)
    return out[out["item_name"].ne("")].copy()

def canonicalize_foody(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame()
    out["input_store_id"] = df["input_store_id"].astype("Int64").astype(str)
    out["input_store_name"] = df["input_store_name"]
    out["crawl_status"] = df.get("crawl_status")
    out["foody_name"] = df.get("name")
    out["foody_address"] = df.get("address")
    out["district"] = df.get("district")
    out["area"] = df.get("area")
    out["city"] = df.get("city")
    out["foody_lat"] = df.get("lat").apply(to_float)
    out["foody_lng"] = df.get("lng").apply(to_float)
    out["foody_rating"] = df.get("avg_rating").apply(to_float)
    out["foody_review_count"] = df.get("total_review").apply(to_int)
    out["price_min"] = df.get("price_min").apply(to_float)
    out["price_max"] = df.get("price_max").apply(to_float)
    out["foody_price_band"] = [price_band_from_bounds(a, b) for a, b in zip(out["price_min"], out["price_max"])]
    out["categories"] = df.get("categories", pd.Series([None]*len(df))).apply(split_semi)
    out["cuisines"] = df.get("cuisines", pd.Series([None]*len(df))).apply(split_semi)
    out["audiences"] = df.get("audiences", pd.Series([None]*len(df))).apply(split_semi)
    out["opening_hours_foody"] = df.get("opening_hours")
    out["rating_quality"] = df.get("rating_quality").apply(to_float)
    out["rating_position"] = df.get("rating_position").apply(to_float)
    out["rating_service"] = df.get("rating_service").apply(to_float)
    out["rating_price"] = df.get("rating_price").apply(to_float)
    out["rating_space"] = df.get("rating_space").apply(to_float)
    out["store_key"] = out["input_store_id"].astype(str)
    out["name_norm"] = out["input_store_name"].apply(slugify_vn)
    return out

restaurants_base = canonicalize_befood_restaurants(raw_befood)
feedback = canonicalize_feedback_from_befood(raw_befood)
menu_items = canonicalize_menu_items(raw_menu)
foody = canonicalize_foody(raw_foody)
gmaps = restaurants_base
print(f"Canonicalized: restaurants={len(restaurants_base)}, menu_items={len(menu_items)}, comments={len(feedback)}, foody_rows={len(foody)}")


Canonicalized: restaurants=200, menu_items=3245, comments=879, foody_rows=49


In [43]:
restaurants = restaurants_base.merge(
    foody.drop_duplicates("store_key"),
    on="store_key",
    how="left",
    suffixes=("", "_foody")
)

restaurants["name"] = restaurants["store_name"].fillna(restaurants["foody_name"]).fillna(restaurants["query_name"])
restaurants["address_final"] = restaurants["address"].fillna(restaurants["foody_address"]).fillna(restaurants["query_address"])
restaurants["district_final"] = restaurants["district_foody"].fillna(restaurants["district"])
restaurants["city_final"] = restaurants["city_foody"].fillna(restaurants["city"]).fillna("Ha Noi")
restaurants["lat_final"] = restaurants["lat"].fillna(restaurants["foody_lat"])
restaurants["lng_final"] = restaurants["lng"].fillna(restaurants["foody_lng"])
restaurants["source_price_band_final"] = restaurants["price_band"].fillna(restaurants["foody_price_band"])

def merge_list_cols(*cols):
    out, seen = [], set()
    for col in cols:
        vals = col if isinstance(col, list) else []
        for x in vals:
            x = str(x).strip()
            key = x.lower()
            if x and key not in seen:
                out.append(x)
                seen.add(key)
    return out

menu_categories_by_store = menu_items.groupby("store_key")["category_name"].apply(lambda s: sorted({x for x in s if x})).to_dict()
menu_top_items_by_store = (
    menu_items.sort_values(["store_key", "order_count", "like_count"], ascending=[True, False, False])
    .groupby("store_key")
    .head(12)
    .groupby("store_key")["item_name"]
    .apply(list)
    .to_dict()
)
menu_price_stats = menu_items.groupby("store_key").agg(
    menu_item_count=("menu_item_id", "count"),
    menu_price_min=("price", "min"),
    menu_price_max=("price", "max"),
    menu_price_median=("price", "median"),
    menu_budget_item_ratio=("price", lambda s: float((s <= 50000).mean()) if len(s) else 0.0),
    menu_price_band=("price", price_band_from_menu_prices),
).reset_index()

restaurants["categories_final"] = [
    merge_list_cols(a, b, menu_categories_by_store.get(k, []), terms)
    for a, b, k, terms in zip(restaurants["categories"], restaurants["categories_foody"], restaurants["store_key"], restaurants["matched_terms"])
]
restaurants["cuisines_final"] = restaurants["cuisines"].apply(lambda x: x if isinstance(x, list) else [])
restaurants["audiences_final"] = restaurants["audiences"].apply(lambda x: x if isinstance(x, list) else [])

summary = pd.DataFrame({
    "store_key": restaurants["store_key"],
    "name": restaurants["name"],
    "address": restaurants["address_final"],
    "district": restaurants["district_final"],
    "city": restaurants["city_final"],
    "gmaps_rating": restaurants["gmaps_rating"],
    "foody_rating": restaurants["foody_rating"],
    "review_count": restaurants["gmaps_review_count"].fillna(restaurants["foody_review_count"]),
    "source_price_band": restaurants["source_price_band_final"],
    "price_min": restaurants["price_min"],
    "price_max": restaurants["price_max"],
    "categories": restaurants["categories_final"],
    "cuisines": restaurants["cuisines_final"],
    "atmosphere": restaurants["atmosphere"],
    "audiences": restaurants["audiences_final"],
    "opening_hours": restaurants["opening_hours_raw"].fillna(restaurants["opening_hours_foody"]),
    "delivery_time": restaurants["delivery_time"],
    "image_url": restaurants["image_url"],
    "top_menu_items": restaurants["store_key"].map(menu_top_items_by_store).apply(lambda x: x if isinstance(x, list) else []),
    "lat": restaurants["lat_final"],
    "lng": restaurants["lng_final"],
}).merge(menu_price_stats, on="store_key", how="left")

if USER_LAT is not None and USER_LNG is not None:
    summary["distance_km"] = [haversine_km(USER_LAT, USER_LNG, lat, lng) for lat, lng in zip(summary["lat"], summary["lng"])]
else:
    summary["distance_km"] = None
summary["distance_score"] = summary["distance_km"].apply(distance_score)

summary["menu_item_count"] = summary["menu_item_count"].fillna(0).astype(int)
summary["price_band"] = summary["menu_price_band"].fillna(summary["source_price_band"])
summary["rating"] = summary["gmaps_rating"].fillna(summary["foody_rating"])

display(summary.head(10))
print("Unified restaurant table ready:", summary.shape)
print("Feedback comments ready:", feedback.shape)
print("Menu items ready:", menu_items.shape)


,store_key,name,address,district,city,gmaps_rating,foody_rating,review_count,source_price_band,price_min,...,menu_item_count,menu_price_min,menu_price_max,menu_price_median,menu_budget_item_ratio,menu_price_band,distance_km,distance_score,price_band,rating
0,25639,Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn,"101A5 ngõ 167 Tây Sơn, Quang Trung, Đống Đa, H...",Quan Dong Da,Ha Noi,4.6,NaN,562,premium,15000.0,...,43,45000.0,50000.0,45000.0,1.000000,budget,1.667,0.642811,budget,4.6
1,97742,"NaBi Bún Chả Quạt, Cơm Tấm & Cơm Văn Phòng - T...","55C Ngách 25 Ngõ 167 Tây Sơn, Phường Quang T...",Quan Dong Da,Ha Noi,4.3,NaN,11,mid,58000.0,...,14,58000.0,95000.0,66000.0,0.000000,mid,1.818,0.622665,mid,4.3
2,130307,"Phở Bò Nội - Bún Chả, Cháo & Cơm - Trần Hưng Đạo","94A Trần Hưng Đạo, Phường Cửa Nam, Hà Nội",None,Ha Noi,4.8,NaN,10,mid,50000.0,...,3,50000.0,60000.0,50000.0,0.666667,budget,2.052,0.593824,budget,4.8
3,22758,PhuongKitchen - Cơm & Đồ Ăn Vặt - Ngõ Chùa Liê...,"22 Ngách 55 Ngõ Chùa Liên Phái Bạch Mai, Phườn...",Quận Hai Bà Trưng,Hà Nội,4.7,0.0,7,mid,11000.0,...,32,31000.0,65000.0,52000.0,0.312500,mid,0.614,0.830105,mid,4.7
4,11471,Thiên Trường - Phở Cồ & Cơm Gà Xối Mắm - Giải ...,"15 ngõ 75 Giải Phóng, Đồng Tâm, Đống Đa, Hà Nội",Quận Hai Bà Trưng,Hà Nội,4.5,0.0,39,mid,10000.0,...,20,30000.0,80000.0,59000.0,0.250000,mid,0.263,0.919399,mid,4.5
5,11326,Huỳnh Duyên - Cơm Văn Phòng & Nước Ép - Giải P...,"17 Ngõ 205 Giải Phóng, P. Đồng Tâm, Quận Hai B...",Quan Hai Ba Trung,Ha Noi,4.8,NaN,88,mid,15000.0,...,21,40000.0,55000.0,40000.0,0.761905,budget,0.480,0.862069,budget,4.8
6,8314,Phở Bò Cơm Rang Duy Nhất - Giải Phóng,"16 Ngõ 205 Giải Phóng, phường Bạch Mai , Hà Nội",None,Ha Noi,4.6,NaN,1787,mid,35000.0,...,22,55000.0,70000.0,60000.0,0.000000,mid,0.489,0.859845,mid,4.6
7,126147,"Nhà Hàng Bin Bo - Bún Riêu Cua, Cơm Rang & Phở...","11 Ngách 8 Ngõ 4 Phương Mai, Phường Kim Liên, ...",None,Ha Noi,4.3,NaN,4,premium,1000.0,...,37,30000.0,90000.0,70000.0,0.270270,mid,0.405,0.881057,mid,4.3
8,127398,"Vũ Gia - Phở Gà, Bún Thang & Cơm Gà Xối Mỡ - P...","337 Phố Huế, Phường Hai Bà Trưng, Hà Nội",Quan Hai Ba Trung,Ha Noi,0.0,NaN,0,premium,6500.0,...,54,52000.0,104000.0,71500.0,0.000000,mid,1.019,0.746454,mid,0.0
9,124125,Gà Rán BigBites - Cơm Gà & Bánh Gà - Đại La,"60E2 Ngõ 128C Đại La, Phường Bạch Mai, Hà Nội",None,Ha Noi,5.0,NaN,1,premium,9000.0,...,9,39000.0,55000.0,55000.0,0.444444,mid,0.796,0.790306,mid,5.0


Unified restaurant table ready: (200, 31)
Feedback comments ready: (879, 9)
Menu items ready: (3245, 17)


## 3.1. Menu-derived entity extraction

In [44]:
from langchain.prompts import ChatPromptTemplate

In [ ]:
# Deprecated: dish/entity extraction is now built from befood_bachkhoa_menu_items.csv in the menu-derived entity cell below.

## 4. PhoBERT-based aspect sentiment và TextUnit evidence


In [45]:

import torch, os
print("ASPECT_SENTIMENT_BACKEND =", ASPECT_SENTIMENT_BACKEND)
print("ASPECT_SENTIMENT_MODEL =", ASPECT_SENTIMENT_MODEL)
print("Num reviews =", len(feedback))
print("Device available =", "cuda" if torch.cuda.is_available() else "cpu")

ASPECT_SENTIMENT_BACKEND = llm
ASPECT_SENTIMENT_MODEL = wonrax/phobert-base-vietnamese-sentiment
Num reviews = 879
Device available = cuda


In [46]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

import os
import json
import time
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional

ASPECTS: Dict[str, str] = {
    "food_quality": "chất lượng món ăn, độ ngon, hương vị, độ tươi",
    "service": "thái độ và chất lượng phục vụ của nhân viên",
    "cleanliness": "vệ sinh, sạch sẽ, mùi, an toàn thực phẩm",
    "packaging": "đóng gói, giao hàng không đổ vỡ, đầy đủ món",
    "price": "giá cả, độ đáng tiền, phù hợp sinh viên",
    "space": "không gian quán, độ rộng, yên tĩnh, thoải mái",
    "speed": "tốc độ phục vụ, thời gian chờ món hoặc giao hàng",
}

LABEL_TO_SCORE = {
    "negative": -1.0, "neg": -1.0, "0": -1.0,
    "neutral": 0.0, "neu": 0.0, "1": 0.0,
    "positive": 1.0, "pos": 1.0, "2": 1.0,
}

def normalize_review_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = unicodedata.normalize("NFKC", s.lower())
    return re.sub(r"\s+", " ", s).strip()

class PhoBERTAspectSentiment:
    """Aspect sentiment with batched multi-aspect inference.

    FIX (Vấn đề 6): Thay vì gọi model 7 lần per review (7 aspects × N reviews = 7N calls),
    ta build 7×N texts trước, rồi chạy batched inference 1 lần dùng tokenizer padding.
    Giảm 5–10× thời gian indexing so với sequential forward pass.
    """
    def __init__(self, model_name: str, device: Optional[str] = None, batch_size: int = 64):
        self.model_name = model_name
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device)
        self.model.eval()
        self.id2label = {int(k): str(v).lower() for k, v in self.model.config.id2label.items()}
        if not self.id2label:
            raise RuntimeError(f"Model {model_name} does not expose id2label; cannot map sentiment labels safely.")
        self.aspect_names = list(ASPECTS.keys())
        self.aspect_descs = list(ASPECTS.values())

    @torch.inference_mode()
    def _batch_infer(self, texts: List[str]) -> List[float]:
        """Run batched inference, return expected sentiment score per text."""
        scores = []
        for i in range(0, len(texts), self.batch_size):
            batch = texts[i:i + self.batch_size]
            enc = self.tokenizer(
                batch,
                truncation=True,
                max_length=256,
                padding=True,
                return_tensors="pt",
            ).to(self.device)
            logits = self.model(**enc).logits  # (B, n_labels)
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
            for row in probs:
                expected = 0.0
                mass = 0.0
                for j, p in enumerate(row):
                    label = self.id2label.get(j, str(j)).lower()
                    score = LABEL_TO_SCORE.get(label)
                    if score is None:
                        raise RuntimeError(f"Unsupported label {label} from {self.model_name}. Update LABEL_TO_SCORE.")
                    expected += float(p) * score
                    mass += float(p)
                scores.append(round(expected / max(mass, 1e-9), 4))
        return scores

    def score_reviews_batch(self, reviews: List[str]) -> List[Dict[str, float]]:
        """Score all aspects for all reviews in one batched forward pass sequence.

        Build (n_aspects × n_reviews) texts, infer in batches, reshape.
        This is O(n_aspects * ceil(N/batch)) batches instead of O(n_aspects * N) sequential calls.
        """
        aspect_texts = []
        for asp_desc in self.aspect_descs:
            for review in reviews:
                aspect_texts.append(f"Khía cạnh: {asp_desc}. Nhận xét: {review}")

        all_scores = self._batch_infer(aspect_texts)

        n = len(reviews)
        results = []
        for review_idx in range(n):
            result = {}
            for asp_idx, asp_name in enumerate(self.aspect_names):
                result[asp_name] = all_scores[asp_idx * n + review_idx]
            results.append(result)
        return results

    # backward compat: single review
    def score_review(self, review: str) -> Dict[str, float]:
        return self.score_reviews_batch([review])[0]

def classify_sentiment_from_aspects(aspect_scores: Dict[str, float]) -> str:
    avg = float(np.mean(list(aspect_scores.values()))) if aspect_scores else 0.0
    if avg >= 0.20:
        return "positive"
    if avg <= -0.20:
        return "negative"
    return "neutral"

feedback_proc = feedback.copy()
feedback_proc["feedback_norm"] = feedback_proc["feedback"].apply(normalize_review_text)
reviews_list = feedback_proc["feedback_norm"].tolist()

# ============================================================
# ✅ SỬA 2: checkpoint/log ngoài, lưu từng comment
# ============================================================
CACHE_DIR_PATH = Path(os.getenv("CACHE_DIR", ".cache/graphrag"))
CACHE_DIR_PATH.mkdir(parents=True, exist_ok=True)

# File này lưu kết quả đã hoàn thành. Lần sau đọc lại để skip comment cũ.
LLM_ASPECT_CHECKPOINT_PATH = CACHE_DIR_PATH / "llm_aspect_sentiment_checkpoint.json"

# File này là log từng comment đã chạy xong, đọc bằng text cũng được.
LLM_ASPECT_LOG_PATH = CACHE_DIR_PATH / "llm_aspect_sentiment_progress.jsonl"

# Đổi version này nếu bạn đổi prompt/aspect schema và muốn ép chạy lại từ đầu.
LLM_ASPECT_CHECKPOINT_VERSION = os.getenv("LLM_ASPECT_CHECKPOINT_VERSION", "v1")


def _current_aspect_model_id() -> str:
    if ASPECT_SENTIMENT_BACKEND == "llm":
        return str(globals().get("LLM_ASPECT_MODEL_ID", os.getenv("LLM_ASPECT_MODEL_ID", "unknown_llm_model")))

    if ASPECT_SENTIMENT_BACKEND == "phobert":
        return str(globals().get("ASPECT_SENTIMENT_MODEL", "unknown_phobert_model"))

    return "unknown_model"


def _review_cache_key(review_text: str) -> str:
    """
    Key theo backend + model + version + aspect list + nội dung review.
    Nếu comment giống y hệt thì lần sau không gọi lại.
    """
    raw = json.dumps(
        {
            "backend": ASPECT_SENTIMENT_BACKEND,
            "model_id": _current_aspect_model_id(),
            "checkpoint_version": LLM_ASPECT_CHECKPOINT_VERSION,
            "aspects": list(ASPECTS.keys()),
            "review_text": review_text,
        },
        ensure_ascii=False,
        sort_keys=True,
    )

    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def _atomic_save_json(path: Path, data: Any) -> None:
    """
    Ghi an toàn: ghi ra file .tmp trước, xong mới replace.
    Tránh mất điện giữa lúc ghi làm hỏng file json chính.
    """
    tmp_path = Path(str(path) + ".tmp")

    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    tmp_path.replace(path)


def _load_aspect_checkpoint() -> Dict[str, Any]:
    if not LLM_ASPECT_CHECKPOINT_PATH.exists():
        print("ℹ️ No external LLM aspect checkpoint found, starting fresh")
        return {}

    try:
        with LLM_ASPECT_CHECKPOINT_PATH.open("r", encoding="utf-8") as f:
            data = json.load(f)

        if not isinstance(data, dict):
            print("⚠️ Checkpoint file is not a dict, starting fresh")
            return {}

        print(f"✅ Loaded external LLM aspect checkpoint: {len(data)} items")
        return data

    except Exception as e:
        print(f"⚠️ Could not load checkpoint, starting fresh: {type(e).__name__}: {e}")
        return {}


def _append_aspect_log(record: Dict[str, Any]) -> None:
    """
    Mỗi comment xong append 1 dòng JSONL.
    File này để nhìn tiến độ rõ, không phải cache chính.
    """
    with LLM_ASPECT_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
        f.flush()


def _normalize_aspect_scores(scores: Any) -> Dict[str, float]:
    """
    Ép output từ LLM/PhoBERT về dict aspect -> float.
    Thiếu aspect thì fill 0.0 để pipeline không crash.
    """
    if hasattr(scores, "model_dump"):
        scores = scores.model_dump()

    if not isinstance(scores, dict):
        raise RuntimeError(f"Aspect score must be dict, got {type(scores).__name__}: {scores}")

    clean = {}

    for asp in ASPECTS.keys():
        value = scores.get(asp, 0.0)

        try:
            value = float(value)
        except Exception:
            value = 0.0

        # clamp nhẹ để tránh LLM trả ngoài [-1, 1]
        value = max(-1.0, min(1.0, value))
        clean[asp] = round(value, 4)

    return clean


def _score_one_review(aspect_model: Any, review_text: str) -> Dict[str, float]:
    """
    ✅ SỬA 3: với LLM, gọi từng comment một.
    Không gọi score_reviews_batch(reviews_list) toàn bộ nữa.
    """
    if hasattr(aspect_model, "score_review"):
        raw = aspect_model.score_review(review_text)

    else:
        raw_list = aspect_model.score_reviews_batch([review_text])

        if not raw_list:
            raise RuntimeError("score_reviews_batch([review]) returned empty result")

        raw = raw_list[0]

    return _normalize_aspect_scores(raw)


def score_reviews_with_per_comment_checkpoint(
    aspect_model: Any,
    reviews: List[str],
) -> List[Dict[str, float]]:
    """
    ✅ SỬA 4: hàm chạy từng comment và lưu ngay sau mỗi comment.
    Nếu mất điện/interrupt:
    - những comment đã lưu trong checkpoint sẽ được skip
    - lần sau chạy tiếp từ comment chưa có
    """
    checkpoint = _load_aspect_checkpoint()

    results: List[Optional[Dict[str, float]]] = [None] * len(reviews)

    # Bước 1: load kết quả cũ
    hit_count = 0
    missing_indices = []

    for i, review_text in enumerate(reviews):
        key = _review_cache_key(review_text)

        if key in checkpoint and isinstance(checkpoint[key], dict):
            saved_scores = checkpoint[key].get("aspect_scores", checkpoint[key])
            results[i] = _normalize_aspect_scores(saved_scores)
            hit_count += 1

        else:
            missing_indices.append(i)

    print(f"Total comments: {len(reviews)}")
    print(f"Checkpoint hits: {hit_count}")
    print(f"Comments still need scoring: {len(missing_indices)}")
    print(f"Checkpoint path: {LLM_ASPECT_CHECKPOINT_PATH.resolve()}")
    print(f"Progress log path: {LLM_ASPECT_LOG_PATH.resolve()}")

    if not missing_indices:
        print("✅ All comments loaded from checkpoint. No API call needed.")
        return [r for r in results if r is not None]

    # Bước 2: chạy từng comment còn thiếu
    try:
        for pos, idx in enumerate(tqdm(missing_indices, desc="Aspect sentiment per comment"), start=1):
            review_text = reviews[idx]
            key = _review_cache_key(review_text)

            t0 = time.time()

            scores = _score_one_review(aspect_model, review_text)

            record = {
                "aspect_scores": scores,
                "backend": ASPECT_SENTIMENT_BACKEND,
                "model_id": _current_aspect_model_id(),
                "checkpoint_version": LLM_ASPECT_CHECKPOINT_VERSION,
                "review_sha": hashlib.sha256(review_text.encode("utf-8")).hexdigest(),
                "review_preview": review_text[:200],
                "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            }

            # Cập nhật RAM
            checkpoint[key] = record
            results[idx] = scores

            # ✅ SỬA 5: lưu NGAY SAU MỖI COMMENT
            _atomic_save_json(LLM_ASPECT_CHECKPOINT_PATH, checkpoint)

            # ✅ SỬA 6: ghi log NGAY SAU MỖI COMMENT
            _append_aspect_log({
                "status": "done",
                "row_index": idx,
                "missing_pos": pos,
                "missing_total": len(missing_indices),
                "backend": ASPECT_SENTIMENT_BACKEND,
                "model_id": _current_aspect_model_id(),
                "latency_sec": round(time.time() - t0, 3),
                "cache_key": key,
                "review_preview": review_text[:120],
                "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            })

    except KeyboardInterrupt:
        print("⚠️ Interrupted. Checkpoint has already been saved after each completed comment.")
        raise

    except Exception as e:
        print(f"⚠️ Error: {type(e).__name__}: {e}")
        print("Checkpoint has already been saved after each completed comment.")
        raise

    # Bước 3: kiểm tra đủ
    missing_after = [i for i, x in enumerate(results) if x is None]

    if missing_after:
        raise RuntimeError(f"Still missing {len(missing_after)} aspect scores after run.")

    print(f"✅ Aspect sentiment done: {len(results)} comments")
    print(f"✅ Checkpoint saved: {LLM_ASPECT_CHECKPOINT_PATH}")

    return [r for r in results if r is not None]


# ============================================================
# Run backend
# ============================================================

if ASPECT_SENTIMENT_BACKEND == "llm":
    from config import load_config as load_module_config
    from aspect_sentiment import LLMAspectSentimentService

    print("Running LLM food-domain aspect sentiment with PER-COMMENT checkpoint...")
    aspect_model = LLMAspectSentimentService(load_module_config(REPO_ROOT))
    aspect_scores_list = score_reviews_with_per_comment_checkpoint(
        aspect_model=aspect_model,
        reviews=reviews_list,
    )

    print(f"✅ LLM aspect inference done: {len(feedback_proc)} reviews")
elif ASPECT_SENTIMENT_BACKEND == "phobert":
    print("Running batched PhoBERT aspect inference...")
    aspect_model = PhoBERTAspectSentiment(ASPECT_SENTIMENT_MODEL)
    aspect_scores_list = aspect_model.score_reviews_batch(reviews_list)
    print(f"✅ PhoBERT batched inference done: {len(feedback_proc)} reviews")
else:
    raise ValueError("ASPECT_SENTIMENT_BACKEND must be 'llm' or 'phobert'")

if len(aspect_scores_list) != len(feedback_proc):
    raise RuntimeError(
        f"aspect_scores_list length mismatch: "
        f"{len(aspect_scores_list)} scores vs {len(feedback_proc)} feedback rows"
    )

feedback_proc["aspect_scores"] = aspect_scores_list
feedback_proc["sentiment"] = feedback_proc["aspect_scores"].apply(classify_sentiment_from_aspects)
feedback_proc.head(3)

Running LLM food-domain aspect sentiment with PER-COMMENT checkpoint...
✅ Loaded external LLM aspect checkpoint: 515 items
Total comments: 879
Checkpoint hits: 879
Comments still need scoring: 0
Checkpoint path: D:\restaurant-kg-recommender-master\.cache\graphrag\llm_aspect_sentiment_checkpoint.json
Progress log path: D:\restaurant-kg-recommender-master\.cache\graphrag\llm_aspect_sentiment_progress.jsonl
✅ All comments loaded from checkpoint. No API call needed.
✅ LLM aspect inference done: 879 reviews


,store_id,store_key,store_name,rated_at,rating,feedback,source,review_id,name_norm,feedback_norm,aspect_scores,sentiment
0,25639,25639,Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn,None,4.6,"ngon đấy, Ngon xỉu",befood_comment,15034de1df0b4e9a,quan-mua-com-ga-bun-mien-tron-tay-son,"ngon đấy, ngon xỉu","{'food_quality': 0.9, 'service': 0.0, 'cleanli...",neutral
1,25639,25639,Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn,None,4.6,đặt phở cho bún?,befood_comment,8ea2b2b2c0447f44,quan-mua-com-ga-bun-mien-tron-tay-son,đặt phở cho bún?,"{'food_quality': 0.0, 'service': 0.0, 'cleanli...",neutral
2,25639,25639,Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn,None,4.6,bún ngan trộn của mình có cả gián ???,befood_comment,d2292fdad09da78a,quan-mua-com-ga-bun-mien-tron-tay-son,bún ngan trộn của mình có cả gián ???,"{'food_quality': 0.0, 'service': 0.0, 'cleanli...",neutral


In [47]:
print(menu_items)

     store_id store_key                                   store_name  \
0       25639     25639  Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn   
1       25639     25639  Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn   
2       25639     25639  Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn   
3       25639     25639  Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn   
4       25639     25639  Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn   
...       ...       ...                                          ...   
3240    53514     53514                Cơm 1985 - Phố Chợ Khâm Thiên   
3241    53514     53514                Cơm 1985 - Phố Chợ Khâm Thiên   
3242    53514     53514                Cơm 1985 - Phố Chợ Khâm Thiên   
3243    53514     53514                Cơm 1985 - Phố Chợ Khâm Thiên   
3244    53514     53514                Cơm 1985 - Phố Chợ Khâm Thiên   

     category_id category_name menu_item_id                         item_name  \
0         114762         COMBO       867932   Phở trộn

In [48]:
# ============================================================
# Menu-derived dish family extraction BY LLM
# MenuItem remains exact; DishFamily is the broad entity used for retrieval constraints.
#
# LLM chỉ trích dish_family từ item_name/category_name.
# Các thông số order_count, avg_price, like_count... vẫn aggregate từ menu_items thật.
#
# Có cache:
#   .cache/graphrag/llm_dish_family_cache.json
#   .cache/graphrag/llm_dish_family_progress.jsonl
#
# Chạy lại notebook sẽ đọc cache, không gọi lại LLM cho món đã xử lý.
# ============================================================

import os
import re
import json
import time
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


# ============================================================
# Config
# ============================================================

CACHE_DIR_PATH = Path(os.getenv("CACHE_DIR", ".cache/graphrag"))
CACHE_DIR_PATH.mkdir(parents=True, exist_ok=True)

LLM_DISH_FAMILY_CACHE_PATH = CACHE_DIR_PATH / "llm_dish_family_cache.json"
LLM_DISH_FAMILY_LOG_PATH = CACHE_DIR_PATH / "llm_dish_family_progress.jsonl"

LLM_DISH_FAMILY_BATCH_SIZE = int(os.getenv("LLM_DISH_FAMILY_BATCH_SIZE", "40"))
LLM_DISH_FAMILY_VERSION = os.getenv("LLM_DISH_FAMILY_VERSION", "v1")


# ============================================================
# Schema canonicalization
# ============================================================

def canonicalize_menu_df(menu_df: pd.DataFrame) -> pd.DataFrame:
    """
    Hỗ trợ cả schema gốc:
        restaurant_id, restaurant_name, restaurant_item_id
    và schema notebook:
        store_key, store_name, menu_item_id
    """
    if menu_df is None or menu_df.empty:
        return pd.DataFrame()

    df = menu_df.copy()

    if "store_key" not in df.columns:
        if "restaurant_id" in df.columns:
            df["store_key"] = df["restaurant_id"].astype(str)
        elif "store_id" in df.columns:
            df["store_key"] = df["store_id"].astype(str)
        else:
            raise KeyError("menu_df must have store_key, restaurant_id, or store_id")

    if "store_name" not in df.columns:
        if "restaurant_name" in df.columns:
            df["store_name"] = df["restaurant_name"]
        else:
            df["store_name"] = ""

    if "menu_item_id" not in df.columns:
        if "restaurant_item_id" in df.columns:
            df["menu_item_id"] = df["restaurant_item_id"].astype(str)
        elif "item_id" in df.columns:
            df["menu_item_id"] = df["item_id"].astype(str)
        else:
            df["menu_item_id"] = df.index.astype(str)

    for col, default in {
        "category_name": "",
        "item_name": "",
        "item_details": "",
        "price": 0,
        "old_price": 0,
        "order_count": 0,
        "like_count": 0,
        "dislike_count": 0,
        "item_image": "",
    }.items():
        if col not in df.columns:
            df[col] = default

    df["store_key"] = df["store_key"].astype(str)
    df["store_name"] = df["store_name"].fillna("").astype(str)
    df["menu_item_id"] = df["menu_item_id"].astype(str)
    df["category_name"] = df["category_name"].fillna("").astype(str)
    df["item_name"] = df["item_name"].fillna("").astype(str)
    df["item_details"] = df["item_details"].fillna("").astype(str)

    for col in ["price", "old_price", "order_count", "like_count", "dislike_count"]:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    return df


# ============================================================
# Cache helpers
# ============================================================

def _sha256_text(s: str) -> str:
    return hashlib.sha256(str(s).encode("utf-8")).hexdigest()


def _dish_family_cache_key(category_name: str, item_name: str, item_details: str = "") -> str:
    """
    Cache theo category + item + details.
    Không cache theo store_key để món giống nhau giữa nhiều quán dùng lại được.
    """
    raw = json.dumps(
        {
            "version": LLM_DISH_FAMILY_VERSION,
            "category_name": category_name,
            "item_name": item_name,
            "item_details": item_details[:300],
        },
        ensure_ascii=False,
        sort_keys=True,
    )
    return _sha256_text(raw)


def _atomic_save_json(path: Path, data: Any) -> None:
    tmp_path = Path(str(path) + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2, default=str)
    tmp_path.replace(path)


def _load_json_dict(path: Path) -> Dict[str, Any]:
    if not path.exists():
        return {}

    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, dict):
            return data

        print(f"⚠️ Cache file is not dict: {path}")
        return {}

    except Exception as e:
        print(f"⚠️ Could not load cache {path}: {type(e).__name__}: {e}")
        return {}


def _append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
        f.flush()


# ============================================================
# LLM prompt + parser
# ============================================================

DISH_FAMILY_SYSTEM = """Bạn là bộ chuẩn hóa nhóm món ăn Việt Nam cho hệ thống tìm quán ăn.

Nhiệm vụ:
Với mỗi menu item, hãy trả về dish_family ngắn gọn, tiếng Việt có dấu, chữ thường.

dish_family là nhóm món chính, không phải tên món đầy đủ.
Ví dụ:
- "Cơm gà sốt chua ngọt" -> "cơm gà"
- "Cơm rang dưa bò" -> "cơm rang"
- "Phở bò tái chín" -> "phở bò"
- "Tái chín" trong category "Phở Bò" -> "phở bò"
- "Bún chả truyền thống" -> "bún chả"
- "Bún bò Huế tô nhỏ" -> "bún bò huế"
- "Bún riêu cua đặc biệt" -> "bún riêu"
- "Miến trộn bò gà" -> "miến trộn"
- "Mì xào bò" -> "mì xào"
- "Bánh cuốn chả lẫn" -> "bánh cuốn"
- "Gà rán sốt cay" -> "gà rán"
- "Combo 1" trong category "Combo Gà Rán" -> "gà rán"

Quy tắc:
1. Ưu tiên món chính, bỏ qua đồ uống tặng kèm như coca, pepsi, trà tắc, hạt chia.
2. Bỏ qua từ quảng cáo như best seller, đặc biệt, quốc dân, must try.
3. Không trả về tên quá chi tiết như "cơm gà sốt chua ngọt"; hãy rút về "cơm gà".
4. Nếu item là topping/đồ thêm như "trứng ốp", "thịt bò", "giò", "mọc ốc" và không rõ món chính, trả về null.
5. Nếu là đồ uống/tráng miệng, vẫn có thể trả về nhóm như "nước ép", "sinh tố", "trà sữa", "cà phê" nếu rõ.
6. Chỉ trả JSON hợp lệ, không markdown.

Output bắt buộc:
{
  "items": [
    {"id": "...", "dish_family": "..." hoặc null}
  ]
}
"""


def _extract_json_object(raw: str) -> Dict[str, Any]:
    raw = str(raw).strip()
    raw = re.sub(r"^```json\s*", "", raw)
    raw = re.sub(r"^```\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)

    try:
        return json.loads(raw)
    except Exception:
        pass

    start = raw.find("{")
    end = raw.rfind("}")

    if start >= 0 and end > start:
        return json.loads(raw[start:end + 1])

    raise ValueError(f"Cannot parse JSON from LLM output: {raw[:500]}")


def _normalize_family_label(x: Any) -> Optional[str]:
    if x is None:
        return None

    s = str(x).strip().lower()
    s = re.sub(r"\s+", " ", s)

    if not s or s in {"none", "null", "n/a", "na", "không rõ", "khong ro"}:
        return None

    # sửa vài variant hay gặp
    alias = {
        "mỳ": "mì",
        "mỳ xào": "mì xào",
        "mỳ trộn": "mì trộn",
        "mỳ ý": "mỳ ý",
        "com ga": "cơm gà",
        "com rang": "cơm rang",
        "pho bo": "phở bò",
        "bun cha": "bún chả",
    }

    return alias.get(s, s)


def _call_llm_dish_family_batch(items: List[Dict[str, Any]]) -> Dict[str, Optional[str]]:
    """
    Gọi LLM cho một batch item.
    Trả về dict: id -> dish_family hoặc None.
    """
    if not items:
        return {}

    llm = get_llm()

    user_payload = {
        "items": [
            {
                "id": str(x["id"]),
                "category_name": str(x.get("category_name", ""))[:120],
                "item_name": str(x.get("item_name", ""))[:180],
                "item_details": str(x.get("item_details", ""))[:250],
            }
            for x in items
        ]
    }

    resp = llm.invoke([
        {"role": "system", "content": DISH_FAMILY_SYSTEM},
        {
            "role": "user",
            "content": "Hãy trích dish_family cho các menu item sau:\n"
                       + json.dumps(user_payload, ensure_ascii=False),
        },
    ])

    raw = getattr(resp, "content", resp)
    parsed = _extract_json_object(str(raw))

    out = {}

    for obj in parsed.get("items", []):
        item_id = str(obj.get("id", "")).strip()
        if not item_id:
            continue
        out[item_id] = _normalize_family_label(obj.get("dish_family"))

    return out


# ============================================================
# LLM dish family extraction with cache
# ============================================================

def add_llm_dish_family(menu_df: pd.DataFrame) -> pd.DataFrame:
    """
    Thêm cột dish_family vào từng menu item bằng LLM + cache.
    """
    df = canonicalize_menu_df(menu_df)

    if df.empty:
        df["dish_family"] = []
        return df

    cache = _load_json_dict(LLM_DISH_FAMILY_CACHE_PATH)

    # Unique theo category + item + details để giảm số lần gọi LLM.
    unique_rows = (
        df[["category_name", "item_name", "item_details"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    unique_rows["cache_key"] = unique_rows.apply(
        lambda r: _dish_family_cache_key(
            r["category_name"],
            r["item_name"],
            r["item_details"],
        ),
        axis=1,
    )

    missing = []

    for _, row in unique_rows.iterrows():
        key = row["cache_key"]

        if key not in cache:
            missing.append({
                "id": key,
                "category_name": row["category_name"],
                "item_name": row["item_name"],
                "item_details": row["item_details"],
            })

    print(f"Total menu rows: {len(df)}")
    print(f"Unique item/category/detail keys: {len(unique_rows)}")
    print(f"LLM dish_family cache hits: {len(unique_rows) - len(missing)}")
    print(f"LLM dish_family missing: {len(missing)}")
    print(f"Cache path: {LLM_DISH_FAMILY_CACHE_PATH.resolve()}")
    print(f"Log path: {LLM_DISH_FAMILY_LOG_PATH.resolve()}")
    print(f"Batch size: {LLM_DISH_FAMILY_BATCH_SIZE}")

    if missing:
        try:
            for start in range(0, len(missing), LLM_DISH_FAMILY_BATCH_SIZE):
                batch = missing[start:start + LLM_DISH_FAMILY_BATCH_SIZE]
                t0 = time.time()

                result = _call_llm_dish_family_batch(batch)

                for item in batch:
                    key = item["id"]
                    family = _normalize_family_label(result.get(key))

                    cache[key] = {
                        "dish_family": family,
                        "category_name": item["category_name"],
                        "item_name": item["item_name"],
                        "item_details_preview": item["item_details"][:200],
                        "version": LLM_DISH_FAMILY_VERSION,
                        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                    }

                # Save sau mỗi batch để mất điện chạy tiếp.
                _atomic_save_json(LLM_DISH_FAMILY_CACHE_PATH, cache)

                _append_jsonl(LLM_DISH_FAMILY_LOG_PATH, {
                    "status": "done",
                    "batch_start": start,
                    "batch_size": len(batch),
                    "missing_total": len(missing),
                    "latency_sec": round(time.time() - t0, 3),
                    "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                })

                print(f"✅ LLM dish_family batch done: {start + len(batch)}/{len(missing)}")

        except KeyboardInterrupt:
            print("⚠️ Interrupted. Cache has been saved after each completed batch.")
            raise

        except Exception as e:
            print(f"⚠️ LLM dish_family extraction failed: {type(e).__name__}: {e}")
            print("Cache has been saved after each completed batch.")
            raise

    # Map cache về từng row menu.
    def _lookup_family(row) -> Optional[str]:
        key = _dish_family_cache_key(
            row["category_name"],
            row["item_name"],
            row["item_details"],
        )
        saved = cache.get(key, {})
        return _normalize_family_label(saved.get("dish_family"))

    df["dish_family"] = df.apply(_lookup_family, axis=1)

    return df


# ============================================================
# Build aggregate dish_families
# ============================================================

def build_menu_dish_families(menu_df: pd.DataFrame) -> pd.DataFrame:
    if menu_df is None or menu_df.empty:
        return pd.DataFrame(columns=[
            "store_key",
            "store_name",
            "dish_family",
            "total_menu_items",
            "avg_price",
            "min_price",
            "max_price",
            "order_count",
            "like_count",
            "dislike_count",
            "menu_item_ids",
            "example_items",
        ])

    df = menu_df.copy()

    if "dish_family" not in df.columns:
        df = add_llm_dish_family(df)

    df = df[df["dish_family"].notna() & df["dish_family"].astype(str).str.strip().ne("")].copy()

    if df.empty:
        return pd.DataFrame(columns=[
            "store_key",
            "store_name",
            "dish_family",
            "total_menu_items",
            "avg_price",
            "min_price",
            "max_price",
            "order_count",
            "like_count",
            "dislike_count",
            "menu_item_ids",
            "example_items",
        ])

    family_df = (
        df.groupby(["store_key", "dish_family"], as_index=False)
        .agg(
            store_name=("store_name", "first"),
            total_menu_items=("menu_item_id", "count"),
            avg_price=("price", "mean"),
            min_price=("price", "min"),
            max_price=("price", "max"),
            order_count=("order_count", "sum"),
            like_count=("like_count", "sum"),
            dislike_count=("dislike_count", "sum"),
            menu_item_ids=("menu_item_id", lambda s: [str(x) for x in s]),
            example_items=("item_name", lambda s: list(dict.fromkeys([str(x) for x in s]))[:8]),
        )
        .reset_index()
    )

    family_df["avg_price"] = family_df["avg_price"].round(2)
    family_df["min_price"] = family_df["min_price"].round(2)
    family_df["max_price"] = family_df["max_price"].round(2)

    for col in ["order_count", "like_count", "dislike_count"]:
        family_df[col] = family_df[col].round(0).astype(int)

    family_df = family_df.sort_values(
        ["order_count", "store_key", "dish_family"],
        ascending=[False, True, True],
    ).reset_index(drop=True)

    return family_df


# ============================================================
# Neo4j upsert
# ============================================================

def upsert_dish_families(client, family_df: pd.DataFrame):
    """Write broad DishFamily nodes and (Restaurant)-[:SERVES_FAMILY]->(DishFamily) edges."""
    if family_df is None or family_df.empty:
        print("No dish families to upsert")
        return

    rows = family_df.rename(columns={"dish_family": "name"}).to_dict("records")

    client.run("""
    UNWIND $rows AS row
    MATCH (r:Restaurant {store_key: row.store_key})
    MERGE (d:DishFamily {name: row.name})
    MERGE (r)-[s:SERVES_FAMILY]->(d)
    SET s.menu_item_count = row.total_menu_items,
        s.like_count = row.like_count,
        s.dislike_count = row.dislike_count,
        s.order_count = row.order_count,
        s.avg_price = row.avg_price,
        s.min_price = row.min_price,
        s.max_price = row.max_price,
        s.menu_item_ids = row.menu_item_ids,
        s.example_items = row.example_items,
        s.updated_at = datetime()
    """, {"rows": rows})

    print(f"Upserted {len(family_df)} dish family edges")


def upsert_menu_items_graph(client, menu_df: pd.DataFrame):
    """Write concrete MenuItem nodes linked to Restaurant and MenuCategory."""
    if menu_df is None or menu_df.empty:
        print("No menu items to upsert")
        return

    df = canonicalize_menu_df(menu_df)
    rows = df.to_dict("records")

    client.run("""
    UNWIND $rows AS row
    MATCH (r:Restaurant {store_key: row.store_key})
    MERGE (mi:MenuItem {menu_item_id: row.menu_item_id})
    SET mi.name = row.item_name,
        mi.details = row.item_details,
        mi.price = row.price,
        mi.old_price = row.old_price,
        mi.order_count = row.order_count,
        mi.like_count = row.like_count,
        mi.dislike_count = row.dislike_count,
        mi.item_image = row.item_image,
        mi.updated_at = datetime()
    MERGE (r)-[:HAS_MENU_ITEM]->(mi)
    FOREACH (_ IN CASE WHEN row.category_name IS NULL OR row.category_name = '' THEN [] ELSE [1] END |
        MERGE (mc:MenuCategory {name: row.category_name})
        MERGE (r)-[:HAS_MENU_CATEGORY]->(mc)
        MERGE (mi)-[:IN_MENU_CATEGORY]->(mc)
    )
    """, {"rows": rows})

    print(f"Upserted {len(df)} menu items")


# ============================================================
# Run
# ============================================================

menu_items = add_llm_dish_family(menu_items)

dish_families = build_menu_dish_families(menu_items)

dish_families_by_store = (
    dish_families.groupby("store_key")["dish_family"]
    .apply(lambda s: sorted(set(s)))
    .to_dict()
    if not dish_families.empty
    else {}
)

summary["store_key"] = summary["store_key"].astype(str)
summary["dish_families"] = (
    summary["store_key"]
    .map(dish_families_by_store)
    .apply(lambda x: x if isinstance(x, list) else [])
)

if dish_families.empty:
    print("No dish families found from menu")
else:
    print("✅ LLM dish family extraction done")
    print(f"Menu items: {len(menu_items)}")
    print(f"Detected menu items: {menu_items['dish_family'].notna().sum()}")
    print(f"Dish family edges: {len(dish_families)}")
    print(f"Unique dish_family names: {dish_families['dish_family'].nunique()}")
    print(f"Stores with dish families: {dish_families['store_key'].nunique()}")

    print("\nAll unique dish_family names:")
    print(sorted(dish_families["dish_family"].dropna().unique().tolist()))

    print("\nAll dish families by store:")
    display(dish_families[[
        "store_key",
        "store_name",
        "dish_family",
        "total_menu_items",
        "order_count",
        "like_count",
        "avg_price",
        "min_price",
        "max_price",
        "example_items",
    ]])

    print("\nTop 30 dish families by order_count:")
    display(dish_families.nlargest(30, "order_count")[[
        "store_key",
        "store_name",
        "dish_family",
        "total_menu_items",
        "order_count",
        "like_count",
        "avg_price",
        "example_items",
    ]])

    print("\nDebug: menu items with LLM dish_family")
    display(menu_items[[
        "store_key",
        "store_name",
        "category_name",
        "menu_item_id",
        "item_name",
        "dish_family",
        "order_count",
        "like_count",
        "price",
    ]])

Total menu rows: 3245
Unique item/category/detail keys: 3186
LLM dish_family cache hits: 3186
LLM dish_family missing: 0
Cache path: D:\restaurant-kg-recommender-master\.cache\graphrag\llm_dish_family_cache.json
Log path: D:\restaurant-kg-recommender-master\.cache\graphrag\llm_dish_family_progress.jsonl
Batch size: 40
✅ LLM dish family extraction done
Menu items: 3245
Detected menu items: 3198
Dish family edges: 1202
Unique dish_family names: 479
Stores with dish families: 196

All unique dish_family names:
['ba chỉ heo nướng', 'ba chỉ luộc', 'bia', 'burger', 'burger bò', 'burger gà', 'burger tôm', 'bánh bao chiên', 'bánh bột lọc', 'bánh chocolate', 'bánh chuối', 'bánh chuối phô mai nướng', 'bánh cuốn', 'bánh cuốn trứng', 'bánh cuộn', 'bánh giò', 'bánh gà', 'bánh gạo', 'bánh gạo chiên', 'bánh gạo lắc', 'bánh gối', 'bánh milo', 'bánh mì', 'bánh mì muối ớt', 'bánh mì nướng sốt', 'bánh mì sốt vang', 'bánh mì thịt nướng', 'bánh quy', 'bánh quy bơ sốt cam', 'bánh quy chấm sốt', 'bánh rán mặ

,store_key,store_name,dish_family,total_menu_items,order_count,like_count,avg_price,min_price,max_price,example_items
0,11305,Quán Cơm 76,cơm gà,10,165061,702,47000.00,42000.0,60000.0,"[Cơm Gà Sốt Chua Ngọt, Cơm Gà Chiên Mắm, Cơm G..."
1,11305,Quán Cơm 76,cơm sườn,1,84011,81,45000.00,45000.0,45000.0,"[Cơm Sườn Xào Chua Ngọt, Trứng Ốp La]"
2,11305,Quán Cơm 76,cơm thịt,7,54763,382,49000.00,42000.0,60000.0,"[Cơm Thịt Rang, Cơm Thịt Xào Cần Tỏi, Cơm Thịt..."
3,9606,Cơm Ngon Quang Thắng,cơm gà,13,30715,192,55607.69,45000.0,65000.0,"[Cơm Gà Chua Ngọt, Cơm Gà xào sả ớt, Cơm 2 Miế..."
4,7635,Happy Food Cơm Gà Ba Vì,cơm gà,10,27287,245,62000.00,55000.0,65000.0,"[Combo Cơm Gà Lọc Xương Sốt BBQ + Pepsi, Combo..."
...,...,...,...,...,...,...,...,...,...,...
1197,96492,Bánh Cuốn Dẻo Gia Truyền Bà Minh - Lương Đình Của,bánh cuốn,6,0,0,48333.33,25000.0,105000.0,"[Bánh chay, Bánh nhân thịt nấm không chả, Bánh..."
1198,97742,"NaBi Bún Chả Quạt, Cơm Tấm & Cơm Văn Phòng - T...",cơm chả lá lốt,1,0,0,58000.00,58000.0,58000.0,[Cơm chả lá lốt vị nhà làm]
1199,99617,Cơm Đốt Thố Thái Lan - Xã Đàn,bắp cải luộc,1,0,0,69000.00,69000.0,69000.0,[Bắp Cải Luộc Chấm Trứng]
1200,99617,Cơm Đốt Thố Thái Lan - Xã Đàn,cá nục hoa,1,0,0,81000.00,81000.0,81000.0,[Cá nục hoa sốt chanh leo]



Top 30 dish families by order_count:


,store_key,store_name,dish_family,total_menu_items,order_count,like_count,avg_price,example_items
0,11305,Quán Cơm 76,cơm gà,10,165061,702,47000.00,"[Cơm Gà Sốt Chua Ngọt, Cơm Gà Chiên Mắm, Cơm G..."
1,11305,Quán Cơm 76,cơm sườn,1,84011,81,45000.00,"[Cơm Sườn Xào Chua Ngọt, Trứng Ốp La]"
2,11305,Quán Cơm 76,cơm thịt,7,54763,382,49000.00,"[Cơm Thịt Rang, Cơm Thịt Xào Cần Tỏi, Cơm Thịt..."
3,9606,Cơm Ngon Quang Thắng,cơm gà,13,30715,192,55607.69,"[Cơm Gà Chua Ngọt, Cơm Gà xào sả ớt, Cơm 2 Miế..."
4,7635,Happy Food Cơm Gà Ba Vì,cơm gà,10,27287,245,62000.00,"[Combo Cơm Gà Lọc Xương Sốt BBQ + Pepsi, Combo..."
5,11493,Tun Tun - Bún Hải Sản Hoa Quả - Bùi Ngọc Dương,bún hải sản,4,19734,130,65000.00,"[Bún Hải Sản Hoa Quả Size S (Cỡ Nhỏ), Bún Hải ..."
6,4486,"Bento Delichi (by Cooky) - Cơm Gà Mắm Tỏi, Xối...",cơm gà,30,19069,147,61056.67,[Cơm Gà Xối Mỡ (Đùi Tỏi/Má Đùi) + Canh Cải Ngọ...
7,1279,Gạch Quán - Bánh Tráng Cuốn Thịt Heo,bánh tráng cuốn,2,18227,76,80000.00,"[Bánh tráng cuốn thịt quay, Bánh tráng cuốn th..."
8,17700,Bún Chả Xưa - Lĩnh Nam,bún chả,8,16024,134,39375.00,"[Bún chả Viên, Bún Chả Thịt Nướng, Bún Chả Nem..."
9,9606,Cơm Ngon Quang Thắng,cơm thịt,8,14614,91,52500.00,"[Cơm Thịt xào hành tây, Cơm Thịt Kho Tàu, Cơm ..."



Debug: menu items with LLM dish_family


,store_key,store_name,category_name,menu_item_id,item_name,dish_family,order_count,like_count,price
0,25639,Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn,COMBO,867932,Phở trộn bò và gà - 1 Coca cola,phở trộn,100,2,49000.0
1,25639,Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn,COMBO,867933,Miến trộn bò và gà - 1 coca cola,miến trộn,85,1,49000.0
2,25639,Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn,COMBO,867935,Bún trộn bò và gà - 1 Sprite,bún trộn,40,0,49000.0
3,25639,Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn,BÚN MIẾN PHỞ,867936,Miến Trộn Bò Gà,miến trộn,934,10,45000.0
4,25639,Quán Mùa - Cơm Gà & Bún Miến Trộn - Tây Sơn,BÚN MIẾN PHỞ,867937,Phở Trộn Bò Gà,phở trộn,759,5,45000.0
...,...,...,...,...,...,...,...,...,...
3240,53514,Cơm 1985 - Phố Chợ Khâm Thiên,Cơm Chiên,5377853,Cơm chiên gà,cơm chiên,2,0,79000.0
3241,53514,Cơm 1985 - Phố Chợ Khâm Thiên,Cơm Chiên,5377854,Cơm chiên bít tết 1985,cơm chiên,4,0,100000.0
3242,53514,Cơm 1985 - Phố Chợ Khâm Thiên,Mỳ,5377855,Mỳ ý sốt bò băm,mỳ ý,6,0,79000.0
3243,53514,Cơm 1985 - Phố Chợ Khâm Thiên,Mỳ,5377856,Mỳ ý xốt kem bò,mỳ ý,9,0,79000.0


In [49]:
if dish_families.empty:
    print("No dish families found from menu")
else:
    print(f"Top dish families found:\n{dish_families.nlargest(10, 'order_count')[['store_key','dish_family','order_count','avg_price']].to_string(index=False)}")


Top dish families found:
store_key     dish_family  order_count  avg_price
    11305          cơm gà       165061   47000.00
    11305        cơm sườn        84011   45000.00
    11305        cơm thịt        54763   49000.00
     9606          cơm gà        30715   55607.69
     7635          cơm gà        27287   62000.00
    11493     bún hải sản        19734   65000.00
     4486          cơm gà        19069   61056.67
     1279 bánh tráng cuốn        18227   80000.00
    17700         bún chả        16024   39375.00
     9606        cơm thịt        14614   52500.00


In [50]:
dish_families.head(20)

,index,store_key,dish_family,store_name,total_menu_items,avg_price,min_price,max_price,order_count,like_count,dislike_count,menu_item_ids,example_items
0,52,11305,cơm gà,Quán Cơm 76,10,47000.00,42000.0,60000.0,165061,702,36,"[388175, 388176, 388180, 388184, 388185, 38818...","[Cơm Gà Sốt Chua Ngọt, Cơm Gà Chiên Mắm, Cơm G..."
1,53,11305,cơm sườn,Quán Cơm 76,1,45000.00,45000.0,45000.0,84011,81,6,[388178],"[Cơm Sườn Xào Chua Ngọt, Trứng Ốp La]"
2,54,11305,cơm thịt,Quán Cơm 76,7,49000.00,42000.0,60000.0,54763,382,26,"[388177, 388179, 388182, 388183, 388189, 38819...","[Cơm Thịt Rang, Cơm Thịt Xào Cần Tỏi, Cơm Thịt..."
3,1158,9606,cơm gà,Cơm Ngon Quang Thắng,13,55607.69,45000.0,65000.0,30715,192,35,"[2695576, 2695579, 2695581, 2695588, 2695599, ...","[Cơm Gà Chua Ngọt, Cơm Gà xào sả ớt, Cơm 2 Miế..."
4,936,7635,cơm gà,Happy Food Cơm Gà Ba Vì,10,62000.00,55000.0,65000.0,27287,245,29,"[7617183, 7617185, 230336, 230338, 230335, 342...","[Combo Cơm Gà Lọc Xương Sốt BBQ + Pepsi, Combo..."
5,87,11493,bún hải sản,Tun Tun - Bún Hải Sản Hoa Quả - Bùi Ngọc Dương,4,65000.00,50000.0,80000.0,19734,130,7,"[395309, 395307, 395310, 395308]","[Bún Hải Sản Hoa Quả Size S (Cỡ Nhỏ), Bún Hải ..."
6,756,4486,cơm gà,"Bento Delichi (by Cooky) - Cơm Gà Mắm Tỏi, Xối...",30,61056.67,35000.0,75000.0,19069,147,21,"[10360352, 11780637, 94965, 94969, 94974, 9497...",[Cơm Gà Xối Mỡ (Đùi Tỏi/Má Đùi) + Canh Cải Ngọ...
7,206,1279,bánh tráng cuốn,Gạch Quán - Bánh Tráng Cuốn Thịt Heo,2,80000.00,75000.0,85000.0,18227,76,10,"[27424, 6945147]","[Bánh tráng cuốn thịt quay, Bánh tráng cuốn th..."
8,423,17700,bún chả,Bún Chả Xưa - Lĩnh Nam,8,39375.00,35000.0,45000.0,16024,134,96,"[589529, 589534, 589535, 589532, 589533, 58953...","[Bún chả Viên, Bún Chả Thịt Nướng, Bún Chả Nem..."
9,1161,9606,cơm thịt,Cơm Ngon Quang Thắng,8,52500.00,45000.0,65000.0,14614,91,20,"[2695583, 2695587, 2695597, 2695604, 5600268, ...","[Cơm Thịt xào hành tây, Cơm Thịt Kho Tàu, Cơm ..."


In [51]:
attr_rows = []
for store_key, grp in feedback_proc.groupby("store_key"):
    bucket = defaultdict(list)
    for asp_map in grp["aspect_scores"]:
        for k, v in asp_map.items():
            bucket[k].append(v)
    for aspect, vals in bucket.items():
        attr_rows.append({
            "store_key": store_key,
            "attribute_type": aspect,
            "attribute_score": round(float(np.mean(vals)), 3),
            "sample_count": len(vals)
        })
restaurant_attrs = pd.DataFrame(attr_rows)
display(restaurant_attrs.head(10))


,store_key,attribute_type,attribute_score,sample_count
0,10227,food_quality,-0.100,7
1,10227,service,-0.100,7
2,10227,cleanliness,0.000,7
3,10227,packaging,-0.200,7
4,10227,price,0.143,7
5,10227,space,0.000,7
6,10227,speed,0.000,7
7,102294,food_quality,0.000,1
8,102294,service,0.000,1
9,102294,cleanliness,0.000,1


In [52]:
def build_text_unit_text(row: pd.Series, restaurant_name: str = "") -> str:
    aspects = ", ".join(f"{k}={v:+.2f}" for k, v in (row["aspect_scores"] or {}).items())
    return (
        f"Tên quán: {restaurant_name or row['store_name']}\n"
        f"Nguồn: {row['source']}\n"
        f"Rating người dùng: {row['rating']}\n"
        f"Sentiment tổng hợp: {row['sentiment']}\n"
        f"Aspect sentiment: {aspects}\n"
        f"Nội dung review: {row['feedback']}"
    )

name_map = summary.set_index("store_key")["name"].to_dict()
text_units = feedback_proc.copy()
text_units["text_unit_id"] = "tu_" + text_units["review_id"].astype(str)
text_units["chunk_text"] = [build_text_unit_text(r, name_map.get(r["store_key"], r["store_name"])) for _, r in text_units.iterrows()]
review_chunks = text_units.copy()  # backward-compatible alias for functions below
text_units[["store_key", "text_unit_id", "chunk_text"]].head(3)


,store_key,text_unit_id,chunk_text
0,25639,tu_15034de1df0b4e9a,Tên quán: Quán Mùa - Cơm Gà & Bún Miến Trộn - ...
1,25639,tu_8ea2b2b2c0447f44,Tên quán: Quán Mùa - Cơm Gà & Bún Miến Trộn - ...
2,25639,tu_d2292fdad09da78a,Tên quán: Quán Mùa - Cơm Gà & Bún Miến Trộn - ...


## 4.1. Text chunking chuẩn GraphRAG (sliding window over long reviews)

In [53]:
def chunk_review_text(
    text: str,
    chunk_size: int = 400,
    overlap: int = 80,
) -> List[str]:
    """FIX (Vấn đề 1): Proper sliding window text chunking.

    Thay vì review = 1 TextUnit, chia review dài thành nhiều chunk:
    - chunk_size ~400 ký tự ≈ 300-500 token Vietnamese
    - overlap 80 ký tự để giữ context continuity giữa các chunk
    - Review ngắn (< chunk_size): vẫn ra 1 chunk, không thay đổi behavior
    """
    text = text.strip()
    if not text:
        return []
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        if end >= len(text):
            chunks.append(text[start:])
            break
        # Ưu tiên cắt tại ranh giới câu
        boundary = max(
            text.rfind(".", start, end),
            text.rfind("!", start, end),
            text.rfind("?", start, end),
            text.rfind("\n", start, end),
        )
        if boundary > start + overlap:
            end = boundary + 1
        chunks.append(text[start:end])
        start = end - overlap
    return [c.strip() for c in chunks if c.strip()]


def build_text_units_with_chunking(
    feedback_proc_df: pd.DataFrame,
    name_map: Dict[str, str],
    chunk_size: int = 400,
    overlap: int = 80,
) -> pd.DataFrame:
    """Build TextUnit DataFrame với proper sliding window chunking.

    Mỗi review dài có thể tạo ra nhiều TextUnit chunk.
    text_unit_id = review_id + "_chunk{i}" cho multi-chunk reviews.
    """
    rows = []
    for _, r in feedback_proc_df.iterrows():
        review_text = str(r["feedback"])
        aspect_str = ", ".join(
            f"{k}={v:+.2f}" for k, v in (r["aspect_scores"] or {}).items()
        )
        header = (
            f"Tên quán: {name_map.get(r['store_key'], r['store_name'])}\n"
            f"Nguồn: {r['source']}\n"
            f"Rating người dùng: {r['rating']}\n"
            f"Sentiment tổng hợp: {r['sentiment']}\n"
            f"Aspect sentiment: {aspect_str}\n"
            f"Nội dung review: "
        )
        chunks = chunk_review_text(review_text, chunk_size=chunk_size, overlap=overlap)
        if not chunks:
            chunks = ["(empty review)"]

        for i, chunk in enumerate(chunks):
            chunk_id = f"{r['review_id']}_{i}" if len(chunks) > 1 else r["review_id"]
            rows.append({
                "text_unit_id": "tu_" + chunk_id,
                "review_id": r["review_id"],
                "store_key": r["store_key"],
                "store_name": r["store_name"],
                "rating": r["rating"],
                "rated_at": r.get("rated_at", None),
                "sentiment": r["sentiment"],
                "aspect_scores": r["aspect_scores"],
                "source": r["source"],
                "feedback": review_text,
                "chunk_text": header + chunk,
                "chunk_index": i,
                "n_chunks": len(chunks),
            })
    return pd.DataFrame(rows)


# Rebuild text_units with proper chunking (replaces flat 1-review-1-unit)
text_units = build_text_units_with_chunking(feedback_proc, name_map)
review_chunks = text_units.copy()
multi_chunk = (text_units["n_chunks"] > 1).sum()
print(f"✅ TextUnits after chunking: {len(text_units)} chunks from {len(feedback_proc)} reviews")
print(f"   Reviews producing multiple chunks: {multi_chunk}")
text_units[["store_key", "text_unit_id", "chunk_index", "n_chunks"]].head(5)

✅ TextUnits after chunking: 881 chunks from 879 reviews
   Reviews producing multiple chunks: 4


,store_key,text_unit_id,chunk_index,n_chunks
0,25639,tu_15034de1df0b4e9a,0,1
1,25639,tu_8ea2b2b2c0447f44,0,1
2,25639,tu_d2292fdad09da78a,0,1
3,25639,tu_d7985d984f2378be,0,1
4,97742,tu_037157be32ed104c,0,1


## 4.2. Optional LLM entity/relation extraction t? review text

In [54]:

import os
import json
import time
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd


# =========================
# Checkpoint paths
# =========================
CACHE_DIR_PATH = Path(os.getenv("CACHE_DIR", ".cache/graphrag"))
CACHE_DIR_PATH.mkdir(parents=True, exist_ok=True)

LLM_GRAPH_EXTRACTION_CHECKPOINT_PATH = CACHE_DIR_PATH / "llm_graph_extraction_checkpoint.json"
LLM_GRAPH_EXTRACTION_LOG_PATH = CACHE_DIR_PATH / "llm_graph_extraction_progress.jsonl"

# Tăng version này nếu bạn đổi prompt/schema extraction và muốn chạy lại từ đầu.
LLM_GRAPH_EXTRACTION_CHECKPOINT_VERSION = os.getenv("LLM_GRAPH_EXTRACTION_CHECKPOINT_VERSION", "v1")


def _graph_extraction_model_id() -> str:
    return str(
        globals().get(
            "LLM_GRAPH_EXTRACTION_MODEL_ID",
            os.getenv("LLM_GRAPH_EXTRACTION_MODEL_ID", "unknown_graph_extraction_model"),
        )
    )


def _safe_json_dump(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False, default=str, sort_keys=True)


def _atomic_save_json(path: Path, data: Any) -> None:
    tmp_path = Path(str(path) + ".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2, default=str)
    tmp_path.replace(path)


def _append_jsonl(path: Path, record: Dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
        f.flush()


def _load_json_dict(path: Path) -> Dict[str, Any]:
    if not path.exists():
        print(f"ℹ️ No checkpoint found: {path}")
        return {}

    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        if not isinstance(data, dict):
            print(f"⚠️ Checkpoint is not dict, starting fresh: {path}")
            return {}

        print(f"✅ Loaded checkpoint: {len(data)} items -> {path}")
        return data

    except Exception as e:
        print(f"⚠️ Could not load checkpoint {path}: {type(e).__name__}: {e}")
        return {}


def _text_unit_cache_key(row: pd.Series) -> str:
    """
    Key phụ thuộc vào:
    - model extraction
    - checkpoint version
    - text_unit_id
    - store_key
    - text content

    Nếu TextUnit giống y hệt thì lần sau không gọi lại LLM.
    """
    text_unit_id = str(row.get("text_unit_id", ""))
    store_key = str(row.get("store_key", ""))
    text = str(row.get("chunk_text", row.get("text", "")) or "")

    raw = {
        "model_id": _graph_extraction_model_id(),
        "checkpoint_version": LLM_GRAPH_EXTRACTION_CHECKPOINT_VERSION,
        "text_unit_id": text_unit_id,
        "store_key": store_key,
        "text": text,
    }

    return hashlib.sha256(
        json.dumps(raw, ensure_ascii=False, sort_keys=True).encode("utf-8")
    ).hexdigest()


def _df_to_records(df: Any) -> List[Dict[str, Any]]:
    if df is None:
        return []

    if isinstance(df, pd.DataFrame):
        if df.empty:
            return []
        return df.to_dict("records")

    if isinstance(df, list):
        return [x for x in df if isinstance(x, dict)]

    return []


def _records_to_df(records: List[Dict[str, Any]]) -> pd.DataFrame:
    if not records:
        return pd.DataFrame()
    return pd.DataFrame(records)


def _extract_one_text_unit(extractor: Any, row: pd.Series) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    Chạy LLM extraction cho đúng 1 TextUnit.
    Cách ít phụ thuộc nhất: gọi lại API batch hiện có bằng DataFrame 1 dòng.
    """
    one_df = pd.DataFrame([row.to_dict()])

    entity_df, relation_df = extractor.extract_text_units(one_df)

    entity_records = _df_to_records(entity_df)
    relation_records = _df_to_records(relation_df)

    return entity_records, relation_records


def extract_text_units_with_checkpoint(
    extractor: Any,
    text_units_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    checkpoint = _load_json_dict(LLM_GRAPH_EXTRACTION_CHECKPOINT_PATH)

    if text_units_df is None or text_units_df.empty:
        print("⚠️ text_units is empty. Skip LLM graph extraction.")
        return pd.DataFrame(), pd.DataFrame()

    all_entities: List[Dict[str, Any]] = []
    all_relations: List[Dict[str, Any]] = []

    missing_indices = []
    hit_count = 0

    # Bước 1: load checkpoint cũ
    for idx, row in text_units_df.iterrows():
        key = _text_unit_cache_key(row)

        if key in checkpoint and isinstance(checkpoint[key], dict):
            saved = checkpoint[key]

            all_entities.extend(saved.get("entities", []) or [])
            all_relations.extend(saved.get("relations", []) or [])

            hit_count += 1
        else:
            missing_indices.append(idx)

    print(f"Total TextUnits: {len(text_units_df)}")
    print(f"Checkpoint hits: {hit_count}")
    print(f"TextUnits still need LLM extraction: {len(missing_indices)}")
    print(f"Checkpoint path: {LLM_GRAPH_EXTRACTION_CHECKPOINT_PATH.resolve()}")
    print(f"Progress log path: {LLM_GRAPH_EXTRACTION_LOG_PATH.resolve()}")

    if not missing_indices:
        print("✅ All TextUnits loaded from checkpoint. No LLM call needed.")
        return _records_to_df(all_entities), _records_to_df(all_relations)

    # Bước 2: chạy từng TextUnit còn thiếu, xong cái nào lưu cái đó
    try:
        for pos, idx in enumerate(tqdm(missing_indices, desc="LLM graph extraction per TextUnit"), start=1):
            row = text_units_df.loc[idx]
            key = _text_unit_cache_key(row)

            text_unit_id = str(row.get("text_unit_id", ""))
            store_key = str(row.get("store_key", ""))
            chunk_text = str(row.get("chunk_text", row.get("text", "")) or "")

            t0 = time.time()

            entity_records, relation_records = _extract_one_text_unit(extractor, row)

            record = {
                "status": "done",
                "model_id": _graph_extraction_model_id(),
                "checkpoint_version": LLM_GRAPH_EXTRACTION_CHECKPOINT_VERSION,
                "text_unit_id": text_unit_id,
                "store_key": store_key,
                "text_sha": hashlib.sha256(chunk_text.encode("utf-8")).hexdigest(),
                "text_preview": chunk_text[:200],
                "entities": entity_records,
                "relations": relation_records,
                "entity_count": len(entity_records),
                "relation_count": len(relation_records),
                "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            }

            # Cập nhật RAM
            checkpoint[key] = record
            all_entities.extend(entity_records)
            all_relations.extend(relation_records)

            # ✅ Lưu ngay sau từng TextUnit
            _atomic_save_json(LLM_GRAPH_EXTRACTION_CHECKPOINT_PATH, checkpoint)

            # ✅ Log ngay sau từng TextUnit
            _append_jsonl(LLM_GRAPH_EXTRACTION_LOG_PATH, {
                "status": "done",
                "row_index": int(idx) if str(idx).isdigit() else str(idx),
                "missing_pos": pos,
                "missing_total": len(missing_indices),
                "backend": "llm_graph_extraction",
                "model_id": _graph_extraction_model_id(),
                "text_unit_id": text_unit_id,
                "store_key": store_key,
                "entity_count": len(entity_records),
                "relation_count": len(relation_records),
                "latency_sec": round(time.time() - t0, 3),
                "cache_key": key,
                "text_preview": chunk_text[:120],
                "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            })

    except KeyboardInterrupt:
        print("⚠️ Interrupted. Graph extraction checkpoint has already been saved after each completed TextUnit.")
        raise

    except Exception as e:
        print(f"⚠️ LLM graph extraction failed: {type(e).__name__}: {e}")
        print("Checkpoint has already been saved after each completed TextUnit.")
        raise

    entity_df = _records_to_df(all_entities)
    relation_df = _records_to_df(all_relations)

    print(f"✅ LLM graph extraction done.")
    print(f"   Entities: {len(entity_df)}")
    print(f"   Relations: {len(relation_df)}")
    print(f"✅ Checkpoint saved: {LLM_GRAPH_EXTRACTION_CHECKPOINT_PATH}")

    return entity_df, relation_df
# Optional full-GraphRAG style extraction from raw review text.
# Disabled by default because it calls an LLM. Results are cached by review hash.
if USE_LLM_GRAPH_EXTRACTION:
    from config import load_config as load_module_config
    from llm_graph_extraction import LLMGraphExtractor

    print("Running LLM graph extraction with per-TextUnit checkpoint...")
    module_config = load_module_config(REPO_ROOT)
    extractor = LLMGraphExtractor(module_config)
    extracted_entities, extracted_relations = extract_text_units_with_checkpoint(
        extractor=extractor,
        text_units_df=text_units,
    )
    print(f"LLM extracted entities: {len(extracted_entities)}, relations: {len(extracted_relations)}")
else:
    extracted_entities = pd.DataFrame()
    extracted_relations = pd.DataFrame()
    print("Skipping LLM entity/relation extraction. Set USE_LLM_GRAPH_EXTRACTION=true to run it.")


Running LLM graph extraction with per-TextUnit checkpoint...
✅ Loaded checkpoint: 881 items -> .cache\graphrag\llm_graph_extraction_checkpoint.json
Total TextUnits: 881
Checkpoint hits: 881
TextUnits still need LLM extraction: 0
Checkpoint path: D:\restaurant-kg-recommender-master\.cache\graphrag\llm_graph_extraction_checkpoint.json
Progress log path: D:\restaurant-kg-recommender-master\.cache\graphrag\llm_graph_extraction_progress.jsonl
✅ All TextUnits loaded from checkpoint. No LLM call needed.
LLM extracted entities: 3265, relations: 1836


In [55]:
# ============================================================
# Deduplicate LLM extracted entities before Neo4j upsert
# Mục tiêu:
# - Cùng entity "cơm gà" chỉ có 1 node ExtractedEntity
# - Nhiều TextUnit vẫn được phép MENTIONS vào cùng entity đó
# ============================================================

import re
import hashlib
import unicodedata
import pandas as pd


def normalize_entity_name(x) -> str:
    x = "" if x is None else str(x)
    x = unicodedata.normalize("NFKC", x).lower()
    x = re.sub(r"\s+", " ", x).strip()
    return x


def normalize_entity_type(x) -> str:
    x = "" if x is None else str(x)
    x = unicodedata.normalize("NFKC", x).lower()
    x = re.sub(r"\s+", "_", x).strip("_")
    return x or "unknown"


def stable_entity_id(entity_type, name) -> str:
    etype = normalize_entity_type(entity_type)
    nname = normalize_entity_name(name)
    raw = f"{etype}::{nname}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:24]


def prepare_extracted_entities_for_neo4j(entity_df: pd.DataFrame) -> pd.DataFrame:
    if entity_df is None or entity_df.empty:
        return pd.DataFrame(columns=[
            "text_unit_id",
            "entity_id",
            "entity_type",
            "entity_name",
            "entity_name_norm",
        ])

    df = entity_df.copy()

    # Chuẩn hóa tên cột phòng trường hợp extractor trả schema khác nhau
    if "name" in df.columns and "entity_name" not in df.columns:
        df["entity_name"] = df["name"]

    if "type" in df.columns and "entity_type" not in df.columns:
        df["entity_type"] = df["type"]

    if "entity_name" not in df.columns:
        raise KeyError("extracted_entities must have column 'name' or 'entity_name'")

    if "entity_type" not in df.columns:
        df["entity_type"] = "unknown"

    if "text_unit_id" not in df.columns:
        raise KeyError("extracted_entities must have column 'text_unit_id'")

    df["text_unit_id"] = df["text_unit_id"].astype(str)
    df["entity_name"] = df["entity_name"].astype(str)
    df["entity_type"] = df["entity_type"].apply(normalize_entity_type)
    df["entity_name_norm"] = df["entity_name"].apply(normalize_entity_name)

    # Bỏ entity rỗng
    df = df[df["entity_name_norm"].ne("")].copy()

    # ID toàn cục: cùng type + same normalized name => cùng node
    df["entity_id"] = df.apply(
        lambda r: stable_entity_id(r["entity_type"], r["entity_name_norm"]),
        axis=1,
    )

    # Quan trọng:
    # Không drop_duplicates(["entity_id"]) vì sẽ mất evidence từ nhiều TextUnit.
    # Chỉ bỏ duplicate cùng một TextUnit nhắc cùng một entity nhiều lần.
    before = len(df)
    df = df.drop_duplicates(
        subset=["text_unit_id", "entity_id"],
        keep="first",
    ).copy()
    after = len(df)

    print(f"Extracted entities before dedup: {before}")
    print(f"Extracted entities after dedup text_unit_id + entity_id: {after}")
    print(f"Removed duplicates: {before - after}")
    print(f"Unique entity nodes expected in Neo4j: {df['entity_id'].nunique()}")

    return df


extracted_entities = prepare_extracted_entities_for_neo4j(extracted_entities)

display(extracted_entities.head(10))

Extracted entities before dedup: 3265
Extracted entities after dedup text_unit_id + entity_id: 3264
Removed duplicates: 1
Unique entity nodes expected in Neo4j: 996


,entity_key,name,entity_type,store_key,review_id,text_unit_id,sentiment,confidence,evidence,entity_name,entity_name_norm,entity_id
0,dish:com-ga,cơm gà,dish,25639,15034de1df0b4e9a,tu_15034de1df0b4e9a,positive,0.82,quán mùa - cơm gà,cơm gà,cơm gà,040a58ebdf5a106e37acfe40
1,dish:bun-mien-tron,bún miến trộn,dish,25639,15034de1df0b4e9a,tu_15034de1df0b4e9a,positive,0.82,& bún miến trộn,bún miến trộn,bún miến trộn,cdb9eaea119261cb14a4cab4
2,food_descriptor:ngon,ngon,food_descriptor,25639,15034de1df0b4e9a,tu_15034de1df0b4e9a,positive,0.96,"ngon đấy, ngon xỉu",ngon,ngon,65d738431f74eda99ffa8744
3,dish:pho,phở,dish,25639,8ea2b2b2c0447f44,tu_8ea2b2b2c0447f44,neutral,0.95,đặt phở,phở,phở,69ae9ace78b2610069d82293
4,dish:bun,bún,dish,25639,8ea2b2b2c0447f44,tu_8ea2b2b2c0447f44,neutral,0.95,cho bún,bún,bún,d82c1a9193fe199bcd5d3d00
5,problem:giao-nham-mon,giao nhầm món,problem,25639,8ea2b2b2c0447f44,tu_8ea2b2b2c0447f44,negative,0.78,đặt phở cho bún?,giao nhầm món,giao nhầm món,28509320613f0b92e1138fc1
6,dish:bun-ngan-tron,bún ngan trộn,dish,25639,d2292fdad09da78a,tu_d2292fdad09da78a,negative,0.96,bún ngan trộn của mình,bún ngan trộn,bún ngan trộn,fa2748a91b1e784011074e22
7,problem:gian,gián,problem,25639,d2292fdad09da78a,tu_d2292fdad09da78a,negative,0.99,có cả gián,gián,gián,b4011c8d2b0a7ce789c1cd08
8,dish:bun-ga,bún gà,dish,25639,d7985d984f2378be,tu_d7985d984f2378be,negative,0.94,bán bún gà,bún gà,bún gà,372d0d278a13abc19ec648af
9,dish:tuong-ot-chanh,tương ớt chanh,dish,25639,d7985d984f2378be,tu_d7985d984f2378be,neutral,0.82,tương ớt chanh,tương ớt chanh,tương ớt chanh,f39ac0c72d74af08e05ffc4b


## 5. Graph schema sâu hơn

In [56]:
GRAPH_SCHEMA_NOTE = """
GraphRAG-style schema used in this notebook

Core indexing nodes:
- (:Restaurant)        domain entity
- (:Review)            raw BeFood comment record
- (:TextUnit)          chunk/text unit; each comment is represented as one text unit
- (:Attribute)         aspect sentiment aggregate per restaurant
- (:MenuItem)          concrete menu item from BeFood
- (:DishFamily)        normalized dish/food entity derived from menu item names
- (:Community)         detected cluster over the restaurant similarity graph
- (:CommunityReport)   LLM-written summary of a community for global/contextual retrieval

Domain/context nodes:
- (:Area), (:Cuisine), (:Category), (:PriceBand), (:AtmosphereTag), (:MenuCategory)

Core relations:
- (Review)-[:HAS_TEXT_UNIT]->(TextUnit)
- (TextUnit)-[:ABOUT]->(Restaurant)
- (TextUnit)-[:MENTIONS_ASPECT]->(Attribute)
- (Restaurant)-[:HAS_ATTRIBUTE]->(Attribute)
- (Restaurant)-[:HAS_MENU_ITEM]->(MenuItem)
- (MenuItem)-[:IN_MENU_CATEGORY]->(MenuCategory)
- (Restaurant)-[:SERVES_FAMILY {menu_item_count, order_count, like_count, avg_price}]->(DishFamily)
- (Restaurant)-[:IN_COMMUNITY]->(Community)
- (Community)-[:HAS_REPORT]->(CommunityReport)
- (Restaurant)-[:SIMILAR_TO {similarity, method:'embedding_cosine'}]-(Restaurant)
"""
print(GRAPH_SCHEMA_NOTE)



GraphRAG-style schema used in this notebook

Core indexing nodes:
- (:Restaurant)        domain entity
- (:Review)            raw BeFood comment record
- (:TextUnit)          chunk/text unit; each comment is represented as one text unit
- (:Attribute)         aspect sentiment aggregate per restaurant
- (:MenuItem)          concrete menu item from BeFood
- (:DishFamily)        normalized dish/food entity derived from menu item names
- (:Community)         detected cluster over the restaurant similarity graph
- (:CommunityReport)   LLM-written summary of a community for global/contextual retrieval

Domain/context nodes:
- (:Area), (:Cuisine), (:Category), (:PriceBand), (:AtmosphereTag), (:MenuCategory)

Core relations:
- (Review)-[:HAS_TEXT_UNIT]->(TextUnit)
- (TextUnit)-[:ABOUT]->(Restaurant)
- (TextUnit)-[:MENTIONS_ASPECT]->(Attribute)
- (Restaurant)-[:HAS_ATTRIBUTE]->(Attribute)
- (Restaurant)-[:HAS_MENU_ITEM]->(MenuItem)
- (MenuItem)-[:IN_MENU_CATEGORY]->(MenuCategory)
- (Restaurant

In [57]:
from neo4j import GraphDatabase

class Neo4jClient:
    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))
        self.driver.verify_connectivity()

    def close(self):
        self.driver.close()

    def run(self, query: str, params: dict | None = None):
        with self.driver.session() as s:
            return [r.data() for r in s.run(query, params or {})]

    def create_schema(self):
        stmts = [
            # Drop old Enterprise-only / conflicting schema if it exists
            "DROP CONSTRAINT attr_key IF EXISTS",
            "DROP CONSTRAINT area_key IF EXISTS",

            # Community-compatible constraints
            "CREATE CONSTRAINT restaurant_key IF NOT EXISTS FOR (r:Restaurant) REQUIRE r.store_key IS UNIQUE",
            "CREATE CONSTRAINT review_key IF NOT EXISTS FOR (r:Review) REQUIRE r.review_id IS UNIQUE",
            "CREATE CONSTRAINT text_unit_key IF NOT EXISTS FOR (t:TextUnit) REQUIRE t.text_unit_id IS UNIQUE",

            # Neo4j Community does NOT support NODE KEY.
            # Use synthetic unique ids instead.
            "CREATE CONSTRAINT attr_id_key IF NOT EXISTS FOR (a:Attribute) REQUIRE a.attribute_id IS UNIQUE",
            "CREATE CONSTRAINT area_id_key IF NOT EXISTS FOR (a:Area) REQUIRE a.area_id IS UNIQUE",

            "CREATE CONSTRAINT cuisine_name IF NOT EXISTS FOR (c:Cuisine) REQUIRE c.name IS UNIQUE",
            "CREATE CONSTRAINT category_name IF NOT EXISTS FOR (c:Category) REQUIRE c.name IS UNIQUE",
            "CREATE CONSTRAINT priceband_name IF NOT EXISTS FOR (p:PriceBand) REQUIRE p.name IS UNIQUE",
            "CREATE CONSTRAINT atmos_name IF NOT EXISTS FOR (a:AtmosphereTag) REQUIRE a.name IS UNIQUE",
            "CREATE CONSTRAINT dish_family_name IF NOT EXISTS FOR (d:DishFamily) REQUIRE d.name IS UNIQUE",
            "CREATE CONSTRAINT extracted_entity_key IF NOT EXISTS FOR (e:ExtractedEntity) REQUIRE e.entity_key IS UNIQUE",
            "CREATE CONSTRAINT extracted_relation_key IF NOT EXISTS FOR (r:ExtractedRelation) REQUIRE r.relation_key IS UNIQUE",
                        "CREATE CONSTRAINT menu_item_key IF NOT EXISTS FOR (m:MenuItem) REQUIRE m.menu_item_id IS UNIQUE",
            "CREATE CONSTRAINT menu_category_name IF NOT EXISTS FOR (m:MenuCategory) REQUIRE m.name IS UNIQUE",
            "CREATE CONSTRAINT community_key IF NOT EXISTS FOR (c:Community) REQUIRE c.community_id IS UNIQUE",
            "CREATE CONSTRAINT community_report_key IF NOT EXISTS FOR (cr:CommunityReport) REQUIRE cr.report_id IS UNIQUE",

            "CREATE INDEX rest_rating IF NOT EXISTS FOR (r:Restaurant) ON (r.gmaps_rating)",
            "CREATE INDEX text_unit_store IF NOT EXISTS FOR (t:TextUnit) ON (t.store_key)",
            "CREATE INDEX menu_item_price IF NOT EXISTS FOR (m:MenuItem) ON (m.price)",
            "CREATE INDEX attr_type IF NOT EXISTS FOR (a:Attribute) ON (a.type)",
            "CREATE INDEX extracted_entity_type IF NOT EXISTS FOR (e:ExtractedEntity) ON (e.type)",
            "CREATE INDEX community_level IF NOT EXISTS FOR (c:Community) ON (c.level)",
        ]

        for q in stmts:
            self.run(q)


neo4j_client = Neo4jClient(
    NEO4J_URI,
    NEO4J_USER,
    NEO4J_PASSWORD,
)

neo4j_client.create_schema()
print("✅ Neo4j connected and schema is ready")

✅ Neo4j connected and schema is ready


In [ ]:
# Optional schema inspection. Run manually if you need to inspect Neo4j indexes.
# with neo4j_client.driver.session() as session:
#     rows = session.run("SHOW INDEXES YIELD name, type, entityType, labelsOrTypes, properties RETURN *").data()
# rows


In [ ]:
# def upsert_extracted_entities_graph(client: Neo4jClient, entities_df: pd.DataFrame, relations_df: pd.DataFrame):
#     if entities_df is not None and not entities_df.empty:
#         client.run("""
#         UNWIND $rows AS row
#         MATCH (r:Restaurant {store_key: row.store_key})
#         MATCH (tu:TextUnit {text_unit_id: row.text_unit_id})
#         MERGE (e:ExtractedEntity {entity_key: row.entity_key})
#         SET e.name = row.name, e.type = row.entity_type, e.updated_at = datetime()
#         MERGE (tu)-[m:MENTIONS_ENTITY]->(e)
#         SET m.sentiment = row.sentiment, m.confidence = row.confidence,
#             m.evidence = row.evidence, m.review_id = row.review_id, m.updated_at = datetime()
#         MERGE (r)-[re:HAS_EXTRACTED_ENTITY]->(e)
#         SET re.last_seen_at = datetime()
#         """, {"rows": entities_df.to_dict("records")})
#     if relations_df is not None and not relations_df.empty:
#         client.run("""
#         UNWIND $rows AS row
#         MATCH (source:ExtractedEntity {entity_key: row.source_entity_key})
#         MATCH (target:ExtractedEntity {entity_key: row.target_entity_key})
#         MATCH (tu:TextUnit {text_unit_id: row.text_unit_id})
#         MERGE (rel:ExtractedRelation {relation_key: row.relation_key})
#         SET rel.type = row.relation_type, rel.sentiment = row.sentiment,
#             rel.confidence = row.confidence, rel.evidence = row.evidence,
#             rel.store_key = row.store_key, rel.review_id = row.review_id,
#             rel.text_unit_id = row.text_unit_id, rel.updated_at = datetime()
#         MERGE (source)-[:SOURCE_OF]->(rel)
#         MERGE (rel)-[:TARGETS]->(target)
#         MERGE (tu)-[:SUPPORTS_RELATION]->(rel)
#         """, {"rows": relations_df.to_dict("records")})
#     print(f"Upserted extracted entities={0 if entities_df is None else len(entities_df)}, relations={0 if relations_df is None else len(relations_df)}")

# # upsert_extracted_entities_graph(neo4j_client, extracted_entities, extracted_relations)


In [31]:
# ============================================================
# Prepare + upsert extracted entities/relations into Neo4j
# Fix duplicate entity nodes:
# - Same entity name + type => same ExtractedEntity node
# - Multiple TextUnits can still MENTIONS_ENTITY the same node
# ============================================================

import re
import hashlib
import unicodedata
from typing import Tuple
import pandas as pd


def _normalize_entity_name(x) -> str:
    x = "" if x is None else str(x)
    x = unicodedata.normalize("NFKC", x).lower()
    x = re.sub(r"\s+", " ", x).strip()
    return x


def _normalize_entity_type(x) -> str:
    x = "" if x is None else str(x)
    x = unicodedata.normalize("NFKC", x).lower()
    x = re.sub(r"\s+", "_", x).strip("_")
    return x or "unknown"


def _stable_entity_key(entity_type, name) -> str:
    """
    Global entity key.
    Same type + same normalized name => same ExtractedEntity node.
    """
    etype = _normalize_entity_type(entity_type)
    nname = _normalize_entity_name(name)
    raw = f"{etype}::{nname}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:24]


def _stable_relation_key(source_entity_key, relation_type, target_entity_key, text_unit_id) -> str:
    """
    Relation key theo từng TextUnit để giữ evidence.
    Same source + relation + target + same TextUnit => same ExtractedRelation node.
    """
    raw = "::".join([
        str(source_entity_key),
        str(relation_type),
        str(target_entity_key),
        str(text_unit_id),
    ])
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:24]


def _clean_records_for_neo4j(df: pd.DataFrame) -> list:
    """
    Convert NaN/NaT to None before sending to Neo4j.
    """
    if df is None or df.empty:
        return []
    clean = df.copy()
    clean = clean.where(pd.notna(clean), None)
    return clean.to_dict("records")


def prepare_extracted_graph_for_neo4j(
    entities_df: pd.DataFrame,
    relations_df: pd.DataFrame,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Chuẩn hóa extracted_entities và extracted_relations trước khi upsert Neo4j.

    Quan trọng:
    - entity_key mới là global: entity_type + normalized name
    - vẫn giữ MENTIONS_ENTITY theo từng TextUnit
    - relations được remap source/target sang entity_key mới
    """
    # =========================
    # Prepare entities
    # =========================
    if entities_df is None or entities_df.empty:
        prepared_entities = pd.DataFrame()
        key_map = {}
    else:
        e = entities_df.copy()

        # Chuẩn hóa tên cột nếu extractor trả schema hơi khác
        if "name" not in e.columns and "entity_name" in e.columns:
            e["name"] = e["entity_name"]

        if "entity_type" not in e.columns and "type" in e.columns:
            e["entity_type"] = e["type"]

        if "entity_type" not in e.columns:
            e["entity_type"] = "unknown"

        required_entity_cols = {"store_key", "text_unit_id", "name"}
        missing = required_entity_cols - set(e.columns)
        if missing:
            raise KeyError(f"entities_df missing required columns: {missing}")

        # Giữ key cũ để remap relations
        if "entity_key" in e.columns:
            e["old_entity_key"] = e["entity_key"].astype(str)
        else:
            e["old_entity_key"] = e.apply(
                lambda r: f"{r.get('text_unit_id')}::{r.get('entity_type')}::{r.get('name')}",
                axis=1,
            )

        e["name"] = e["name"].astype(str)
        e["entity_type"] = e["entity_type"].apply(_normalize_entity_type)
        e["name_norm"] = e["name"].apply(_normalize_entity_name)

        # Bỏ entity rỗng
        e = e[e["name_norm"].ne("")].copy()

        # Entity key mới: global theo type + normalized name
        e["entity_key"] = e.apply(
            lambda r: _stable_entity_key(r["entity_type"], r["name_norm"]),
            axis=1,
        )

        # Cột optional nếu thiếu thì tạo
        for col, default in {
            "sentiment": None,
            "confidence": None,
            "evidence": "",
            "review_id": "",
        }.items():
            if col not in e.columns:
                e[col] = default

        # Map old key -> new global key để remap relations
        key_map = dict(zip(e["old_entity_key"].astype(str), e["entity_key"].astype(str)))

        # Thêm cả new key -> new key, phòng khi relations đã dùng key mới
        for new_key in e["entity_key"].astype(str).unique():
            key_map[new_key] = new_key

        before = len(e)

        # Không drop_duplicates(["entity_key"]) vì như vậy mất evidence từ nhiều TextUnit.
        # Chỉ bỏ duplicate cùng một TextUnit nhắc cùng một entity nhiều lần.
        e = e.drop_duplicates(
            subset=["text_unit_id", "entity_key"],
            keep="first",
        ).copy()

        after = len(e)

        prepared_entities = e

        print("Prepared extracted entities")
        print(f"  rows before dedup: {before}")
        print(f"  rows after dedup text_unit_id + entity_key: {after}")
        print(f"  removed duplicates: {before - after}")
        print(f"  unique ExtractedEntity nodes expected: {prepared_entities['entity_key'].nunique()}")

    # =========================
    # Prepare relations
    # =========================
    if relations_df is None or relations_df.empty:
        prepared_relations = pd.DataFrame()
    else:
        r = relations_df.copy()

        required_relation_cols = {
            "source_entity_key",
            "target_entity_key",
            "text_unit_id",
            "relation_type",
        }
        missing = required_relation_cols - set(r.columns)
        if missing:
            raise KeyError(f"relations_df missing required columns: {missing}")

        # Remap source/target từ old entity_key sang global entity_key
        r["old_source_entity_key"] = r["source_entity_key"].astype(str)
        r["old_target_entity_key"] = r["target_entity_key"].astype(str)

        r["source_entity_key"] = r["old_source_entity_key"].map(key_map)
        r["target_entity_key"] = r["old_target_entity_key"].map(key_map)

        before_rel = len(r)

        # Bỏ relation không map được entity
        r = r[
            r["source_entity_key"].notna()
            & r["target_entity_key"].notna()
        ].copy()

        dropped_rel = before_rel - len(r)

        for col, default in {
            "store_key": "",
            "review_id": "",
            "sentiment": None,
            "confidence": None,
            "evidence": "",
        }.items():
            if col not in r.columns:
                r[col] = default

        # Rebuild relation_key theo key mới
        r["relation_key"] = r.apply(
            lambda row: _stable_relation_key(
                row["source_entity_key"],
                row["relation_type"],
                row["target_entity_key"],
                row["text_unit_id"],
            ),
            axis=1,
        )

        before_dedup_rel = len(r)

        r = r.drop_duplicates(
            subset=["relation_key"],
            keep="first",
        ).copy()

        prepared_relations = r

        print("Prepared extracted relations")
        print(f"  rows before remap: {before_rel}")
        print(f"  dropped because source/target entity not found: {dropped_rel}")
        print(f"  rows after dedup relation_key: {len(prepared_relations)}")
        print(f"  removed duplicate relations: {before_dedup_rel - len(prepared_relations)}")

    return prepared_entities, prepared_relations


def ensure_extracted_graph_constraints(client: Neo4jClient):
    """
    Nên chạy trước upsert để Neo4j MERGE nhanh và chống duplicate tốt hơn.
    """
    client.run("""
    CREATE CONSTRAINT extracted_entity_key_unique IF NOT EXISTS
    FOR (e:ExtractedEntity)
    REQUIRE e.entity_key IS UNIQUE
    """)

    client.run("""
    CREATE CONSTRAINT extracted_relation_key_unique IF NOT EXISTS
    FOR (r:ExtractedRelation)
    REQUIRE r.relation_key IS UNIQUE
    """)


def upsert_extracted_entities_graph(
    client: Neo4jClient,
    entities_df: pd.DataFrame,
    relations_df: pd.DataFrame,
):
    """
    Upsert LLM extracted graph.

    Entity node:
    - MERGE by entity_key global
    - entity_key = hash(entity_type + normalized name)

    Mention edge:
    - TextUnit -> ExtractedEntity
    - giữ evidence/sentiment/confidence theo từng TextUnit

    Restaurant edge:
    - Restaurant -> ExtractedEntity
    - cho biết quán này có entity đó được nhắc trong review
    """
    ensure_extracted_graph_constraints(client)

    entities_df, relations_df = prepare_extracted_graph_for_neo4j(
        entities_df=entities_df,
        relations_df=relations_df,
    )

    if entities_df is not None and not entities_df.empty:
        entity_cols = [
            "store_key",
            "text_unit_id",
            "review_id",
            "entity_key",
            "name",
            "name_norm",
            "entity_type",
            "sentiment",
            "confidence",
            "evidence",
        ]

        rows = _clean_records_for_neo4j(entities_df[entity_cols])

        client.run("""
        UNWIND $rows AS row
        MATCH (r:Restaurant {store_key: row.store_key})
        MATCH (tu:TextUnit {text_unit_id: row.text_unit_id})

        MERGE (e:ExtractedEntity {entity_key: row.entity_key})
        SET e.name = row.name,
            e.name_norm = row.name_norm,
            e.type = row.entity_type,
            e.updated_at = datetime()

        MERGE (tu)-[m:MENTIONS_ENTITY]->(e)
        SET m.sentiment = row.sentiment,
            m.confidence = row.confidence,
            m.evidence = row.evidence,
            m.review_id = row.review_id,
            m.updated_at = datetime()

        MERGE (r)-[re:HAS_EXTRACTED_ENTITY]->(e)
        SET re.last_seen_at = datetime()
        """, {"rows": rows})

    if relations_df is not None and not relations_df.empty:
        relation_cols = [
            "source_entity_key",
            "target_entity_key",
            "relation_key",
            "relation_type",
            "sentiment",
            "confidence",
            "evidence",
            "store_key",
            "review_id",
            "text_unit_id",
        ]

        rows = _clean_records_for_neo4j(relations_df[relation_cols])

        client.run("""
        UNWIND $rows AS row
        MATCH (source:ExtractedEntity {entity_key: row.source_entity_key})
        MATCH (target:ExtractedEntity {entity_key: row.target_entity_key})
        MATCH (tu:TextUnit {text_unit_id: row.text_unit_id})

        MERGE (rel:ExtractedRelation {relation_key: row.relation_key})
        SET rel.type = row.relation_type,
            rel.sentiment = row.sentiment,
            rel.confidence = row.confidence,
            rel.evidence = row.evidence,
            rel.store_key = row.store_key,
            rel.review_id = row.review_id,
            rel.text_unit_id = row.text_unit_id,
            rel.updated_at = datetime()

        MERGE (source)-[:SOURCE_OF]->(rel)
        MERGE (rel)-[:TARGETS]->(target)
        MERGE (tu)-[:SUPPORTS_RELATION]->(rel)
        """, {"rows": rows})

    print(
        f"Upserted extracted entity mentions="
        f"{0 if entities_df is None else len(entities_df)}, "
        f"unique entity nodes="
        f"{0 if entities_df is None or entities_df.empty else entities_df['entity_key'].nunique()}, "
        f"relations="
        f"{0 if relations_df is None else len(relations_df)}"
    )

In [ ]:
# No destructive schema changes in the default notebook path.
# If an index/constraint must be dropped, do it explicitly in a separate maintenance cell.


In [32]:
def upsert_restaurants_graph(client: Neo4jClient, restaurants_df: pd.DataFrame):
    def _clean_str(x: Any):
        if x is None or pd.isna(x):
            return None
        s = str(x).strip()
        return s if s else None

    def _clean_float(x: Any):
        if x is None or pd.isna(x):
            return None
        try:
            return float(x)
        except Exception:
            return None

    def _clean_int(x: Any):
        if x is None or pd.isna(x):
            return None
        try:
            return int(x)
        except Exception:
            return None

    rows = []
    for _, r in restaurants_df.iterrows():
        district = _clean_str(r.get("district"))
        city = _clean_str(r.get("city"))
        price_band = _clean_str(r.get("price_band"))

        area_id = None
        if city and district:
            area_id = f"{city}:{district}"

        rows.append({
            "store_key": str(r.get("store_key")),
            "name": _clean_str(r.get("name")) or "",
            "address": _clean_str(r.get("address")),
            "district": district,
            "city": city,
            "area_id": area_id,
            "lat": _clean_float(r.get("lat")),
            "lng": _clean_float(r.get("lng")),
            #"distance_km": _clean_float(r.get("distance_km")),
            "gmaps_rating": _clean_float(r.get("gmaps_rating")),
            "foody_rating": _clean_float(r.get("foody_rating")),
            "rating": _clean_float(r.get("rating")),
            "review_count": _clean_int(r.get("review_count")),
            "price_band": price_band,
            "price_min": _clean_float(r.get("price_min")),
            "price_max": _clean_float(r.get("price_max")),
            "menu_item_count": int(_clean_int(r.get("menu_item_count")) or 0),
            "menu_price_min": _clean_float(r.get("menu_price_min")),
            "menu_price_max": _clean_float(r.get("menu_price_max")),
            "menu_price_median": _clean_float(r.get("menu_price_median")),
            "top_menu_items": r.get("top_menu_items") if isinstance(r.get("top_menu_items"), list) else [],
            "opening_hours": _clean_str(r.get("opening_hours")),
            "delivery_time": _clean_int(r.get("delivery_time")),
            "image_url": _clean_str(r.get("image_url")),
            "categories": r.get("categories"),
            "cuisines": r.get("cuisines"),
            "atmosphere": r.get("atmosphere"),
            "audiences": r.get("audiences"),
        })

    client.run("""
    UNWIND $rows AS row
    MERGE (r:Restaurant {store_key: row.store_key})
    SET r.name = row.name, r.address = row.address, r.district = row.district, r.city = row.city,
        r.lat = row.lat, r.lng = row.lng,
        r.gmaps_rating = row.gmaps_rating, r.foody_rating = row.foody_rating,
        r.rating = row.rating, r.review_count = row.review_count,
        r.price_band = row.price_band,
        r.price_min = row.price_min, r.price_max = row.price_max,
        r.menu_item_count = row.menu_item_count, r.menu_price_min = row.menu_price_min,
        r.menu_price_max = row.menu_price_max, r.menu_price_median = row.menu_price_median,
        r.top_menu_items = row.top_menu_items, r.opening_hours = row.opening_hours,
        r.delivery_time = row.delivery_time, r.image_url = row.image_url, r.audiences = row.audiences,
        r.updated_at = datetime()
    WITH r, row
    FOREACH (cat IN coalesce(row.categories, []) | MERGE (c:Category {name: cat}) MERGE (r)-[:HAS_CATEGORY]->(c))
    FOREACH (cui IN coalesce(row.cuisines, []) | MERGE (c:Cuisine {name: cui}) MERGE (r)-[:HAS_CUISINE]->(c))
    FOREACH (atm IN coalesce(row.atmosphere, []) | MERGE (a:AtmosphereTag {name: atm}) MERGE (r)-[:HAS_ATMOSPHERE]->(a))

    // Prevent NaN/empty from creating (:PriceBand {name: NaN})
    FOREACH (_ IN CASE
        WHEN row.price_band IS NULL THEN []
        WHEN trim(toString(row.price_band)) = '' THEN []
        ELSE [1]
    END |
        MERGE (p:PriceBand {name: row.price_band})
        MERGE (r)-[:HAS_PRICE_BAND]->(p)
    )

    // Prevent NaN/empty area data
    FOREACH (_ IN CASE
        WHEN row.area_id IS NULL THEN []
        WHEN row.district IS NULL OR row.city IS NULL THEN []
        WHEN trim(toString(row.district)) = '' OR trim(toString(row.city)) = '' THEN []
        ELSE [1]
    END |
        MERGE (a:Area {area_id: row.area_id})
        SET a.name = row.district, a.city = row.city
        MERGE (r)-[:IN_AREA]->(a)
    )
    """, {"rows": rows})


def upsert_attributes_graph(client: Neo4jClient, attrs_df: pd.DataFrame):
    rows = []
    for _, r in attrs_df.iterrows():
        rows.append({
            "attribute_id": f"{r['store_key']}:{r['attribute_type']}",
            "store_key": r["store_key"], "attribute_type": r["attribute_type"],
            "attribute_score": r["attribute_score"], "sample_count": r["sample_count"],
        })
    client.run("""
    UNWIND $rows AS row
    MATCH (r:Restaurant {store_key: row.store_key})
    MERGE (a:Attribute {attribute_id: row.attribute_id})
    SET a.store_key = row.store_key, a.type = row.attribute_type, a.score = row.attribute_score,
        a.sample_count = row.sample_count, a.updated_at = datetime()
    MERGE (r)-[:HAS_ATTRIBUTE]->(a)
    """, {"rows": rows})


def upsert_reviews_and_text_units_graph(client: Neo4jClient, units_df: pd.DataFrame):
    rows = []
    for _, r in units_df.iterrows():
        rated_at = r.get("rated_at", None)
        if pd.isna(rated_at):
            rated_at = None
        else:
            rated_at = str(rated_at)
        rows.append({
            "review_id": r["review_id"], "text_unit_id": r["text_unit_id"], "store_key": r["store_key"],
            "feedback": r["feedback"], "chunk_text": r["chunk_text"], "rating": r["rating"],
            "rated_at": rated_at, "sentiment": r["sentiment"],
            "aspect_scores": json.dumps(r["aspect_scores"], ensure_ascii=False), "source": r["source"],
        })
    client.run("""
    UNWIND $rows AS row
    MATCH (rest:Restaurant {store_key: row.store_key})
    MERGE (rv:Review {review_id: row.review_id})
    SET rv.feedback = row.feedback, rv.rating = row.rating, rv.rated_at = row.rated_at,
        rv.sentiment = row.sentiment, rv.aspect_scores = row.aspect_scores, rv.source = row.source
    MERGE (tu:TextUnit {text_unit_id: row.text_unit_id})
    SET tu.text = row.chunk_text, tu.store_key = row.store_key, tu.source = row.source,
        tu.review_id = row.review_id, tu.sentiment = row.sentiment, tu.rating = row.rating, tu.updated_at = datetime()
    MERGE (rv)-[:HAS_TEXT_UNIT]->(tu)
    MERGE (tu)-[:ABOUT]->(rest)
    WITH tu, row
    MATCH (att:Attribute {store_key: row.store_key})
    MERGE (tu)-[:MENTIONS_ASPECT]->(att)
    """, {"rows": rows})

# Ensure store_key consistency before Neo4j MATCH
summary["store_key"] = summary["store_key"].astype(str)

if restaurant_attrs is not None and not restaurant_attrs.empty:
    restaurant_attrs["store_key"] = restaurant_attrs["store_key"].astype(str)

if text_units is not None and not text_units.empty:
    text_units["store_key"] = text_units["store_key"].astype(str)

if menu_items is not None and not menu_items.empty:
    menu_items["store_key"] = menu_items["store_key"].astype(str)

if dish_families is not None and not dish_families.empty:
    dish_families["store_key"] = dish_families["store_key"].astype(str)

if extracted_entities is not None and not extracted_entities.empty:
    extracted_entities["store_key"] = extracted_entities["store_key"].astype(str)
    extracted_entities["text_unit_id"] = extracted_entities["text_unit_id"].astype(str)

if extracted_relations is not None and not extracted_relations.empty:
    extracted_relations["store_key"] = extracted_relations["store_key"].astype(str)
    extracted_relations["text_unit_id"] = extracted_relations["text_unit_id"].astype(str)

upsert_restaurants_graph(neo4j_client, summary)
upsert_attributes_graph(neo4j_client, restaurant_attrs)
upsert_reviews_and_text_units_graph(neo4j_client, text_units)
upsert_menu_items_graph(neo4j_client, menu_items)
upsert_dish_families(neo4j_client, dish_families)
upsert_extracted_entities_graph(neo4j_client, extracted_entities, extracted_relations)
print("Graph upsert complete: Restaurant, Review, TextUnit, Attribute, MenuItem, DishFamily and domain nodes")


Upserted 3245 menu items
Upserted 1202 dish family edges
Prepared extracted entities
  rows before dedup: 3264
  rows after dedup text_unit_id + entity_key: 3264
  removed duplicates: 0
  unique ExtractedEntity nodes expected: 996
Prepared extracted relations
  rows before remap: 1836
  dropped because source/target entity not found: 0
  rows after dedup relation_key: 1836
  removed duplicate relations: 0
Upserted extracted entity mentions=3264, unique entity nodes=996, relations=1836
Graph upsert complete: Restaurant, Review, TextUnit, Attribute, MenuItem, DishFamily and domain nodes


## 6. Embedding similarity edges + Leiden communities


In [ ]:
# Similarity edges and communities are built after embeddings are created in Section 7.1.


## 7. Vietnamese embeddings và Qdrant indexing


In [17]:
import sys, torch
print(sys.executable)
print(torch.__version__)
print("cuda:", torch.version.cuda)
print("is_available:", torch.cuda.is_available())

d:\restaurant-kg-recommender-master\.venv\Scripts\python.exe
2.5.1+cu121
cuda: 12.1
is_available: True


In [18]:
import torch 
import os
import json
import hashlib
import atexit
from pathlib import Path
from typing import List
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print("Loading Vietnamese embedding model...")
embed_model = SentenceTransformer(EMBED_MODEL)
EMBED_DIM = len(embed_model.encode("kiểm tra chiều embedding", normalize_embeddings=True))
print("✅ Embedding dim:", EMBED_DIM)

# =========================
# Embedding disk cache
# =========================
CACHE_DIR_PATH = Path(os.getenv("CACHE_DIR", ".cache/graphrag"))
CACHE_DIR_PATH.mkdir(parents=True, exist_ok=True)

EMBED_CACHE_PATH = CACHE_DIR_PATH / "embeddings.json"

if EMBED_CACHE_PATH.exists():
    try:
        with EMBED_CACHE_PATH.open("r", encoding="utf-8") as f:
            EMBED_CACHE = json.load(f)
        print(f"✅ Loaded embedding cache: {len(EMBED_CACHE)} items")
    except Exception as e:
        print(f"⚠️ Could not load embedding cache, starting fresh: {e}")
        EMBED_CACHE = {}
else:
    EMBED_CACHE = {}
    print("ℹ️ No embedding cache found, starting fresh")

EMBED_CACHE_DIRTY = False
EMBED_CACHE_HITS = 0
EMBED_CACHE_MISSES = 0


def _embedding_cache_key(text: str, prefix: str = "") -> str:
    raw = json.dumps(
        {
            "model": EMBED_MODEL,
            "prefix": prefix,
            "text": text,
        },
        ensure_ascii=False,
        sort_keys=True,
    )
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def save_embedding_cache() -> None:
    global EMBED_CACHE_DIRTY

    if not EMBED_CACHE_DIRTY:
        print(
            f"Embedding cache unchanged | "
            f"items={len(EMBED_CACHE)}, hits={EMBED_CACHE_HITS}, misses={EMBED_CACHE_MISSES}"
        )
        return

    tmp_path = EMBED_CACHE_PATH.with_suffix(".json.tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(EMBED_CACHE, f, ensure_ascii=False, separators=(",", ":"))

    tmp_path.replace(EMBED_CACHE_PATH)
    EMBED_CACHE_DIRTY = False

    print(
        f"✅ Saved embedding cache: {len(EMBED_CACHE)} items -> {EMBED_CACHE_PATH} | "
        f"hits={EMBED_CACHE_HITS}, misses={EMBED_CACHE_MISSES}"
    )


atexit.register(save_embedding_cache)

def _embed(text: str, prefix: str = "") -> List[float]:
    global EMBED_CACHE_DIRTY, EMBED_CACHE_HITS, EMBED_CACHE_MISSES

    text = "" if text is None else str(text)
    prefix = "" if prefix is None else str(prefix)

    key = _embedding_cache_key(text, prefix)

    if key in EMBED_CACHE:
        EMBED_CACHE_HITS += 1
        return EMBED_CACHE[key]

    EMBED_CACHE_MISSES += 1
    vec = embed_model.encode(prefix + text, normalize_embeddings=True).tolist()
    EMBED_CACHE[key] = vec
    EMBED_CACHE_DIRTY = True
    return vec

def emb_passage(text: str) -> List[float]:
    return _embed(text, EMBED_PREFIX_PASSAGE)

def emb_query(text: str) -> List[float]:
    return _embed(text, EMBED_PREFIX_QUERY)

qdrant = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
qdrant.get_collections()  # fail fast if the service is unavailable
print("✅ Qdrant connected")

def ensure_collection(client: QdrantClient, name: str, dim: int, recreate: bool = RECREATE_QDRANT):
    existing = {c.name for c in client.get_collections().collections}
    if name in existing and recreate:
        client.delete_collection(name)
        print("Deleted collection:", name)
    if name not in existing or recreate:
        client.create_collection(name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
        print("Created collection:", name)
    else:
        print("Using existing collection:", name)

ensure_collection(qdrant, COLL_TEXT_UNIT, EMBED_DIM)
ensure_collection(qdrant, COLL_RESTAURANT, EMBED_DIM)


Loading Vietnamese embedding model...
✅ Embedding dim: 768
✅ Loaded embedding cache: 1081 items
✅ Qdrant connected
Deleted collection: graphrag_text_units_vietnamese
Created collection: graphrag_text_units_vietnamese
Deleted collection: graphrag_restaurants_vietnamese
Created collection: graphrag_restaurants_vietnamese


In [ ]:
%pip install pywin32

In [58]:
def build_restaurant_summary_doc(store_key: str) -> str:
    row = summary.set_index("store_key").loc[store_key]
    attrs = restaurant_attrs[restaurant_attrs["store_key"].eq(store_key)]
    attr_text = ", ".join(f"{r.attribute_type}={r.attribute_score:+.2f}" for _, r in attrs.sort_values("attribute_type").iterrows())
    menu_text = ", ".join(row["top_menu_items"] or [])
    return "\n".join([
        f"Restaurant name: {row['name']}",
        f"Address: {row['address']}",
        f"Area: {row['district']}, {row['city']}",
        f"Rating: {row['rating']}",
        f"Review count: {row['review_count']}",
        f"Price band: {row['price_band']}",
        f"Source price range: {row['price_min']} - {row['price_max']}",
        f"Menu price range: {row['menu_price_min']} - {row['menu_price_max']}, median={row['menu_price_median']}",
        f"Categories: {', '.join(row['categories'] or [])}",
        f"Cuisines: {', '.join(row['cuisines'] or [])}",
        f"Dish families: {', '.join(row.get('dish_families') or [])}",
        f"Top menu items: {menu_text}",
        f"Opening hours: {row['opening_hours']}",
        f"Delivery time estimate: {row['delivery_time']}",
        f"Atmosphere: {', '.join(row['atmosphere'] or [])}",
        f"Audience: {', '.join(row['audiences'] or [])}",
        f"Aggregated aspect sentiment: {attr_text}",
    ])

restaurant_docs = pd.DataFrame({
    "store_key": summary["store_key"],
    "doc_text": summary["store_key"].apply(build_restaurant_summary_doc),
})
restaurant_docs["embedding"] = [emb_passage(x) for x in tqdm(restaurant_docs["doc_text"], desc="Embed restaurants")]
text_units["embedding"] = [emb_passage(x) for x in tqdm(text_units["chunk_text"], desc="Embed text units")]

def stable_int_id(text: str) -> int:
    return int(hashlib.md5(text.encode("utf-8")).hexdigest()[:8], 16)

def index_restaurant_docs():
    points = []
    meta_by_key = summary.set_index("store_key")
    for _, row in restaurant_docs.iterrows():
        meta = meta_by_key.loc[row["store_key"]]
        points.append(PointStruct(
            id=stable_int_id("rest-" + row["store_key"]),
            vector=row["embedding"],
            payload={
                "store_key": row["store_key"], "name": meta["name"], "address": meta["address"],
                "district": meta["district"], "city": meta["city"], "rating": meta["rating"],
                "lat": meta["lat"], "lng": meta["lng"], 
                "price_band": meta["price_band"], "top_menu_items": meta["top_menu_items"],
                "dish_families": meta.get("dish_families"), "categories": meta.get("categories"),
                "cuisines": meta.get("cuisines"), "menu_budget_item_ratio": meta.get("menu_budget_item_ratio"),
                "menu_price_min": meta["menu_price_min"], "menu_price_max": meta["menu_price_max"],
                "menu_price_median": meta.get("menu_price_median"),
                "doc_text": row["doc_text"], "doc_type": "restaurant_summary",
            }
        ))
    qdrant.upsert(collection_name=COLL_RESTAURANT, points=points)
    print(f"Indexed restaurant docs: {len(points)}")

def index_text_units():
    points = []
    for _, row in text_units.iterrows():
        points.append(PointStruct(
            id=stable_int_id("tu-" + row["text_unit_id"]),
            vector=row["embedding"],
            payload={
                "text_unit_id": row["text_unit_id"], "review_id": row["review_id"], "store_key": row["store_key"],
                "store_name": name_map.get(row["store_key"], row["store_name"]), "rating": row["rating"],
                "sentiment": row["sentiment"], "feedback": row["feedback"], "aspect_scores": row["aspect_scores"],
                "doc_text": row["chunk_text"], "doc_type": "text_unit",
            }
        ))
    qdrant.upsert(collection_name=COLL_TEXT_UNIT, points=points)
    print(f"Indexed text units: {len(points)}")

try:
    index_restaurant_docs()
    index_text_units()
finally:
    save_embedding_cache()


Embed restaurants:   0%|          | 0/200 [00:00<?, ?it/s]

Embed text units:   0%|          | 0/881 [00:00<?, ?it/s]

Indexed restaurant docs: 200
Indexed text units: 881
✅ Saved embedding cache: 1085 items -> .cache\graphrag\embeddings.json | hits=1117, misses=4


## 7.1. Embedding similarity edges + Leiden communities


In [36]:
def cosine_np(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / ((np.linalg.norm(a) * np.linalg.norm(b)) + 1e-9))


def build_similarity_edges_from_embeddings(
    client: Neo4jClient,
    docs_df: pd.DataFrame,
    top_k: int = SIMILARITY_TOP_K,
    min_score: float = SIMILARITY_MIN_SCORE,
):
    """Build an undirected kNN similarity graph from restaurant embeddings.

    Important detail: each restaurant gets to choose its own top-k neighbors first.
    Then edges are deduplicated as undirected pairs. This avoids the old bug where
    `i > j` skipped half of the candidates before a node could select neighbors,
    which made the graph too sparse and produced ~1 community per restaurant.
    """
    keys = docs_df["store_key"].astype(str).tolist()
    mat = np.asarray(docs_df["embedding"].tolist(), dtype=np.float32)
    mat = mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-9)
    sim = mat @ mat.T

    edge_by_pair: Dict[Tuple[str, str], float] = {}
    for i, a in enumerate(keys):
        order = np.argsort(-sim[i])
        kept = 0
        for j in order:
            if i == j:
                continue
            score = float(sim[i, j])
            if score < min_score:
                continue
            b = keys[j]
            pair = tuple(sorted((a, b)))
            edge_by_pair[pair] = max(edge_by_pair.get(pair, -1.0), score)
            kept += 1
            if kept >= top_k:
                break

    edges = [
        {"a": a, "b": b, "sim": round(score, 4)}
        for (a, b), score in edge_by_pair.items()
    ]

    client.run("MATCH (:Restaurant)-[s:SIMILAR_TO]-(:Restaurant) DELETE s")
    if edges:
        client.run("""
        UNWIND $edges AS e
        MATCH (a:Restaurant {store_key: e.a})
        MATCH (b:Restaurant {store_key: e.b})
        MERGE (a)-[s:SIMILAR_TO]-(b)
        SET s.similarity = e.sim, s.method = 'embedding_cosine_knn', s.updated_at = datetime()
        """, {"edges": edges})

    degrees = defaultdict(int)
    for e in edges:
        degrees[e["a"]] += 1
        degrees[e["b"]] += 1
    isolated = len([k for k in keys if degrees[k] == 0])
    avg_degree = round((sum(degrees.values()) / max(len(keys), 1)), 2)
    print(f"Similarity graph stats: nodes={len(keys)}, edges={len(edges)}, isolated={isolated}, avg_degree={avg_degree}, min_score={min_score}, top_k={top_k}")
    return len(edges), edges


def cleanup_communities(client: Neo4jClient):
    """Remove stale communities/reports before writing a fresh Leiden partition."""
    client.run("MATCH (:Restaurant)-[rel:IN_COMMUNITY]->(:Community) DELETE rel")
    client.run("MATCH (cr:CommunityReport) DETACH DELETE cr")
    client.run("MATCH (c:Community) DETACH DELETE c")


def build_communities_igraph(
    client: Neo4jClient,
    keys: List[str],
    edges: List[dict],
    level: int = COMMUNITY_LEVEL,
) -> List[dict]:
    """Leiden community detection over the embedding kNN graph."""
    import igraph as ig

    cleanup_communities(client)

    if not edges:
        print("No similarity edges - skipping community detection")
        return []

    key_to_idx = {str(k): i for i, k in enumerate(keys)}
    g = ig.Graph(n=len(keys), directed=False)
    g.vs["store_key"] = [str(k) for k in keys]

    edge_list = []
    weights = []
    seen_pairs = set()
    for e in edges:
        a, b = str(e["a"]), str(e["b"])
        if a not in key_to_idx or b not in key_to_idx:
            continue
        pair = tuple(sorted((key_to_idx[a], key_to_idx[b])))
        if pair in seen_pairs:
            continue
        seen_pairs.add(pair)
        edge_list.append(pair)
        weights.append(float(e["sim"]))

    g.add_edges(edge_list)
    g.es["weight"] = weights

    components = g.components()
    isolated = sum(1 for comp in components if len(comp) == 1)
    print(f"Community input graph: vertices={g.vcount()}, edges={g.ecount()}, components={len(components)}, isolated_components={isolated}")

    partition = g.community_leiden(
        weights="weight",
        objective_function="modularity",
        n_iterations=10,
    )

    community_map = {}
    for community_id, members in enumerate(partition):
        for idx in members:
            community_map[str(keys[idx])] = str(community_id)

    rows = [{"store_key": k, "community_id": cid} for k, cid in community_map.items()]
    client.run("""
    UNWIND $rows AS row
    MATCH (r:Restaurant {store_key: row.store_key})
    SET r.community_id = row.community_id
    WITH r, row
    MERGE (c:Community {community_id: row.community_id})
    SET c.level = $level,
        c.algorithm = 'igraph.leiden.embedding_knn',
        c.updated_at = datetime()
    MERGE (r)-[:IN_COMMUNITY]->(c)
    """, {"rows": rows, "level": level})

    result = client.run("""
    MATCH (c:Community)<-[:IN_COMMUNITY]-(r:Restaurant)
    RETURN c.community_id AS community_id, count(r) AS size
    ORDER BY size DESC
    """)
    print(f"Leiden result: {len(partition)} communities, modularity={partition.modularity:.4f}")
    return result


n_edges, edge_list = build_similarity_edges_from_embeddings(neo4j_client, restaurant_docs)
print("Embedding-based SIMILAR_TO edges:", n_edges)

communities = build_communities_igraph(neo4j_client, restaurant_docs["store_key"].astype(str).tolist(), edge_list)
print("Communities:")
display(pd.DataFrame(communities).head(10))

def get_restaurant_subgraph(store_key: str, top_reviews: int = 3) -> Dict[str, Any]:
    q = """
    MATCH (r:Restaurant {store_key: $store_key})
    OPTIONAL MATCH (r)-[:IN_AREA]->(a:Area)
    OPTIONAL MATCH (r)-[:HAS_CUISINE]->(c:Cuisine)
    OPTIONAL MATCH (r)-[:HAS_CATEGORY]->(g:Category)
    OPTIONAL MATCH (r)-[:HAS_ATMOSPHERE]->(t:AtmosphereTag)
    OPTIONAL MATCH (r)-[:HAS_ATTRIBUTE]->(att:Attribute)
    OPTIONAL MATCH (r)<-[:ABOUT]-(tu:TextUnit)
    WITH r, a, collect(DISTINCT c.name) AS cuisines, collect(DISTINCT g.name) AS categories,
         collect(DISTINCT t.name) AS atmos, collect(DISTINCT {type: att.type, score: att.score}) AS attrs,
         collect(DISTINCT {text_unit_id: tu.text_unit_id, text: tu.text, sentiment: tu.sentiment, rating: tu.rating})[..$top_reviews] AS text_units
    OPTIONAL MATCH (r)-[s:SIMILAR_TO]-(nbr:Restaurant)
    OPTIONAL MATCH (r)-[:IN_COMMUNITY]->(com:Community)-[:HAS_REPORT]->(rep:CommunityReport)
    WITH r, a, cuisines, categories, atmos, attrs, text_units, rep,
         collect(DISTINCT {store_key: nbr.store_key, name: nbr.name, similarity: s.similarity})[..5] AS neighbors
    RETURN r.store_key AS store_key, r.name AS name, r.address AS address, r.gmaps_rating AS rating,
           a.name AS district, a.city AS city, cuisines, categories, atmos, attrs, text_units, neighbors,
           rep.summary AS community_report
    """
    rows = neo4j_client.run(q, {"store_key": store_key, "top_reviews": top_reviews})
    return rows[0] if rows else {}

Similarity graph stats: nodes=200, edges=1220, isolated=0, avg_degree=12.2, min_score=0.62, top_k=8
Embedding-based SIMILAR_TO edges: 1220
Community input graph: vertices=200, edges=1220, components=1, isolated_components=0
Leiden result: 7 communities, modularity=0.4566
Communities:


,community_id,size
0,6,50
1,2,38
2,0,31
3,1,29
4,4,22
5,5,20
6,3,10


## 8. LLM-primary intent parsing bằng Pydantic structured output


In [19]:
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or None

In [20]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

class RestaurantIntent(BaseModel):
    query_type: Literal["search", "similar", "compare", "personalized"] = "search"
    district: Optional[str] = None
    cuisines: List[str] = Field(default_factory=list)
    categories: List[str] = Field(default_factory=list)
    dish_name: Optional[str] = None
    min_rating: Optional[float] = Field(default=None, ge=0, le=5)
    max_distance_km: Optional[float] = Field(default=None, ge=0)
    price_band: Optional[Literal["budget", "mid", "premium"]] = None
    geo_intent: Literal["nearest", "nearby", "normal"] = "normal"
    entity_terms: List[str] = Field(default_factory=list)
    required_attributes: List[Literal["food_quality", "service", "cleanliness", "packaging", "price", "space", "speed"]] = Field(default_factory=list)
    sentiment_pref: Optional[Literal["positive", "neutral", "negative"]] = None
    top_k: int = Field(default=5, ge=1, le=20)

class CommunityReport(BaseModel):
    title: str
    summary: str
    key_strengths: List[str] = Field(default_factory=list)
    cautions: List[str] = Field(default_factory=list)
    representative_restaurants: List[str] = Field(default_factory=list)

class RecommendationAnswer(BaseModel):
    answer: str

def get_llm():
    if LLM_PROVIDER == "anthropic":
        if not ANTHROPIC_API_KEY:
            raise RuntimeError("LLM_PROVIDER=anthropic but ANTHROPIC_API_KEY is missing.")
        return ChatAnthropic(model=ANTHROPIC_MODEL, api_key=ANTHROPIC_API_KEY, temperature=0)
    if LLM_PROVIDER == "openai":
        if not OPENAI_API_KEY:
            raise RuntimeError("LLM_PROVIDER=openai but OPENAI_API_KEY is missing.")
        return ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL, temperature=0)
    raise ValueError(f"Unsupported LLM_PROVIDER={LLM_PROVIDER}. Use 'openai' or 'anthropic'.")

llm = get_llm()
intent_llm = llm.with_structured_output(RestaurantIntent)
report_llm = llm.with_structured_output(CommunityReport)
answer_llm = llm.with_structured_output(RecommendationAnswer)

KNOWN_CUISINES = sorted({x for xs in summary["cuisines"] for x in (xs or [])})
KNOWN_CATEGORIES = sorted({x for xs in summary["categories"] for x in (xs or [])})
KNOWN_DISTRICTS = sorted({str(x) for x in summary["district"].dropna().unique().tolist() if str(x).strip()})

INTENT_SYSTEM = """Bạn là bộ phân tích intent cho hệ gợi ý quán ăn GraphRAG.
Trích xuất truy vấn thành schema RestaurantIntent.
Chỉ chọn district/cuisine/category nếu người dùng thật sự nêu hoặc suy ra rõ.
Các required_attributes hợp lệ: food_quality, service, cleanliness, packaging, price, space, speed.
"""
intent_prompt = ChatPromptTemplate.from_messages([
    ("system", INTENT_SYSTEM + "\nDistrict đã biết: {districts}\nCuisine đã biết: {cuisines}\nCategory đã biết: {categories}"),
    ("human", "Query: {query}"),
])

def parse_intent(query: str) -> dict:
    parsed = (intent_prompt | intent_llm).invoke({
        "query": query,
        "districts": KNOWN_DISTRICTS,
        "cuisines": KNOWN_CUISINES,
        "categories": KNOWN_CATEGORIES,
    })
    #sửa từ đây
    # Nếu LangChain trả Pydantic object
    if hasattr(parsed, "model_dump"):
        return parsed.model_dump()

    # Nếu OpenRouter/LangChain trả dict
    if isinstance(parsed, dict):
        return parsed

    # Fallback nếu trả kiểu khác
    return {
        "query_type": "search",
        "district": None,
        "cuisines": [],
        "categories": [],
        "dish_name": None,
        "min_rating": None,
        "max_distance_km": None,
        "price_band": None,
        "geo_intent": "normal",
        "entity_terms": [],
        "required_attributes": [],
        "sentiment_pref": None,
        "top_k": 5,
        "raw": str(parsed),
    }


## 9. Graph retrieval + neighborhood / subgraph retrieval

In [66]:
def graph_candidate_search(
    intent: dict,
    top_k: int = 10,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[dict]:
    match_lines = ["MATCH (r:Restaurant)"]
    where = []
    params = {"top_k": int(top_k)}
    lat, lng = get_user_location(user_lat, user_lng)
    max_distance = intent.get("max_distance_km") or MAX_DISTANCE_KM
    geo_intent = intent.get("geo_intent", "normal")
    has_user_location = lat is not None and lng is not None
    params.update({"user_lat": lat, "user_lng": lng, "max_distance_km": max_distance})

    if intent.get("district"):
        match_lines.append("MATCH (r)-[:IN_AREA]->(area:Area)")
        where.append("toLower(area.name) CONTAINS toLower($district)")
        params["district"] = intent["district"]
    if intent.get("price_band"):
        match_lines.append("MATCH (r)-[:HAS_PRICE_BAND]->(pb:PriceBand)")
        where.append("pb.name = $price_band")
        params["price_band"] = intent["price_band"]
    if intent.get("cuisines"):
        match_lines.append("MATCH (r)-[:HAS_CUISINE]->(cui:Cuisine)")
        where.append("cui.name IN $cuisines")
        params["cuisines"] = intent["cuisines"]
    if intent.get("categories"):
        match_lines.append("MATCH (r)-[:HAS_CATEGORY]->(cat:Category)")
        where.append("cat.name IN $categories")
        params["categories"] = intent["categories"]
    if intent.get("dish_name"):
        dish_family = normalize_dish_family(intent["dish_name"]) or intent["dish_name"]
        match_lines.append("MATCH (r)-[:SERVES_FAMILY]->(dish:DishFamily)")
        where.append("toLower(dish.name) CONTAINS toLower($dish_name)")
        params["dish_name"] = dish_family
    if intent.get("entity_terms"):
        match_lines.append("MATCH (r)-[:HAS_EXTRACTED_ENTITY]->(ee_filter:ExtractedEntity)")
        where.append("any(term IN $entity_terms WHERE toLower(ee_filter.name) CONTAINS toLower(term))")
        params["entity_terms"] = intent["entity_terms"]
    if intent.get("min_rating") is not None:
        where.append("coalesce(r.rating, r.gmaps_rating, r.foody_rating, 0) >= $min_rating")
        params["min_rating"] = float(intent["min_rating"])
    if has_user_location and max_distance is not None:
        where.append("r.lat IS NOT NULL AND r.lng IS NOT NULL AND point.distance(point({latitude: $user_lat, longitude: $user_lng}), point({latitude: r.lat, longitude: r.lng})) / 1000.0 <= $max_distance_km")
    for i, attr in enumerate(intent.get("required_attributes", [])):
        alias = f"att{i}"
        match_lines.append(f"MATCH (r)-[:HAS_ATTRIBUTE]->({alias}:Attribute {{type: $attr_{i}}})")
        where.append(f"{alias}.score >= 0.15")
        params[f"attr_{i}"] = attr
    q = "\n".join(match_lines) + "\n"
    if where:
        q += "WHERE " + " AND ".join(where) + "\n"
    distance_expr = "point.distance(point({latitude: $user_lat, longitude: $user_lng}), point({latitude: r.lat, longitude: r.lng})) / 1000.0" if has_user_location else "null"
    order_clause = """
    ORDER BY CASE WHEN distance_km IS NULL THEN 1 ELSE 0 END, distance_km ASC,
             CASE WHEN coalesce(r.rating, r.gmaps_rating, r.foody_rating) IS NULL THEN 1 ELSE 0 END,
             coalesce(r.rating, r.gmaps_rating, r.foody_rating) DESC
    """ if geo_intent == "nearest" else """
    ORDER BY CASE WHEN coalesce(r.rating, r.gmaps_rating, r.foody_rating) IS NULL THEN 1 ELSE 0 END,
             coalesce(r.rating, r.gmaps_rating, r.foody_rating) DESC,
             CASE WHEN distance_km IS NULL THEN 1 ELSE 0 END, distance_km ASC
    """
    q += f"""
    OPTIONAL MATCH (r)-[:HAS_ATTRIBUTE]->(att:Attribute)
    OPTIONAL MATCH (r)-[:IN_AREA]->(a:Area)
    OPTIONAL MATCH (r)-[:HAS_CATEGORY]->(cat_ret:Category)
    OPTIONAL MATCH (r)-[:HAS_CUISINE]->(cui_ret:Cuisine)
    OPTIONAL MATCH (r)-[:SERVES_FAMILY]->(df_ret:DishFamily)
    OPTIONAL MATCH (r)-[:HAS_EXTRACTED_ENTITY]->(ee_ret:ExtractedEntity)
    OPTIONAL MATCH (r)-[:IN_COMMUNITY]->(com:Community)-[:HAS_REPORT]->(rep:CommunityReport)
    WITH r, a, rep, collect(DISTINCT {{type: att.type, score: att.score}}) AS attributes,
         collect(DISTINCT cat_ret.name) AS categories,
         collect(DISTINCT cui_ret.name) AS cuisines,
         collect(DISTINCT df_ret.name) AS dish_families,
         collect(DISTINCT {{name: ee_ret.name, type: ee_ret.type}}) AS extracted_entities,
         CASE WHEN r.lat IS NULL OR r.lng IS NULL THEN null ELSE {distance_expr} END AS distance_km
    RETURN r.store_key AS store_key, r.name AS name, r.address AS address,
           a.name AS district, a.city AS city, coalesce(r.rating, r.gmaps_rating, r.foody_rating) AS rating,
           r.lat AS lat, r.lng AS lng, distance_km,
           r.price_band AS price_band, r.top_menu_items AS top_menu_items,
           r.menu_price_min AS menu_price_min, r.menu_price_max AS menu_price_max,
           r.menu_price_median AS menu_price_median,
           categories, cuisines, dish_families, extracted_entities, attributes, rep.summary AS community_report
    {order_clause}
    LIMIT $top_k
    """
    rows = neo4j_client.run(q, params)
    return [add_user_distance_to_record(r, lat, lng) for r in rows]


def subgraph_expand_candidates(
    seed_store_keys: List[str],
    max_neighbors: int = 6,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[dict]:
    if not seed_store_keys:
        return []
    lat, lng = get_user_location(user_lat, user_lng)
    has_user_location = lat is not None and lng is not None
    distance_expr = "point.distance(point({latitude: $user_lat, longitude: $user_lng}), point({latitude: nbr.lat, longitude: nbr.lng})) / 1000.0" if has_user_location else "null"
    q = f"""
    MATCH (seed:Restaurant)-[s:SIMILAR_TO]-(nbr:Restaurant)
    WHERE seed.store_key IN $seed_keys
    OPTIONAL MATCH (nbr)-[:HAS_ATTRIBUTE]->(att:Attribute)
    OPTIONAL MATCH (nbr)-[:IN_AREA]->(a:Area)
    OPTIONAL MATCH (nbr)-[:HAS_CATEGORY]->(cat_ret:Category)
    OPTIONAL MATCH (nbr)-[:HAS_CUISINE]->(cui_ret:Cuisine)
    OPTIONAL MATCH (nbr)-[:SERVES_FAMILY]->(df_ret:DishFamily)
    OPTIONAL MATCH (nbr)-[:HAS_EXTRACTED_ENTITY]->(ee_ret:ExtractedEntity)
    OPTIONAL MATCH (nbr)-[:IN_COMMUNITY]->(:Community)-[:HAS_REPORT]->(rep:CommunityReport)
    WITH nbr, s, a, rep, collect(DISTINCT {{type: att.type, score: att.score}}) AS attributes,
         collect(DISTINCT cat_ret.name) AS categories,
         collect(DISTINCT cui_ret.name) AS cuisines,
         collect(DISTINCT df_ret.name) AS dish_families,
         collect(DISTINCT {{name: ee_ret.name, type: ee_ret.type}}) AS extracted_entities,
         CASE WHEN nbr.lat IS NULL OR nbr.lng IS NULL THEN null ELSE {distance_expr} END AS distance_km
    RETURN nbr.store_key AS store_key, nbr.name AS name, nbr.address AS address,
           a.name AS district, a.city AS city, coalesce(nbr.rating, nbr.gmaps_rating, nbr.foody_rating) AS rating,
           nbr.lat AS lat, nbr.lng AS lng, distance_km,
           nbr.price_band AS price_band, nbr.top_menu_items AS top_menu_items,
           nbr.menu_price_min AS menu_price_min, nbr.menu_price_max AS menu_price_max,
           nbr.menu_price_median AS menu_price_median,
           categories, cuisines, dish_families, extracted_entities,
           s.similarity AS sim, attributes, rep.summary AS community_report
    ORDER BY sim DESC
    LIMIT $limit
    """
    rows = neo4j_client.run(q, {"seed_keys": seed_store_keys, "limit": max_neighbors, "user_lat": lat, "user_lng": lng})
    return [add_user_distance_to_record(r, lat, lng) for r in rows]

def graph_candidate_top5_only(query: str, top_k: int = 5) -> list[str]:
    """
    Lấy top-K candidate chỉ từ graph (Neo4j), đảm bảo luôn có 5 kết quả.
    Không fallback sang vector/text_unit.
    """
    intent = parse_intent(query)

    # 1. Lấy candidate trực tiếp từ graph
    graph_candidates = graph_candidate_search(intent, top_k=10)  # tăng limit lên 10
    graph_candidates = sorted(graph_candidates, key=lambda x: x.get("distance_score", 0), reverse=True)

    # 2. Nếu <5, lấy thêm neighbors từ subgraph
    if len(graph_candidates) < top_k:
        seed_keys = [c["store_key"] for c in graph_candidates]
        neighbors = subgraph_expand_candidates(seed_store_keys=seed_keys, max_neighbors=6)
        # Dedupe
        seen = set(seed_keys)
        for n in neighbors:
            if n["store_key"] not in seen:
                graph_candidates.append(n)
                seen.add(n["store_key"])
            if len(graph_candidates) >= top_k:
                break

    # 3. Trả về top-K store_key
    return [c["store_key"] for c in graph_candidates[:top_k]]

## 10. Vector retrieval cho Restaurant summary và TextUnit


In [22]:
def vector_search_restaurants(query: str, top_k: int = 8, user_lat: Optional[float] = None, user_lng: Optional[float] = None) -> List[dict]:
    hits = qdrant.search(collection_name=COLL_RESTAURANT, query_vector=emb_query(query), limit=top_k, with_payload=True)
    rows = []
    for h in hits:
        rec = {
            "store_key": h.payload["store_key"], "name": h.payload["name"], "address": h.payload.get("address"),
            "district": h.payload.get("district"), "city": h.payload.get("city"), "rating": h.payload.get("rating"),
            "lat": h.payload.get("lat"), "lng": h.payload.get("lng"),
            "price_band": h.payload.get("price_band"), "top_menu_items": h.payload.get("top_menu_items"),
            "dish_families": h.payload.get("dish_families"), "categories": h.payload.get("categories"),
            "cuisines": h.payload.get("cuisines"),
            "menu_price_min": h.payload.get("menu_price_min"), "menu_price_max": h.payload.get("menu_price_max"),
            "menu_price_median": h.payload.get("menu_price_median"),
            "doc_text": h.payload.get("doc_text"),
            "vec_score": round(float(h.score), 4), "source": "restaurant_vector",
        }
        rows.append(add_user_distance_to_record(rec, user_lat, user_lng))
    return rows

def vector_search_text_units(query: str, top_k: int = 16, store_keys: Optional[List[str]] = None) -> List[dict]:
    raw_limit = max(top_k * 5, 100) if store_keys else top_k
    hits = qdrant.search(collection_name=COLL_TEXT_UNIT, query_vector=emb_query(query), limit=raw_limit, with_payload=True)
    rows = []
    allow = set(store_keys) if store_keys else None
    for h in hits:
        p = h.payload
        if allow and p.get("store_key") not in allow:
            continue
        rows.append({
            "store_key": p["store_key"], "text_unit_id": p["text_unit_id"], "review_id": p["review_id"],
            "store_name": p.get("store_name"), "rating": p.get("rating"), "sentiment": p.get("sentiment"),
            "feedback": p.get("feedback"), "aspect_scores": p.get("aspect_scores"), "doc_text": p.get("doc_text"),
            "vec_score": round(float(h.score), 4), "source": "text_unit_vector",
        })
    return rows[:top_k]


## 11. Fusion + rerank bằng Reciprocal Rank Fusion


In [23]:
import math
import numpy as np

def safe_float(x, default=0.0):
    try:
        if x is None:
            return default
        x = float(x)
        if math.isnan(x) or math.isinf(x):
            return default
        return x
    except Exception:
        return default

## 11.1. Cross-encoder reranking (BGE-Reranker)

In [ ]:
%pip install FlagEmbedding

In [24]:
# ============================================================
# STRICT post-fusion validation for retrieval
# Fix:
# - Enforce dish/category constraints from intent["dish_name"] and intent["categories"]
# - Enrich text_unit_vector-only candidates from summary metadata
# - Enforce min_rating and price_band more correctly
# ============================================================

import ast
import re
import unicodedata
from typing import Any, Dict, List, Optional


def safe_float(x: Any, default: float = 0.0) -> float:
    try:
        if x is None:
            return default
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def _strip_accents(s: Any) -> str:
    s = "" if s is None else str(s)
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return s.replace("đ", "d").replace("Đ", "D")


def _norm_text(s: Any) -> str:
    s = _strip_accents(s).lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def _to_list(x: Any) -> List[Any]:
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, tuple) or isinstance(x, set):
        return list(x)
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if s.startswith("[") and s.endswith("]"):
            try:
                y = ast.literal_eval(s)
                if isinstance(y, list):
                    return y
            except Exception:
                pass
        return [s]
    return [x]


def _split_or_terms(term: Any) -> List[str]:
    """
    Tách các query kiểu:
    - "bún hoặc phở" -> ["bún", "phở"]
    - "gà rán hoặc cơm gà" -> ["gà rán", "cơm gà"]
    """
    raw = str(term or "").strip()
    if not raw:
        return []

    norm = _norm_text(raw)

    # bỏ các prefix không phải món
    norm = re.sub(r"\b(quan|mon|tim|goi y|an|muon|toi|minh)\b", " ", norm)
    norm = re.sub(r"\s+", " ", norm).strip()

    parts = re.split(r"\b(?:hoac|or|va|and)\b", norm)
    parts = [p.strip() for p in parts if p.strip()]

    return parts or [norm]


def _canonical_food_term(term: Any) -> Optional[str]:
    """
    Chuẩn hóa food constraint về dạng không dấu để match.
    """
    if term is None:
        return None

    s = _norm_text(term)
    if not s:
        return None

    # bỏ nhiễu
    noise = {
        "nhieu comment that",
        "comment that",
        "rating tot",
        "rating on",
        "gia sinh vien",
        "phu hop sinh vien",
        "an sang",
        "an trua",
        "an toi",
        "khong qua xa",
        "gan day",
        "gan toi",
        "quan an",
    }
    if s in noise:
        return None

    alias = {
        "com ga": "com ga",
        "com ga ran": "com ga",
        "com tam": "com tam",
        "com rang": "com rang",
        "com chien": "com rang",
        "com van phong": "com",
        "com": "com",

        "pho bo": "pho bo",
        "pho ga": "pho ga",
        "pho xao": "pho xao",
        "pho": "pho",

        "bun cha": "bun cha",
        "bun bo hue": "bun bo hue",
        "bun bo": "bun bo",
        "bun rieu": "bun rieu",
        "bun": "bun",

        "banh cuon": "banh cuon",
        "banh cuon nong": "banh cuon",
        "banh trang cuon": "banh trang",
        "banh trang": "banh trang",

        "ga ran": "ga ran",
        "ga": "ga",
    }

    return alias.get(s, s)


def _extract_food_terms_from_intent(intent: Optional[dict]) -> List[str]:
    """
    Lấy constraint món từ intent.
    Quan trọng: phải đọc cả categories và dish_name.
    """
    intent = intent or {}

    raw_terms = []

    for key in ["dish_name", "dish_family", "dish_families", "families"]:
        raw_terms.extend(_to_list(intent.get(key)))

    # Đây là field đang có trong file result của bạn
    raw_terms.extend(_to_list(intent.get("categories")))

    # Chỉ lấy entity_terms nếu nó giống món ăn, tránh lấy "nhiều comment thật"
    food_roots = [
        "com", "pho", "bun", "banh", "ga", "mi", "my", "mien", "chao", "xoi"
    ]

    for x in _to_list(intent.get("entity_terms")):
        nx = _norm_text(x)
        if any(root in nx.split() for root in food_roots):
            raw_terms.append(x)

    terms = []

    for raw in raw_terms:
        for part in _split_or_terms(raw):
            can = _canonical_food_term(part)
            if can and can not in terms:
                terms.append(can)

    return terms


def _candidate_food_fields(c: dict) -> Dict[str, List[str]]:
    families = [_canonical_food_term(x) for x in _to_list(c.get("dish_families"))]
    families = [x for x in families if x]

    categories = [_canonical_food_term(x) for x in _to_list(c.get("categories"))]
    categories = [x for x in categories if x]

    top_items = [_canonical_food_term(x) for x in _to_list(c.get("top_menu_items"))]
    top_items = [x for x in top_items if x]

    name = _canonical_food_term(c.get("name"))

    return {
        "families": families,
        "categories": categories,
        "top_items": top_items,
        "name": [name] if name else [],
    }


def _term_matches_candidate(term: str, c: dict) -> bool:
    fields = _candidate_food_fields(c)

    # Nếu có dish_families thì ưu tiên match bằng dish_families.
    # Không để category nhiễu làm match sai.
    if fields["families"]:
        haystack = fields["families"]
    else:
        # text_unit_vector-only có thể thiếu dish_families, fallback sang name/categories/top_items
        haystack = fields["name"] + fields["categories"] + fields["top_items"]

    if not haystack:
        return False

    term = _canonical_food_term(term)
    if not term:
        return True

    for h in haystack:
        if not h:
            continue

        # term cụ thể: "com ga" không được match "com rang"
        if " " in term:
            if term == h or term in h:
                return True
        else:
            # term rộng: "com", "bun", "pho", "ga"
            if h == term or h.startswith(term + " "):
                return True

    return False


def _candidate_matches_food_intent(c: dict, intent: Optional[dict]) -> bool:
    terms = _extract_food_terms_from_intent(intent)

    if not terms:
        return True

    # OR logic: query "gà rán hoặc cơm gà" chỉ cần match một món
    return any(_term_matches_candidate(term, c) for term in terms)


def _candidate_matches_rating(c: dict, intent: Optional[dict]) -> bool:
    intent = intent or {}

    min_rating = intent.get("min_rating") or intent.get("rating_min")
    if min_rating is None:
        return True

    try:
        min_rating = float(min_rating)
    except Exception:
        return True

    rating = c.get("rating")
    if rating is None:
        rating = c.get("gmaps_rating")
    if rating is None:
        rating = c.get("foody_rating")

    # Nếu user yêu cầu rating mà candidate không có rating hoặc rating 0 thì loại
    if rating is None:
        return False

    rating = safe_float(rating, 0.0)
    if rating <= 0:
        return False

    return rating >= min_rating


def _candidate_matches_price(c: dict, intent: Optional[dict]) -> bool:
    intent = intent or {}

    price_band = str(intent.get("price_band") or "").lower().strip()
    max_price = (
        intent.get("max_price")
        or intent.get("price_max")
        or intent.get("budget_max")
    )

    candidate_band = str(c.get("price_band") or "").lower().strip()

    median = c.get("menu_price_median")
    pmin = c.get("menu_price_min")
    pmax = c.get("menu_price_max")

    numeric_prices = [
        safe_float(x, None)
        for x in [median, pmin, pmax]
        if x is not None and safe_float(x, None) is not None
    ]

    price_ref = numeric_prices[0] if numeric_prices else None

    if max_price is not None:
        try:
            max_price = float(max_price)
            if price_ref is not None:
                return price_ref <= max_price * 1.15
        except Exception:
            pass

    if price_band == "budget":
        if candidate_band == "budget":
            return True
        if price_ref is not None and price_ref <= 70000:
            return True
        return False

    if price_band == "mid":
        if candidate_band in {"budget", "mid"}:
            return True
        if price_ref is not None and price_ref <= 120000:
            return True
        return False

    return True


def _candidate_matches_distance(c: dict, intent: Optional[dict]) -> bool:
    intent = intent or {}

    max_distance = (
        intent.get("max_distance_km")
        or intent.get("distance_km")
        or intent.get("radius_km")
    )

    if max_distance is None:
        return True

    try:
        max_distance = float(max_distance)
    except Exception:
        return True

    distance = c.get("distance_km")

    # Nếu user yêu cầu radius mà candidate không có distance thì loại
    if distance is None:
        return False

    return safe_float(distance, 1e9) <= max_distance * 1.15


def infer_geo_intent(query: str, intent: Optional[dict] = None) -> str:
    intent = intent or {}

    val = str(intent.get("geo_intent") or "").lower().strip()
    if val in {"nearest", "nearby", "near_me", "closest"}:
        return "nearest" if val == "nearest" else "nearby"

    q = _norm_text(query)

    if any(p in q for p in ["gan nhat", "closest", "nearest"]):
        return "nearest"

    if any(p in q for p in ["gan day", "gan toi", "khong qua xa", "quanh day", "ban kinh"]):
        return "nearby"

    return "normal"


# Build summary metadata for enriching candidates missing metadata
def _build_summary_meta_by_key() -> Dict[str, dict]:
    """
    Build metadata lookup từ summary DataFrame.
    An toàn nếu summary bị None / dict / không phải DataFrame.
    """
    if "summary" not in globals() or summary is None:
        print("⚠️ summary not found; cannot build SUMMARY_META_BY_KEY")
        return {}

    if not isinstance(summary, pd.DataFrame):
        print(f"⚠️ summary is not a DataFrame, got {type(summary)}; cannot build SUMMARY_META_BY_KEY")
        return {}

    if summary.empty:
        print("⚠️ summary DataFrame is empty")
        return {}

    if "store_key" not in summary.columns:
        print("⚠️ summary has no store_key column")
        return {}

    meta = {}

    for _, r in summary.iterrows():
        sk = str(r.get("store_key", "")).strip()
        if not sk:
            continue
        meta[sk] = r.to_dict()

    print(f"✅ SUMMARY_META_BY_KEY built: {len(meta)} restaurants")
    return meta





SUMMARY_META_BY_KEY = _build_summary_meta_by_key()


def enrich_candidate_from_summary(c: dict) -> dict:
    """
    Fix text_unit_vector-only rows missing address/dish_families/categories.
    """
    if not isinstance(c, dict):
        return c

    sk = str(c.get("store_key", "") or "").strip()
    if not sk:
        return c

    meta = SUMMARY_META_BY_KEY.get(sk)
    if not meta:
        return c

    alias = {
        "name": "name",
        "address": "address",
        "district": "district",
        "city": "city",
        "rating": "rating",
        "distance_km": "distance_km",
        "price_band": "price_band",
        "menu_price_min": "menu_price_min",
        "menu_price_max": "menu_price_max",
        "menu_price_median": "menu_price_median",
        "top_menu_items": "top_menu_items",
        "dish_families": "dish_families",
        "categories": "categories",
        "cuisines": "cuisines",
        "lat": "lat",
        "lng": "lng",
    }

    for out_col, meta_col in alias.items():
        if c.get(out_col) in [None, "", [], {}] and meta_col in meta:
            c[out_col] = meta.get(meta_col)

    return c


def validate_post_fusion(
    candidates: List[dict],
    intent: Optional[dict] = None,
    query: str = "",
) -> List[dict]:
    """
    Hard validation after RRF and before cross-encoder.

    Nếu query có món rõ ràng thì bắt buộc candidate phải match món đó.
    Không fallback trả bừa candidates sai món nữa.
    """
    intent = intent or {}

    if not candidates:
        return []

    cleaned = []
    seen = set()

    for c in candidates:
        if not isinstance(c, dict):
            continue

        c = enrich_candidate_from_summary(c)

        store_key = str(c.get("store_key", "") or "").strip()
        if not store_key or store_key in seen:
            continue

        # Hard filters
        if not _candidate_matches_food_intent(c, intent):
            continue

        if not _candidate_matches_rating(c, intent):
            continue

        if not _candidate_matches_price(c, intent):
            continue

        if not _candidate_matches_distance(c, intent):
            continue

        seen.add(store_key)
        cleaned.append(c)

    # Không fallback toàn bộ nữa, vì fallback làm lọt sai món.
    # Nếu cleaned rỗng thì trả rỗng để biết retrieval thiếu dữ liệu hoặc filter quá chặt.
    return cleaned

✅ SUMMARY_META_BY_KEY built: 200 restaurants


In [25]:
# ============================================================
# Base Hybrid Retrieval before Cross-Encoder
# Đây là hàm hybrid RRF gốc.
# Cross-encoder sẽ gọi hàm này rồi rerank lại.
# ============================================================

from typing import Any, Dict, List, Optional, Tuple
import math


def _safe_float_local(x: Any, default: float = 0.0) -> float:
    try:
        if x is None:
            return default
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def _get_store_key(c: dict) -> str:
    payload = c.get("payload") if isinstance(c.get("payload"), dict) else {}

    return str(
        c.get("store_key")
        or payload.get("store_key")
        or c.get("id")
        or ""
    ).strip()


def _merge_candidate(base: dict, new: dict) -> dict:
    """
    Merge metadata từ nhiều nguồn retrieval vào cùng một candidate theo store_key.
    Ưu tiên giữ field đã có, bổ sung field còn thiếu.
    """
    if not isinstance(base, dict):
        base = {}

    if not isinstance(new, dict):
        return base

    payload = new.get("payload") if isinstance(new.get("payload"), dict) else {}
    merged_new = {**payload, **new}

    for k, v in merged_new.items():
        if k == "payload":
            continue

        if k not in base or base.get(k) in [None, "", [], {}]:
            base[k] = v

    # merge evidence
    old_ev = base.get("evidence") or []
    new_ev = merged_new.get("evidence") or []

    if isinstance(old_ev, str):
        old_ev = [old_ev]
    if isinstance(new_ev, str):
        new_ev = [new_ev]

    ev = []
    for x in old_ev + new_ev:
        sx = str(x).strip()
        if sx and sx not in ev:
            ev.append(sx)

    base["evidence"] = ev[:5]

    # merge source flags
    old_flags = base.get("source_flags") or []
    new_flags = merged_new.get("source_flags") or []

    if isinstance(old_flags, str):
        old_flags = [old_flags]
    if isinstance(new_flags, str):
        new_flags = [new_flags]

    flags = []
    for x in old_flags + new_flags:
        sx = str(x).strip()
        if sx and sx not in flags:
            flags.append(sx)

    base["source_flags"] = flags

    return base


def _rrf_fuse_sources(
    source_results: Dict[str, List[dict]],
    rrf_k: int = 60,
) -> List[dict]:
    """
    Reciprocal Rank Fusion theo store_key.

    Mỗi nguồn retrieval trả list candidate.
    Candidate cùng store_key sẽ được gộp lại.
    """
    by_store = {}

    for source_name, rows in source_results.items():
        rows = rows or []

        for rank, row in enumerate(rows, start=1):
            if not isinstance(row, dict):
                continue

            store_key = _get_store_key(row)

            if not store_key:
                continue

            if store_key not in by_store:
                by_store[store_key] = {
                    "store_key": store_key,
                    "final_score": 0.0,
                    "source_flags": [],
                    "evidence": [],
                }

            c = by_store[store_key]

            c = _merge_candidate(c, row)
            c["store_key"] = store_key
            c = enrich_candidate_from_summary(c)

            # RRF score
            c["final_score"] = float(c.get("final_score") or 0.0) + 1.0 / (rrf_k + rank)

            # Track source
            flags = c.get("source_flags") or []
            if source_name not in flags:
                flags.append(source_name)
            c["source_flags"] = flags

            by_store[store_key] = c

    ranked = list(by_store.values())
    ranked.sort(key=lambda x: float(x.get("final_score") or 0.0), reverse=True)

    return ranked


def hybrid_retrieve_rrf(
    query: str,
    top_k: int = 5,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> Tuple[dict, List[dict]]:
    """
    Base hybrid retrieval:
    1. parse_intent(query)
    2. graph_candidate_search(intent)
    3. vector_search_restaurants(query)
    4. vector_search_text_units(query)
    5. RRF fuse
    6. validate_post_fusion
    """

    # 1. Intent
    intent = parse_intent(query)

    if intent is None:
        intent = {}

    if not isinstance(intent, dict):
        intent = {"raw_intent": intent}

    # inject user location nếu có
    if user_lat is not None:
        intent["user_lat"] = float(user_lat)
    if user_lng is not None:
        intent["user_lng"] = float(user_lng)

    # 2. Retrieval sources
    graph_rows = []
    restaurant_vec_rows = []
    text_unit_rows = []

    try:
        graph_rows = graph_candidate_search(intent, top_k=max(top_k * 4, 20)) or []
    except Exception as e:
        print(f"⚠️ graph_candidate_search failed: {type(e).__name__}: {e}")

    try:
        restaurant_vec_rows = vector_search_restaurants(query, top_k=max(top_k * 4, 20)) or []
    except Exception as e:
        print(f"⚠️ vector_search_restaurants failed: {type(e).__name__}: {e}")

    try:
        text_unit_rows = vector_search_text_units(query, top_k=max(top_k * 6, 30)) or []
    except Exception as e:
        print(f"⚠️ vector_search_text_units failed: {type(e).__name__}: {e}")

    # 3. Mark source flags
    for r in graph_rows:
        if isinstance(r, dict):
            r["source_flags"] = list(set((r.get("source_flags") or []) + ["graph"]))

    for r in restaurant_vec_rows:
        if isinstance(r, dict):
            r["source_flags"] = list(set((r.get("source_flags") or []) + ["restaurant_vector"]))

    for r in text_unit_rows:
        if isinstance(r, dict):
            r["source_flags"] = list(set((r.get("source_flags") or []) + ["text_unit_vector"]))

    # 4. RRF fusion
    fused = _rrf_fuse_sources({
        "graph": graph_rows,
        "restaurant_vector": restaurant_vec_rows,
        "text_unit_vector": text_unit_rows,
    })

    # 5. Constraint validation
    fused = validate_post_fusion(fused, intent, query)

    # 6. Sort geo nếu query là nearest
    try:
        geo_intent = infer_geo_intent(query, intent)
    except Exception:
        geo_intent = None

    if geo_intent == "nearest":
        fused.sort(
            key=lambda x: (
                x.get("distance_km") is None,
                _safe_float_local(x.get("distance_km"), 1e9),
                -_safe_float_local(x.get("final_score"), 0.0),
            )
        )
    else:
        fused.sort(key=lambda x: _safe_float_local(x.get("final_score"), 0.0), reverse=True)

    return intent, fused[:top_k]


print("✅ Base hybrid_retrieve_rrf ready")

✅ Base hybrid_retrieve_rrf ready


In [26]:
# Cross-encoder reranking after RRF/constraint validation.
try:
    from FlagEmbedding import FlagReranker
    _RERANKER_AVAILABLE = True
except ImportError:
    _RERANKER_AVAILABLE = False
    print("FlagEmbedding not installed. Cross-encoder reranking disabled. Run: pip install FlagEmbedding")

CROSS_ENCODER_MODEL = os.getenv("CROSS_ENCODER_MODEL", "BAAI/bge-reranker-base")
_reranker = None

def get_reranker():
    global _reranker
    if not _RERANKER_AVAILABLE:
        return None
    if _reranker is None:
        print(f"Loading cross-encoder: {CROSS_ENCODER_MODEL}...")
        _reranker = FlagReranker(CROSS_ENCODER_MODEL, use_fp16=torch.cuda.is_available())
        print("Cross-encoder loaded")
    return _reranker

def build_cross_encoder_passage(c: dict) -> str:
    evidence = " | ".join((c.get("evidence") or [])[:2])
    attrs = ", ".join(f"{a.get('type')}={safe_float(a.get('score')):+.2f}" for a in (c.get("attributes") or []) if a and a.get("score") is not None)
    extracted = ", ".join(f"{e.get('type')}:{e.get('name')}" for e in (c.get("extracted_entities") or []) if isinstance(e, dict) and e.get("name"))
    parts = [
        f"Name: {c.get('name')}",
        f"Address: {c.get('address')}",
        f"District: {c.get('district')}, {c.get('city')}",
        f"Rating: {c.get('rating')}",
        f"Distance_km: {c.get('distance_km')}",
        f"Price_band: {c.get('price_band')}",
        f"Menu_price: {c.get('menu_price_min')} - {c.get('menu_price_max')}, median={c.get('menu_price_median')}",
        f"Categories: {', '.join(c.get('categories') or [])}",
        f"Cuisines: {', '.join(c.get('cuisines') or [])}",
        f"Dish_families: {', '.join(c.get('dish_families') or [])}",
        f"Top_menu_items: {', '.join(c.get('top_menu_items') or [])}",
        f"Attributes: {attrs}",
        f"Extracted_entities: {extracted}",
        f"Community_report: {c.get('community_report') or ''}",
        f"Evidence: {evidence}",
    ]
    return "\n".join([p for p in parts if p and not p.endswith("None")])[:1200]

def _minmax_normalize(values: List[float]) -> List[float]:
    if not values:
        return []
    lo, hi = min(values), max(values)
    if abs(hi - lo) < 1e-9:
        return [0.5 for _ in values]
    return [(v - lo) / (hi - lo) for v in values]

def cross_encoder_rerank(
    query: str,
    candidates: List[dict],
    top_k: int = 5,
    ce_weight: float = 0.3,
    intent: Optional[dict] = None,
) -> List[dict]:
    intent = intent or {}
    candidates = validate_post_fusion(candidates, intent, query)
    reranker = get_reranker()
    if reranker is None or not candidates:
        return candidates[:top_k]

    passages = [build_cross_encoder_passage(c) for c in candidates]
    raw = reranker.compute_score([[query, p] for p in passages], normalize=False)
    raw_scores = [float(x) for x in (raw if isinstance(raw, list) else [raw] * len(candidates))]
    ce_scores = _minmax_normalize(raw_scores)

    for c, raw_s, ce_s in zip(candidates, raw_scores, ce_scores):
        rrf_s = float(c.get("final_score") or 0.0)
        c["ce_raw_score"] = round(raw_s, 4)
        c["ce_score"] = round(float(ce_s), 4)
        c["final_score_before_ce"] = rrf_s
        c["final_score"] = (1.0 - ce_weight) * rrf_s + ce_weight * float(ce_s)

    geo_intent = infer_geo_intent(query, intent)
    if geo_intent == "nearest":
        candidates.sort(key=lambda x: (x.get("distance_km") is None, float(x.get("distance_km") or 1e9), -float(x.get("final_score") or 0)))
    else:
        candidates.sort(key=lambda x: x["final_score"], reverse=True)
    return candidates[:top_k]

_hybrid_retrieve_rrf = hybrid_retrieve_rrf

def hybrid_retrieve(
    query: str,
    top_k: int = 5,
    use_cross_encoder: bool = True,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> Tuple[dict, List[dict]]:
    intent, ranked_rrf = _hybrid_retrieve_rrf(
        query,
        top_k=max(top_k * 2, 10),
        user_lat=user_lat,
        user_lng=user_lng,
    )
    if use_cross_encoder:
        ranked = cross_encoder_rerank(query, ranked_rrf, top_k=top_k, intent=intent)
    else:
        ranked = validate_post_fusion(ranked_rrf, intent, query)[:top_k]
    return intent, ranked

print("Cross-encoder reranker module ready (lazy-loads on first use)")


Cross-encoder reranker module ready (lazy-loads on first use)


## 12. Community reports + generation/recommendation


In [45]:
# FIX (Vấn đề 9): Community report prompt mở rộng với anti-hallucination,
# hierarchy guidance, format rõ ràng — tăng chất lượng global query.
REPORT_SYSTEM = """Bạn là chuyên gia phân tích hệ thống quán ăn sử dụng phương pháp GraphRAG.
Nhiệm vụ: Viết CommunityReport cho một cụm (community) nhà hàng được phát hiện tự động.

== YÊU CẦU BẮT BUỘC ==
1. Chỉ dùng thông tin từ context cung cấp. Không được suy diễn ngoài context.
2. title: tên ngắn gọn mô tả đặc trưng cụm (ví dụ: "Cụm quán bún bò Hoàng Mai rating cao")
3. summary: 3-5 câu mô tả đặc điểm chung, khu vực địa lý, phân khúc giá, điểm nổi bật.
4. key_strengths: 2-5 điểm mạnh cụ thể với dẫn chứng từ aspect scores hoặc review.
   Ví dụ: "food_quality trung bình +0.72 — phần lớn review khen đồ ăn ngon"
5. cautions: 1-3 điểm yếu hoặc thông tin cần lưu ý (nếu context không có, để list rỗng).
6. representative_restaurants: 2-5 tên quán tiêu biểu nhất trong cụm.

== CHẤT LƯỢNG ==
- Không viết câu chung chung như "quán có không khí tốt" nếu không có evidence.
- Không lặp lại tên quán trong summary.
- Nếu cụm nhỏ (< 3 quán): ghi rõ "Cụm nhỏ, kết quả có thể không đại diện."
- Nếu context thiếu dữ liệu: ghi rõ "Dữ liệu aspect không đầy đủ" trong cautions.
- Không bịa tên món ăn, địa điểm hoặc số liệu không có trong context.

== FORMAT ==
Trả về JSON theo Pydantic schema CommunityReport.
"""

report_prompt = ChatPromptTemplate.from_messages([
    ("system", REPORT_SYSTEM),
    ("human", "Community id: {community_id}\n\nContext (JSON list of restaurants in this community):\n{context}\n\nViết CommunityReport cho community trên."),
])
# =========================
# CommunityReport disk cache
# =========================
import os
import json
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional

CACHE_DIR_PATH = Path(os.getenv("CACHE_DIR", ".cache/graphrag"))
CACHE_DIR_PATH.mkdir(parents=True, exist_ok=True)

COMMUNITY_REPORT_CACHE_PATH = CACHE_DIR_PATH / "community_reports.json"
COMMUNITY_REPORT_LOG_PATH = CACHE_DIR_PATH / "community_report_progress.jsonl"


def append_community_report_log(record: Dict[str, Any]) -> None:
    with COMMUNITY_REPORT_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
        f.flush()
# Tăng version này nếu bạn đổi prompt/schema và muốn ép generate lại report.
COMMUNITY_REPORT_PROMPT_VERSION = os.getenv("COMMUNITY_REPORT_PROMPT_VERSION", "v1")



def load_community_report_cache() -> Dict[str, Any]:
    if not COMMUNITY_REPORT_CACHE_PATH.exists():
        print("ℹ️ No community report cache found, starting fresh")
        return {}

    try:
        with COMMUNITY_REPORT_CACHE_PATH.open("r", encoding="utf-8") as f:
            cache = json.load(f)
        print(f"✅ Loaded community report cache: {len(cache)} items")
        return cache
    except Exception as e:
        print(f"⚠️ Could not load community report cache, starting fresh: {e}")
        return {}


COMMUNITY_REPORT_CACHE = load_community_report_cache()
COMMUNITY_REPORT_CACHE_DIRTY = False
COMMUNITY_REPORT_CACHE_HITS = 0
COMMUNITY_REPORT_CACHE_MISSES = 0


def save_community_report_cache() -> None:
    global COMMUNITY_REPORT_CACHE_DIRTY

    if not COMMUNITY_REPORT_CACHE_DIRTY:
        print(
            f"CommunityReport cache unchanged | "
            f"items={len(COMMUNITY_REPORT_CACHE)}, "
            f"hits={COMMUNITY_REPORT_CACHE_HITS}, "
            f"misses={COMMUNITY_REPORT_CACHE_MISSES}"
        )
        return

    tmp_path = Path(str(COMMUNITY_REPORT_CACHE_PATH) + ".tmp")

    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(
            COMMUNITY_REPORT_CACHE,
            f,
            ensure_ascii=False,
            indent=2,
        )

    tmp_path.replace(COMMUNITY_REPORT_CACHE_PATH)
    COMMUNITY_REPORT_CACHE_DIRTY = False

    print(
        f"✅ Saved CommunityReport cache: {len(COMMUNITY_REPORT_CACHE)} items -> {COMMUNITY_REPORT_CACHE_PATH} | "
        f"hits={COMMUNITY_REPORT_CACHE_HITS}, misses={COMMUNITY_REPORT_CACHE_MISSES}"
    )


def _sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def _community_report_cache_key(context: str) -> str:
    """
    Cache key phụ thuộc vào:
    - model
    - prompt version
    - REPORT_SYSTEM
    - toàn bộ community context

    Nếu reset Neo4j nhưng community context giống nhau thì vẫn hit cache.
    Nếu dữ liệu/prompt/model đổi thì tự generate lại.
    """
    raw = json.dumps(
        {
            "model": OPENAI_MODEL,
            "prompt_version": COMMUNITY_REPORT_PROMPT_VERSION,
            "report_system_sha": _sha256_text(REPORT_SYSTEM),
            "context_sha": _sha256_text(context),
        },
        ensure_ascii=False,
        sort_keys=True,
    )
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def _as_str_list(value: Any) -> List[str]:
    if value is None:
        return []

    if isinstance(value, str):
        value = [value]

    if not isinstance(value, list):
        value = [value]

    out = []
    for x in value:
        if x is None:
            continue
        if isinstance(x, (dict, list)):
            out.append(json.dumps(x, ensure_ascii=False))
        else:
            out.append(str(x))
    return out


def _strip_json_fence(text: str) -> str:
    text = str(text).strip()

    if text.startswith("```json"):
        text = text[len("```json"):].strip()
    elif text.startswith("```"):
        text = text[len("```"):].strip()

    if text.endswith("```"):
        text = text[:-3].strip()

    return text


def _normalize_community_report(report: Any) -> Dict[str, Any]:
    """
    Chuẩn hóa output từ Pydantic/dict/AIMessage về dict thuần
    để lưu cache và upsert Neo4j.
    """
    if hasattr(report, "model_dump"):
        report = report.model_dump()

    elif isinstance(report, dict):
        report = report

    elif hasattr(report, "content"):
        content = _strip_json_fence(report.content)
        try:
            report = json.loads(content)
        except Exception:
            report = {
                "title": "",
                "summary": str(report.content),
                "key_strengths": [],
                "cautions": ["LLM returned non-JSON content."],
                "representative_restaurants": [],
            }

    else:
        report = {
            "title": "",
            "summary": str(report),
            "key_strengths": [],
            "cautions": ["LLM returned unexpected object type."],
            "representative_restaurants": [],
        }

    return {
        "title": str(report.get("title", "") or ""),
        "summary": str(report.get("summary", "") or ""),
        "key_strengths": _as_str_list(report.get("key_strengths")),
        "cautions": _as_str_list(report.get("cautions")),
        "representative_restaurants": _as_str_list(report.get("representative_restaurants")),
    }


def get_or_create_community_report(cid: str, context: str) -> Dict[str, Any]:
    """
    Có cache thì lấy từ .cache/graphrag/community_reports.json.
    Chưa có thì mới gọi OpenAI, sau đó lưu cache ngay.
    """
    global COMMUNITY_REPORT_CACHE_DIRTY
    global COMMUNITY_REPORT_CACHE_HITS
    global COMMUNITY_REPORT_CACHE_MISSES

    key = _community_report_cache_key(context)

    if key in COMMUNITY_REPORT_CACHE:
        COMMUNITY_REPORT_CACHE_HITS += 1
        cached = COMMUNITY_REPORT_CACHE[key].get("report", {})
        report = _normalize_community_report(cached)

        append_community_report_log({
            "status": "cache_hit",
            "community_id": cid,
            "cache_key": key,
            "model": OPENAI_MODEL,
            "prompt_version": COMMUNITY_REPORT_PROMPT_VERSION,
            "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        })

        return report

    COMMUNITY_REPORT_CACHE_MISSES += 1

    t0 = time.time()
    report = (report_prompt | report_llm).invoke({
        "community_id": cid,
        "context": context,
    })

    report = _normalize_community_report(report)

    COMMUNITY_REPORT_CACHE[key] = {
        "report": report,
        "model": OPENAI_MODEL,
        "prompt_version": COMMUNITY_REPORT_PROMPT_VERSION,
        "report_system_sha": _sha256_text(REPORT_SYSTEM),
        "context_sha": _sha256_text(context),
    }

    COMMUNITY_REPORT_CACHE_DIRTY = True

    # Lưu ngay sau mỗi lần gọi OpenAI để nếu notebook bị dừng giữa chừng
    # thì không mất kết quả đã gọi.
    save_community_report_cache()

    append_community_report_log({
        "status": "llm_done",
        "community_id": cid,
        "cache_key": key,
        "model": OPENAI_MODEL,
        "prompt_version": COMMUNITY_REPORT_PROMPT_VERSION,
        "latency_sec": round(time.time() - t0, 3),
        "title": report.get("title", ""),
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    })
    return dict(report)

def build_community_context(community_id: str, limit: int = 30) -> str:
    rows = neo4j_client.run("""
    MATCH (c:Community {community_id: $community_id})<-[:IN_COMMUNITY]-(r:Restaurant)
    OPTIONAL MATCH (r)-[:HAS_ATTRIBUTE]->(a:Attribute)
    WITH r,
         [x IN collect(DISTINCT CASE
             WHEN a IS NULL THEN NULL
             ELSE {type: a.type, score: a.score}
         END) WHERE x IS NOT NULL] AS attrs
    OPTIONAL MATCH (r)<-[:ABOUT]-(tu:TextUnit)
    WITH r, attrs,
         [x IN collect(DISTINCT tu.text) WHERE x IS NOT NULL][..3] AS text_units
    OPTIONAL MATCH (r)-[:SERVES_FAMILY]->(df:DishFamily)
    WITH r, attrs, text_units, [x IN collect(DISTINCT df.name) WHERE x IS NOT NULL] AS dish_families,
         coalesce(r.rating, r.gmaps_rating, r.foody_rating) AS rating
    RETURN r.name AS name, r.address AS address, rating,
           r.district AS district, r.city AS city, r.price_band AS price_band,
           r.menu_price_median AS menu_price_median, r.top_menu_items AS top_menu_items,
           dish_families, attrs, text_units
    ORDER BY rating DESC, name
    LIMIT $limit
    """, {"community_id": str(community_id), "limit": limit})
    return json.dumps(rows, ensure_ascii=False, indent=2)

def upsert_community_reports():
    communities_list = neo4j_client.run(
        "MATCH (c:Community)<-[:IN_COMMUNITY]-(:Restaurant) RETURN c.community_id AS community_id ORDER BY community_id"
    )
    rows = []
    for c in tqdm(communities_list, desc="Community reports"):
        cid = str(c["community_id"])
        context = build_community_context(cid)
        if not context or context == "[]":
            print(f"  ⚠️  Community {cid}: empty context, skipping")
            append_community_report_log({
                "status": "skipped_empty_context",
                "community_id": cid,
                "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            })
            continue
        try:
            report = get_or_create_community_report(cid, context)

            row = {
                "community_id": cid,
                "report_id": f"community_report_{cid}",
                "title": report.get("title", ""),
                "summary": report.get("summary", ""),
                "key_strengths": report.get("key_strengths", []),
                "cautions": report.get("cautions", []),
                "representative_restaurants": report.get("representative_restaurants", []),
            }

            rows.append(row)

            # ✅ Log cả bước chuẩn bị upsert
            append_community_report_log({
                "status": "prepared_for_neo4j_upsert",
                "community_id": cid,
                "report_id": row["report_id"],
                "title": row["title"],
                "cache_hits": COMMUNITY_REPORT_CACHE_HITS,
                "cache_misses": COMMUNITY_REPORT_CACHE_MISSES,
                "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            })

        except Exception as e:
            print(f"  ⚠️ Community {cid} report failed: {type(e).__name__}: {e}")

            append_community_report_log({
                "status": "failed",
                "community_id": cid,
                "error_type": type(e).__name__,
                "error": str(e),
                "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            })
    if not rows:
        print("⚠️ No community reports to upsert.")
        save_community_report_cache()
        return 
    neo4j_client.run("""
    UNWIND $rows AS row
    MATCH (c:Community {community_id: row.community_id})
    MERGE (cr:CommunityReport {report_id: row.report_id})
    SET cr.title = row.title, cr.summary = row.summary, cr.key_strengths = row.key_strengths,
        cr.cautions = row.cautions, cr.representative_restaurants = row.representative_restaurants,
        cr.updated_at = datetime()
    MERGE (c)-[:HAS_REPORT]->(cr)
    """, {"rows": rows})

    save_community_report_cache()

    append_community_report_log({
        "status": "neo4j_upsert_done",
        "num_reports": len(rows),
        "cache_hits": COMMUNITY_REPORT_CACHE_HITS,
        "cache_misses": COMMUNITY_REPORT_CACHE_MISSES,
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    })

    print(
        f"✅ Upserted community reports: {len(rows)} | "
        f"cache_hits={COMMUNITY_REPORT_CACHE_HITS}, "
        f"cache_misses={COMMUNITY_REPORT_CACHE_MISSES}"
    )


def check_community_report_coverage() -> dict:
    rows = neo4j_client.run("""
    MATCH (r:Restaurant)
    OPTIONAL MATCH (r)-[:IN_COMMUNITY]->(c:Community)
    OPTIONAL MATCH (c)-[:HAS_REPORT]->(cr:CommunityReport)
    WITH count(DISTINCT r) AS restaurants,
         count(DISTINCT c) AS communities,
         count(DISTINCT cr) AS reports,
         count(DISTINCT CASE WHEN cr IS NOT NULL THEN r END) AS restaurants_with_report
    RETURN restaurants, communities, reports, restaurants_with_report,
           CASE WHEN restaurants = 0 THEN 0.0 ELSE toFloat(restaurants_with_report) / restaurants END AS restaurant_report_coverage
    """)
    stats = dict(rows[0]) if rows else {"restaurants": 0, "communities": 0, "reports": 0, "restaurants_with_report": 0, "restaurant_report_coverage": 0.0}
    print("CommunityReport coverage:", stats)
    if stats.get("communities", 0) == 0 or stats.get("reports", 0) == 0:
        print("WARNING: CommunityReport is empty; do not claim full GraphRAG behavior until reports exist.")
    elif float(stats.get("restaurant_report_coverage") or 0) < 0.8:
        print("WARNING: CommunityReport coverage is low; global/community context may be incomplete.")
    return stats

if RUN_COMMUNITY_REPORTS:
    upsert_community_reports()
else:
    print("Skipping community report generation. Set RUN_COMMUNITY_REPORTS=true to run it.")
community_report_stats = check_community_report_coverage()

RECOMMEND_SYSTEM = """Bạn là trợ lý gợi ý quán ăn.
QUY TẮC BẮT BUỘC:
1. Chỉ dùng các quán có trong Context đã retrieval/rerank.
2. Giữ nguyên thứ tự quán theo Context, không tự reorder.
3. Giữ nguyên tên quán như trong Context, không đổi tên, không gom thành "cụm quán".
4. Nếu một quán thiếu evidence, vẫn có thể nêu nhưng phải ghi rõ "chưa có review evidence trực tiếp".
5. Không tự bịa rating, địa chỉ, review, món ăn hoặc ưu điểm không có trong Context, nếu context thiếu, nói rõ thiếu dữ liệu, không tự bịa.
6. Trả lời bằng tiếng Việt, ngắn gọn, dễ đọc.
7. Với mỗi quán, nêu:
   - Tên quán
   - Địa chỉ
   - Rating nếu có
   - Vì sao phù hợp với query
   - Evidence nếu có

- nếu context thiếu, nói rõ thiếu dữ liệu, không tự bịa.
"""
recommend_prompt = ChatPromptTemplate.from_messages([
    ("system", RECOMMEND_SYSTEM),
    ("human", "Query: {query}\nIntent: {intent}\n\nContext:\n{context}"),
])

# def is_valid_result(r):
#     score = safe_float(r.get("final_score"))
#     evidence = r.get("evidence") or []
#     rating = r.get("rating")

#     return score > 0 and (evidence or rating is not None)


def _as_str_join(value) -> str:
    if not value:
        return ""
    if isinstance(value, str):
        return value
    return ", ".join(str(x) for x in value if x is not None)


def format_retrieval_context(rows: List[dict]) -> str:
    rows = [
        r for r in rows
        if (r.get("evidence") and len(r.get("evidence")) > 0) or r.get("rating") is not None
    ]

    lines = []

    for i, r in enumerate(rows, 1):
        evidence = r.get("evidence") or []
        ev = " | ".join(str(x) for x in evidence[:2]) if evidence else ""

        score = safe_float(r.get("final_score"))
        ce_score = r.get("ce_score")
        ce_score = safe_float(ce_score) if ce_score is not None else None

        distance_km = r.get("distance_km")
        distance_km = safe_float(distance_km) if distance_km is not None else None

        ce = f" | ce_score={ce_score:.4f}" if ce_score is not None else ""
        dist = f" | distance_km={distance_km:.2f}" if distance_km is not None else ""

        source_flags = _as_str_join(r.get("source_flags") or [])
        dish_families = _as_str_join(r.get("dish_families") or [])
        categories = _as_str_join(r.get("categories") or [])

        lines.append(
            f"{i}. {r.get('name')} | rating={r.get('rating')} | district={r.get('district')}{dist} | "
            f"score={score:.4f}{ce} | sources={source_flags}\n"
            f"   address={r.get('address')}\n"
            f"   price_band={r.get('price_band')} | dish_families={dish_families} | categories={categories}\n"
            f"   community_report={r.get('community_report') or ''}\n"
            f"   evidence={ev}"
        )

    return "\n\n".join(lines)

def recommend(query: str, top_k: int = 5, user_lat: Optional[float] = None, user_lng: Optional[float] = None) -> str:
    intent, ranked = hybrid_retrieve(query, top_k=top_k, user_lat=user_lat, user_lng=user_lng)

    context = format_retrieval_context(ranked)
    if not context.strip():
        return "Mình chưa tìm thấy quán phù hợp trong dữ liệu hiện có. Có thể query quá hẹp hoặc dữ liệu review/evidence chưa đủ."

    answer = (recommend_prompt | answer_llm).invoke({
        "query": query,
        "intent": json.dumps(intent, ensure_ascii=False),
        "context": context,
    })
    #return answer.answer
    # Nếu trả Pydantic object
    if hasattr(answer, "answer"):
        return answer.answer

    # Nếu trả dict
    if isinstance(answer, dict):
        return answer.get("answer", str(answer))

    # Nếu trả AIMessage
    if hasattr(answer, "content"):
        return answer.content

    return str(answer)


ℹ️ No community report cache found, starting fresh


Community reports:   0%|          | 0/200 [00:00<?, ?it/s]

✅ Saved CommunityReport cache: 1 items -> .cache\graphrag\community_reports.json | hits=0, misses=1
✅ Saved CommunityReport cache: 2 items -> .cache\graphrag\community_reports.json | hits=30, misses=2
✅ Saved CommunityReport cache: 3 items -> .cache\graphrag\community_reports.json | hits=58, misses=3
✅ Saved CommunityReport cache: 4 items -> .cache\graphrag\community_reports.json | hits=95, misses=4
✅ Saved CommunityReport cache: 5 items -> .cache\graphrag\community_reports.json | hits=104, misses=5
✅ Saved CommunityReport cache: 6 items -> .cache\graphrag\community_reports.json | hits=125, misses=6
✅ Saved CommunityReport cache: 7 items -> .cache\graphrag\community_reports.json | hits=144, misses=7
CommunityReport cache unchanged | items=7, hits=193, misses=7
✅ Upserted community reports: 200 | cache_hits=193, cache_misses=7
CommunityReport coverage: {'restaurants': 200, 'communities': 7, 'reports': 7, 'restaurants_with_report': 200, 'restaurant_report_coverage': 1.0}


## 13. Evaluation cho retrieval quality

In [ ]:
# ============================================================
# Wrapper hybrid retrieval
# ============================================================

from typing import List

def canon_store_key(x):
    """Normalize store_key to string"""
    if x is None:
        return ""
    s = str(x).strip()
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]
    return s

def _unique_store_keys(rows, top_k: int = 5) -> List[str]:
    seen = set()
    out = []
    for r in rows or []:
        if not isinstance(r, dict):
            continue
        sk = canon_store_key(r.get("store_key"))
        if not sk or sk in seen:
            continue
        seen.add(sk)
        out.append(sk)
        if len(out) >= top_k:
            break
    return out

def retrieve_hybrid_no_ce(query: str, top_k: int = 5, user_lat: float = None, user_lng: float = None) -> list[str]:
    """Hybrid retrieval without cross-encoder, tính cả khoảng cách user nếu có"""
    _, ranked = hybrid_retrieve(
        query,
        top_k=top_k,
        use_cross_encoder=False,
        user_lat=user_lat,
        user_lng=user_lng
    )
    return _unique_store_keys(ranked, top_k=top_k)


def retrieve_hybrid_with_ce(query: str, top_k: int = 5, user_lat: float = None, user_lng: float = None) -> list[str]:
    """Hybrid retrieval with cross-encoder, tính cả khoảng cách user nếu có"""
    _, ranked = hybrid_retrieve(
        query,
        top_k=top_k,
        use_cross_encoder=True,
        user_lat=user_lat,
        user_lng=user_lng
    )
    return _unique_store_keys(ranked, top_k=top_k)

In [95]:
def recall_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    rel = set(relevant); ret = retrieved[:k]
    return len(rel.intersection(ret)) / len(rel) if rel else 0.0

def mrr_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    rel = set(relevant)
    for i, sid in enumerate(retrieved[:k], start=1):
        if sid in rel:
            return 1.0 / i
    return 0.0

def ndcg_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    rel = set(relevant)
    dcg = 0.0
    for i, sid in enumerate(retrieved[:k], start=1):
        gain = 1.0 if sid in rel else 0.0
        dcg += gain / math.log2(i + 1)
    ideal_hits = min(len(rel), k)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def generate_eval_dataset_from_graph(
    n_queries: int = 20,
    seed: int = 42,
) -> List[dict]:
    """FIX (Vấn đề 8): Auto-generate ground truth eval dataset bằng LLM từ graph.

    Strategy:
    1. Sample n_queries community reports hoặc restaurant clusters từ Neo4j
    2. Gọi LLM sinh câu hỏi tự nhiên dựa trên context của cluster
    3. Dùng restaurants trong cluster làm relevant_store_keys (silver labels)

    Đây là "silver standard" — không phải human labels, nhưng đủ để:
    - So sánh các variant (vector-only vs graph vs hybrid)
    - Detect regression khi thay đổi pipeline
    """
    GEN_SYSTEM = """Bạn tạo câu hỏi tìm kiếm quán ăn tự nhiên bằng tiếng Việt dựa trên thông tin về một nhóm nhà hàng.
Câu hỏi phải tự nhiên như người dùng thật hỏi (không quá kỹ thuật).
Trả về JSON: {"query": "...", "reasoning": "tại sao nhà hàng này relevant"}
Chỉ trả về JSON, không giải thích thêm."""

    rows = neo4j_client.run("""
    MATCH (c:Community)-[:HAS_REPORT]->(cr:CommunityReport)
    MATCH (c)<-[:IN_COMMUNITY]-(r:Restaurant)
    WITH c.community_id AS cid, cr.summary AS summary, cr.title AS title,
         collect(r.store_key) AS store_keys, collect(r.name) AS names, count(r) AS n
    WHERE n >= 2
    RETURN cid, summary, title, store_keys, names
    ORDER BY rand()
    LIMIT $n
    """, {"n": n_queries})

    if not rows:
        print("⚠️  No community reports found. Run upsert_community_reports() first.")
        return []

    test_cases = []
    gen_llm_raw = get_llm()
    for row in tqdm(rows, desc="Generating eval queries"):
        context = f"Community: {row['title']}\nRestaurants: {', '.join(row['names'][:5])}\nSummary: {row['summary']}"
        try:
            resp = gen_llm_raw.invoke([
                {"role": "system", "content": GEN_SYSTEM},
                {"role": "user", "content": f"Context:\n{context}\n\nSinh 1 câu hỏi tìm quán ăn phù hợp với cụm này."},
            ])
            raw = resp.content.strip()
            raw = re.sub(r"```json|```", "", raw).strip()
            parsed = json.loads(raw)
            test_cases.append({
                "query": parsed["query"],
                "relevant_store_keys": row["store_keys"],
                "community_id": row["cid"],
                "reasoning": parsed.get("reasoning", ""),
            })
        except Exception as e:
            print(f"  ⚠️  Failed to generate query for community {row['cid']}: {e}")

    print(f"✅ Generated {len(test_cases)} eval queries from graph")
    return test_cases


def evaluate_retrieval(
    test_cases: Optional[List[dict]] = None,
    k: int = 5,
    auto_generate: bool = True,
    n_auto_queries: int = 20,
) -> pd.DataFrame:
    """Evaluate retrieval pipeline.

    Args:
        test_cases: User-provided labeled queries [{query, relevant_store_keys}].
                    If None and auto_generate=True, generates from graph automatically.
        auto_generate: If True and test_cases is None, auto-generate from community reports.
        n_auto_queries: Number of queries to auto-generate.
    """
    if test_cases is None or len(test_cases) == 0:
        if auto_generate:
            print("No test_cases provided. Auto-generating from community reports...")
            test_cases = generate_eval_dataset_from_graph(n_queries=n_auto_queries)
        else:
            raise ValueError(
                "test_cases is empty and auto_generate=False. "
                "Provide labeled queries or set auto_generate=True."
            )
    if not test_cases:
        raise ValueError("Could not generate test cases. Check that community reports exist.")

    rows = []
    for tc in tqdm(test_cases, desc="Evaluating"):
        _, ranked = hybrid_retrieve(tc["query"], top_k=k)
        retrieved = [r["store_key"] for r in ranked]
        rows.append({
            "query": tc["query"],
            "relevant": tc["relevant_store_keys"],
            f"recall@{k}": recall_at_k(retrieved, tc["relevant_store_keys"], k),
            f"mrr@{k}": mrr_at_k(retrieved, tc["relevant_store_keys"], k),
            f"ndcg@{k}": ndcg_at_k(retrieved, tc["relevant_store_keys"], k),
            "retrieved": retrieved,
        })
    df = pd.DataFrame(rows)
    print("\n=== Evaluation Results ===")
    print(df[[f"recall@{k}", f"mrr@{k}", f"ndcg@{k}"]].mean().round(4).to_string())
    return df


# --- Ablation using evaluate_retrieval ---
def evaluate_ablation(test_cases: Optional[List[dict]] = None, k: int = 5) -> pd.DataFrame:
    """So sánh 4 variants: vector-only, graph-only, text-unit-only, hybrid."""
    if test_cases is None:
        test_cases = generate_eval_dataset_from_graph(n_queries=15)

    def _eval_fn(retrieve_fn):
        results = []
        for tc in test_cases:
            retrieved = retrieve_fn(tc["query"], top_k=k)
            results.append({
                "recall": recall_at_k(retrieved, tc["relevant_store_keys"], k),
                "mrr": mrr_at_k(retrieved, tc["relevant_store_keys"], k),
                "ndcg": ndcg_at_k(retrieved, tc["relevant_store_keys"], k),
            })
        return pd.DataFrame(results).mean()

    variants = {
        "vector_only": lambda q, top_k: [r["store_key"] for r in vector_search_restaurants(q, top_k=top_k)],
        "graph_only": lambda q, top_k: [r["store_key"] for r in graph_candidate_search(parse_intent(q), top_k=top_k)],
        "text_unit_only": lambda q, top_k: [r["store_key"] for r in vector_search_text_units(q, top_k=top_k)],
        "hybrid_no_ce": retrieve_hybrid_no_ce,
        "hybrid_with_ce": retrieve_hybrid_with_ce,
    }

    records = []
    for name, fn in variants.items():
        print(f"Evaluating: {name}...")
        scores = _eval_fn(fn)
        records.append({"variant": name, **scores.to_dict()})
    df = pd.DataFrame(records).set_index("variant")
    print("\n=== Ablation Results ===")
    print(df.round(4).to_string())
    return df

## 14. Ablation gợi ý

In [69]:
def retrieve_vector_only(query: str, top_k: int = 5) -> List[str]:
    return [r["store_key"] for r in vector_search_restaurants(query, top_k=top_k)]

def retrieve_graph_only(query: str, top_k: int = 5) -> List[str]:
    intent = parse_intent(query)
    return [r["store_key"] for r in graph_candidate_search(intent, top_k=top_k)]

def retrieve_text_unit_only(query: str, top_k: int = 5) -> List[str]:
    return [r["store_key"] for r in vector_search_text_units(query, top_k=top_k)]

def retrieve_hybrid(query: str, top_k: int = 5) -> List[str]:
    _, ranked = hybrid_retrieve(query, top_k=top_k)
    return [r["store_key"] for r in ranked]
def retrieve_text_unit_top5_unique(query: str, top_k: int = 5) -> list[str]:
    """
    Trả về top-K quán dựa trên text_unit embeddings, đảm bảo các store_key khác nhau.
    """
    # 1. Lấy nhiều text_unit candidate để đủ quán
    all_units = vector_search_text_units(query, top_k=50)  # tăng limit để cover đủ quán

    # 2. Deduplicate store_key, giữ top score
    seen = set()
    unique_stores = []
    for tu in all_units:
        sk = str(tu["store_key"])
        if sk not in seen:
            seen.add(sk)
            unique_stores.append(sk)
        if len(unique_stores) >= top_k:
            break

    return unique_stores


In [89]:
# ============================================================
# TEST GRAPH RAG - 50 USER QUERIES CÓ LAT/LNG
# HIỆN ĐỦ TOP 5 + QUALITY PROXY METRICS
#
# Dựa đúng vào hàm hybrid_retrieve hiện tại:
# hybrid_retrieve(query, top_k, use_cross_encoder, user_lat, user_lng)
#
# Output:
#   graphrag_50_geo_test_results.csv
#   graphrag_50_geo_test_results_long.csv
#   graphrag_50_geo_test_quality_summary.csv
#
# Ghi chú:
# - Cell này có thể gọi LLM nếu hybrid_retrieve() dùng parse_intent bằng LLM.
# - Chạy xong query nào lưu CSV query đó.
# - Mất điện / interrupt thì chạy lại sẽ skip query đã thành công.
# ============================================================

import json
import time
import traceback
import math
from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


# =========================
# Config
# =========================
TOP_K_TEST = 5
USE_CE_IN_TEST = True

# None = chạy đủ 50 query.
# Đổi thành 5 nếu muốn smoke test nhanh.
SMOKE_TEST_N = None

OUT_WIDE = "graphrag_50_geo_test_results.csv"
OUT_LONG = "graphrag_50_geo_test_results_long.csv"
OUT_QUALITY = "graphrag_50_geo_test_quality_summary.csv"

OUT_WIDE_PATH = Path(OUT_WIDE)
OUT_LONG_PATH = Path(OUT_LONG)
OUT_QUALITY_PATH = Path(OUT_QUALITY)


# =========================
# 50 test queries
# =========================
GRAPH_RAG_GEO_TEST_QUERIES = [
    {"test_id": "T001", "query": "Mình muốn ăn cơm gà ngon gần đây, rating ổn và có nhiều comment thật.", "user_lat": 21.005118, "user_lng": 105.845592},
    {"test_id": "T002", "query": "Tìm quán phở bò ăn sáng giá sinh viên dưới 50k càng tốt.", "user_lat": 21.00554, "user_lng": 105.84330},
    {"test_id": "T003", "query": "Có quán bún chả nào gần tôi không, ưu tiên quán được khen ngon trong comment.", "user_lat": 21.00452, "user_lng": 105.84320},
    {"test_id": "T004", "query": "Mình muốn ăn bánh cuốn nóng, quán nào sạch sẽ và giá vừa phải?", "user_lat": 21.00375, "user_lng": 105.84640},
    {"test_id": "T005", "query": "Gợi ý gà rán hoặc cơm gà cho bữa tối, giá không quá 100k, giao nhanh thì tốt.", "user_lat": 21.00491, "user_lng": 105.84820},

    {"test_id": "T006", "query": "Quán cơm văn phòng nào phù hợp ăn trưa, phần ăn đầy đặn và không quá đắt?", "user_lat": 21.00546, "user_lng": 105.84370},
    {"test_id": "T007", "query": "Tôi muốn tìm phở gà gần đây, ưu tiên quán có rating từ 4.5 trở lên.", "user_lat": 21.00697, "user_lng": 105.84590},
    {"test_id": "T008", "query": "Có quán cơm rang hoặc phở xào nào ăn tối ổn không?", "user_lat": 21.00816, "user_lng": 105.84610},
    {"test_id": "T009", "query": "Gợi ý quán bún hoặc phở giá rẻ, nhiều review thật, phù hợp sinh viên.", "user_lat": 21.00574, "user_lng": 105.84420},
    {"test_id": "T010", "query": "Mình cần quán ăn gần nhất, món gì cũng được miễn rating ổn.", "user_lat": 21.00539, "user_lng": 105.84340},

    {"test_id": "T011", "query": "Tìm quán cơm tấm hoặc cơm gà có comment khen ngon và giá không quá đắt.", "user_lat": 21.00120, "user_lng": 105.85200},
    {"test_id": "T012", "query": "Quán nào có món phở hoặc bún, phù hợp ăn tối một mình?", "user_lat": 20.99970, "user_lng": 105.84820},
    {"test_id": "T013", "query": "Muốn đặt cơm trưa, ưu tiên quán giao nhanh và có nhiều comment tích cực.", "user_lat": 21.00660, "user_lng": 105.84510},
    {"test_id": "T014", "query": "Gợi ý quán bún chả giá khoảng 50-100k, không quá xa.", "user_lat": 21.00620, "user_lng": 105.83150},
    {"test_id": "T015", "query": "Tôi muốn ăn phở bò, quán nào có nhiều review và rating tốt?", "user_lat": 21.01320, "user_lng": 105.83590},

    {"test_id": "T016", "query": "Có quán cơm nào phù hợp sinh viên, giá dưới 50k không?", "user_lat": 21.01020, "user_lng": 105.83980},
    {"test_id": "T017", "query": "Gợi ý quán gà rán đang mở cửa buổi tối, rating ổn.", "user_lat": 21.01080, "user_lng": 105.83170},
    {"test_id": "T018", "query": "Tìm quán bánh cuốn, quán nào sạch sẽ và comment ổn?", "user_lat": 21.00580, "user_lng": 105.84420},
    {"test_id": "T019", "query": "Mình muốn tìm quán cơm hoặc phở có giá hợp lý trong bán kính vài km.", "user_lat": 21.01650, "user_lng": 105.84360},
    {"test_id": "T020", "query": "Quán nào có món bún hoặc phở, nhiều đánh giá, phù hợp ăn trưa?", "user_lat": 21.01560, "user_lng": 105.85530},

    {"test_id": "T021", "query": "Tôi muốn tránh quán bị chê vệ sinh, tìm quán cơm gà có comment tốt.", "user_lat": 21.005118, "user_lng": 105.845592},
    {"test_id": "T022", "query": "Tìm quán giao nhanh, món cơm hoặc bún đều được, rating trên 4.", "user_lat": 21.00343, "user_lng": 105.84710},
    {"test_id": "T023", "query": "Có quán phở nào giá mềm, ăn sáng được không?", "user_lat": 21.00290, "user_lng": 105.83650},
    {"test_id": "T024", "query": "Gợi ý quán cơm rang phần ăn nhiều, phù hợp ăn trưa.", "user_lat": 21.00870, "user_lng": 105.83680},
    {"test_id": "T025", "query": "Mình muốn ăn bún chả, quán nào rating cao và có comment thật?", "user_lat": 21.01390, "user_lng": 105.84740},

    {"test_id": "T026", "query": "Tìm quán cơm giá không quá 70k, đang mở cửa giờ tối.", "user_lat": 21.00940, "user_lng": 105.85640},
    {"test_id": "T027", "query": "Có quán gà nào không quá xa, review ổn không?", "user_lat": 20.99890, "user_lng": 105.85510},
    {"test_id": "T028", "query": "Gợi ý quán phở hoặc cơm giá rẻ và giao nhanh.", "user_lat": 20.99260, "user_lng": 105.86220},
    {"test_id": "T029", "query": "Mình muốn tìm quán bún/phở/cơm có rating tốt quanh khu này.", "user_lat": 20.99490, "user_lng": 105.86720},
    {"test_id": "T030", "query": "Tìm quán cơm hoặc gà rán giá tầm trung, comment không tệ.", "user_lat": 20.98980, "user_lng": 105.85090},

    {"test_id": "T031", "query": "Quán nào có cơm/phở, nhiều món trong menu, hợp đi nhóm nhỏ?", "user_lat": 21.00190, "user_lng": 105.83240},
    {"test_id": "T032", "query": "Tôi cần ăn nhanh, quán cơm hoặc phở nào gần và ổn nhất?", "user_lat": 21.00060, "user_lng": 105.83920},
    {"test_id": "T033", "query": "Gợi ý quán bún chả hoặc bánh cuốn, ưu tiên quán không bị chê đồ nguội.", "user_lat": 21.00375, "user_lng": 105.84640},
    {"test_id": "T034", "query": "Có quán cơm gà nào ăn tối, giá khoảng 50-100k không?", "user_lat": 21.00641, "user_lng": 105.84470},
    {"test_id": "T035", "query": "Tìm quán phở xào hoặc cơm rang, quán nào comment khen nhiều?", "user_lat": 21.00816, "user_lng": 105.84610},

    {"test_id": "T036", "query": "Tôi muốn quán ăn trưa gần đây, ưu tiên rẻ, rating ổn và không quá xa.", "user_lat": 21.005118, "user_lng": 105.845592},
    {"test_id": "T037", "query": "Mình muốn đặt món lúc 22h, còn quán cơm/phở/gà nào mở cửa không?", "user_lat": 21.005118, "user_lng": 105.845592},
    {"test_id": "T038", "query": "Tìm quán có comment nói đóng gói tốt, giao nhanh, món cơm hoặc gà.", "user_lat": 21.00697, "user_lng": 105.84590},
    {"test_id": "T039", "query": "Gợi ý quán có rating cao nhất trong nhóm cơm, phở, bún chả.", "user_lat": 21.005118, "user_lng": 105.845592},
    {"test_id": "T040", "query": "Quán nào có nhiều review nhất, món ăn chính là cơm hoặc bún?", "user_lat": 21.00491, "user_lng": 105.84820},

    {"test_id": "T041", "query": "Tìm quán phở mở cửa sáng sớm, phù hợp ăn sáng.", "user_lat": 21.01160, "user_lng": 105.85020},
    {"test_id": "T042", "query": "Có quán cơm giá sinh viên và có comment về suất ăn nhiều không?", "user_lat": 20.99680, "user_lng": 105.84320},
    {"test_id": "T043", "query": "Mình muốn bún hoặc phở rating tốt, không quá xa.", "user_lat": 21.01940, "user_lng": 105.84980},
    {"test_id": "T044", "query": "Tìm quán bánh cuốn hoặc bún chả đang mở cửa buổi trưa.", "user_lat": 21.00120, "user_lng": 105.85200},
    {"test_id": "T045", "query": "Gợi ý quán cơm giá không quá đắt, có review thật để tin tưởng.", "user_lat": 21.01180, "user_lng": 105.82680},

    {"test_id": "T046", "query": "Mình cần quán gần nhất có món phở bò hoặc phở gà.", "user_lat": 21.00870, "user_lng": 105.83680},
    {"test_id": "T047", "query": "Tìm quán cơm hoặc gà rán có ảnh, rating ổn.", "user_lat": 21.00413, "user_lng": 105.84700},
    {"test_id": "T048", "query": "Quán nào phù hợp đặt cho nhóm 3 người, menu đa dạng, có cơm và bún/phở?", "user_lat": 21.005118, "user_lng": 105.845592},
    {"test_id": "T049", "query": "Tìm quán có comment tốt về phục vụ, món cơm hoặc bánh cuốn.", "user_lat": 21.00637, "user_lng": 105.84470},
    {"test_id": "T050", "query": "Mình muốn quán đáng tin nhất quanh đây: rating cao, nhiều review, có comment thật, món cơm/phở/bún đều được.", "user_lat": 21.005118, "user_lng": 105.845592},
]


# =========================
# Helpers
# =========================
def _safe_json(obj: Any) -> str:
    return json.dumps(obj, ensure_ascii=False, default=str)


def _safe_float(x: Any, default: Optional[float] = None) -> Optional[float]:
    if x is None:
        return default
    try:
        if pd.isna(x):
            return default
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        return default


def _mean_float(values: List[Any]) -> Optional[float]:
    vals = [_safe_float(v, None) for v in values]
    vals = [v for v in vals if v is not None and not math.isnan(v)]
    if not vals:
        return None
    return round(sum(vals) / len(vals), 6)


def _get_candidate_name(c: Dict[str, Any]) -> str:
    return (
        c.get("name")
        or c.get("restaurant_name")
        or c.get("store_name")
        or c.get("title")
        or ""
    )


def _get_candidate_id(c: Dict[str, Any]) -> str:
    return str(
        c.get("store_key")
        or c.get("restaurant_id")
        or c.get("id")
        or ""
    )


def _evidence_count(c: Dict[str, Any]) -> int:
    ev = c.get("evidence") or []
    if isinstance(ev, list):
        return len([x for x in ev if str(x).strip()])
    if isinstance(ev, str):
        return 1 if ev.strip() else 0
    return 0


def _normalize_candidate(c: Dict[str, Any], rank: int) -> Dict[str, Any]:
    evidence = c.get("evidence", [])
    if isinstance(evidence, str):
        evidence = [evidence]
    elif not isinstance(evidence, list):
        evidence = []

    return {
        "rank": rank,
        "store_key": _get_candidate_id(c),
        "name": _get_candidate_name(c),
        "address": c.get("address", ""),
        "district": c.get("district", ""),
        "city": c.get("city", ""),
        "distance_km": c.get("distance_km", c.get("distance", "")),
        "final_score": c.get("final_score", c.get("score", c.get("rrf_score", ""))),
        "ce_score": c.get("ce_score", ""),
        "final_score_before_ce": c.get("final_score_before_ce", ""),
        "rating": c.get("rating", c.get("gmaps_rating", "")),
        "price_band": c.get("price_band", ""),
        "dish_families": c.get("dish_families", []),
        "categories": c.get("categories", []),
        "source_flags": c.get("source_flags", []),
        "community_id": c.get("community_id", ""),
        "community_report": c.get("community_report", ""),
        "evidence": evidence,
        "evidence_count": len([x for x in evidence if str(x).strip()]),
    }


def _quality_metrics(top_k_items: List[Dict[str, Any]], latency_ms: float, error: str) -> Dict[str, Any]:
    result_count = len(top_k_items)
    store_keys = [str(x.get("store_key", "")).strip() for x in top_k_items if str(x.get("store_key", "")).strip()]
    unique_store_count = len(set(store_keys))
    duplicate_count = max(0, len(store_keys) - unique_store_count)

    scores = [_safe_float(x.get("final_score"), None) for x in top_k_items]
    scores = [x for x in scores if x is not None]

    ce_scores = [_safe_float(x.get("ce_score"), None) for x in top_k_items]
    ce_scores = [x for x in ce_scores if x is not None]

    distances = [_safe_float(x.get("distance_km"), None) for x in top_k_items]
    distances = [x for x in distances if x is not None]

    evidence_counts = [int(x.get("evidence_count") or 0) for x in top_k_items]
    evidence_hit_count = sum(1 for x in evidence_counts if x > 0)

    top1_score = scores[0] if len(scores) >= 1 else None
    top5_score = scores[4] if len(scores) >= 5 else None

    score_gap = None
    if top1_score is not None and top5_score is not None:
        score_gap = round(top1_score - top5_score, 6)

    return {
        "result_count": result_count,
        "has_top5": int(result_count >= 5),
        "unique_store_count": unique_store_count,
        "duplicate_count": duplicate_count,
        "duplicate_rate": round(duplicate_count / max(1, result_count), 4),
        "evidence_hit_count": evidence_hit_count,
        "evidence_coverage": round(evidence_hit_count / max(1, result_count), 4),
        "avg_evidence_count": round(sum(evidence_counts) / max(1, result_count), 4),
        "top1_score": top1_score,
        "avg_score_top5": round(sum(scores) / len(scores), 6) if scores else None,
        "score_gap_top1_top5": score_gap,
        "top1_distance_km": distances[0] if len(distances) >= 1 else None,
        "avg_distance_km_top5": round(sum(distances) / len(distances), 4) if distances else None,
        "max_distance_km_top5": round(max(distances), 4) if distances else None,
        "avg_ce_score_top5": round(sum(ce_scores) / len(ce_scores), 6) if ce_scores else None,
        "latency_ms": latency_ms,
        "has_error": int(bool(str(error).strip())),
        "error": error,
    }


def run_one_geo_query(row: Dict[str, Any], top_k: int = TOP_K_TEST) -> Dict[str, Any]:
    t0 = time.time()
    error = ""
    error_trace = ""
    intent = {}
    ranked = []

    try:
        intent, ranked = hybrid_retrieve(
            row["query"],
            top_k=top_k,
            use_cross_encoder=USE_CE_IN_TEST,
            user_lat=row["user_lat"],
            user_lng=row["user_lng"],
        )
        ranked = ranked or []

    except Exception as e:
        error = repr(e)
        error_trace = traceback.format_exc(limit=3)

    latency_ms = round((time.time() - t0) * 1000, 2)

    top_k_items = [
        _normalize_candidate(c, i + 1)
        for i, c in enumerate(ranked[:top_k])
        if isinstance(c, dict)
    ]

    quality = _quality_metrics(top_k_items, latency_ms, error)

    return {
        "test_id": row["test_id"],
        "query": row["query"],
        "user_lat": row["user_lat"],
        "user_lng": row["user_lng"],
        "intent": intent,
        "top_k": top_k_items,
        "quality": quality,
        "latency_ms": latency_ms,
        "error": error,
        "error_trace": error_trace,
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }


def _result_to_wide_row(res: Dict[str, Any]) -> Dict[str, Any]:
    q = res["quality"]

    wide_row = {
        "test_id": res["test_id"],
        "query": res["query"],
        "user_lat": res["user_lat"],
        "user_lng": res["user_lng"],
        "top_k_json": _safe_json(res["top_k"]),
        "intent_json": _safe_json(res["intent"]),

        # quality proxy metrics
        "result_count": q["result_count"],
        "has_top5": q["has_top5"],
        "unique_store_count": q["unique_store_count"],
        "duplicate_count": q["duplicate_count"],
        "duplicate_rate": q["duplicate_rate"],
        "evidence_hit_count": q["evidence_hit_count"],
        "evidence_coverage": q["evidence_coverage"],
        "avg_evidence_count": q["avg_evidence_count"],
        "top1_score": q["top1_score"],
        "avg_score_top5": q["avg_score_top5"],
        "score_gap_top1_top5": q["score_gap_top1_top5"],
        "top1_distance_km": q["top1_distance_km"],
        "avg_distance_km_top5": q["avg_distance_km_top5"],
        "max_distance_km_top5": q["max_distance_km_top5"],
        "avg_ce_score_top5": q["avg_ce_score_top5"],

        "latency_ms": res["latency_ms"],
        "error": res["error"],
        "error_trace": res["error_trace"],
        "saved_at": res["saved_at"],
    }

    for rank in range(1, TOP_K_TEST + 1):
        item = res["top_k"][rank - 1] if rank <= len(res["top_k"]) else {}

        wide_row[f"top{rank}_store_key"] = item.get("store_key", "")
        wide_row[f"top{rank}_name"] = item.get("name", "")
        wide_row[f"top{rank}_distance_km"] = item.get("distance_km", "")
        wide_row[f"top{rank}_score"] = item.get("final_score", "")
        wide_row[f"top{rank}_ce_score"] = item.get("ce_score", "")
        wide_row[f"top{rank}_rating"] = item.get("rating", "")
        wide_row[f"top{rank}_price_band"] = item.get("price_band", "")
        wide_row[f"top{rank}_evidence_count"] = item.get("evidence_count", "")
        wide_row[f"top{rank}_address"] = item.get("address", "")

    return wide_row


def _result_to_long_rows(res: Dict[str, Any]) -> List[Dict[str, Any]]:
    rows = []

    for item in res["top_k"]:
        rows.append({
            "test_id": res["test_id"],
            "query": res["query"],
            "user_lat": res["user_lat"],
            "user_lng": res["user_lng"],
            **item,
            "intent_json": _safe_json(res["intent"]),
            "latency_ms": res["latency_ms"],
            "error": res["error"],
            "error_trace": res["error_trace"],
            "saved_at": res["saved_at"],
        })

    return rows


from pandas.errors import EmptyDataError
import shutil
import time


def _atomic_write_csv(df: pd.DataFrame, path: Path) -> None:
    """
    Write CSV safely.

    Fix:
    - Không ghi file CSV rỗng không có cột.
    - Retry nếu Windows lock file.
    - Nếu vẫn lock thì lưu backup timestamp.
    """
    path = Path(path)

    # Nếu DataFrame rỗng và không có cột, không ghi file.
    # Nếu ghi sẽ tạo CSV không có header -> lần sau read_csv bị EmptyDataError.
    if df is None or (df.empty and len(df.columns) == 0):
        print(f"⚠️ Skip writing empty no-column CSV: {path}")
        return

    tmp_path = Path(str(path) + ".tmp")
    df.to_csv(tmp_path, index=False, encoding="utf-8-sig")

    last_err = None

    for attempt in range(5):
        try:
            tmp_path.replace(path)
            return
        except PermissionError as e:
            last_err = e
            print(f"⚠️ CSV locked retry {attempt + 1}/5: {path}")
            time.sleep(1.0)

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    backup_path = path.with_name(f"{path.stem}_{timestamp}{path.suffix}")

    shutil.copyfile(tmp_path, backup_path)

    try:
        tmp_path.unlink(missing_ok=True)
    except Exception:
        pass

    print(f"⚠️ Could not overwrite locked CSV: {path}")
    print(f"✅ Saved backup instead: {backup_path}")

    if last_err:
        print(f"Last error: {last_err}")


def _load_existing_csv(path: Path) -> pd.DataFrame:
    """
    Load CSV safely.

    Fix:
    - File không tồn tại -> empty DataFrame.
    - File 0 byte hoặc không có header -> xóa và return empty DataFrame.
    """
    path = Path(path)

    if not path.exists():
        return pd.DataFrame()

    if path.stat().st_size == 0:
        print(f"⚠️ Empty CSV file found, deleting: {path}")
        path.unlink()
        return pd.DataFrame()

    try:
        df = pd.read_csv(path)
        print(f"✅ Loaded existing CSV: {path} - {len(df)} rows")
        return df

    except EmptyDataError:
        print(f"⚠️ CSV has no columns, deleting: {path}")
        path.unlink(missing_ok=True)
        return pd.DataFrame()

    except Exception as e:
        print(f"⚠️ Could not load {path}: {type(e).__name__}: {e}")
        return pd.DataFrame()


def _is_successful_existing_row(row: pd.Series) -> bool:
    error = str(row.get("error", "") or "").strip()
    if error:
        return False

    try:
        top_k = json.loads(str(row.get("top_k_json", "[]")))
    except Exception:
        return False

    return isinstance(top_k, list) and len(top_k) > 0


# =========================
# Run with resume/catch-up
# =========================
queries_to_run = GRAPH_RAG_GEO_TEST_QUERIES

if SMOKE_TEST_N is not None:
    queries_to_run = GRAPH_RAG_GEO_TEST_QUERIES[:int(SMOKE_TEST_N)]

wide_df = _load_existing_csv(OUT_WIDE_PATH)
long_df = _load_existing_csv(OUT_LONG_PATH)

done_success_ids = set()

if not wide_df.empty and "test_id" in wide_df.columns:
    for _, row in wide_df.iterrows():
        if _is_successful_existing_row(row):
            done_success_ids.add(str(row["test_id"]))

print(f"Already successful queries: {len(done_success_ids)}")
print(f"Total queries in this run: {len(queries_to_run)}")

try:
    for i, q in enumerate(queries_to_run, start=1):
        test_id = str(q["test_id"])

        if test_id in done_success_ids:
            print(f"[{i:02d}/{len(queries_to_run)}] {test_id} - SKIP existing successful result")
            continue

        print(f"[{i:02d}/{len(queries_to_run)}] {test_id} - {q['query'][:90]}")

        res = run_one_geo_query(q, TOP_K_TEST)

        new_wide_row = _result_to_wide_row(res)
        new_long_rows = _result_to_long_rows(res)

        # Xóa kết quả cũ của test_id này nếu có, rồi ghi bản mới
        if not wide_df.empty and "test_id" in wide_df.columns:
            wide_df = wide_df[wide_df["test_id"].astype(str) != test_id]

        if not long_df.empty and "test_id" in long_df.columns:
            long_df = long_df[long_df["test_id"].astype(str) != test_id]

        wide_df = pd.concat([wide_df, pd.DataFrame([new_wide_row])], ignore_index=True)

        if new_long_rows:
            long_df = pd.concat([long_df, pd.DataFrame(new_long_rows)], ignore_index=True)

        # Lưu ngay sau mỗi query
        _atomic_write_csv(wide_df, OUT_WIDE_PATH)
        _atomic_write_csv(long_df, OUT_LONG_PATH)

        if not res["error"] and len(res["top_k"]) > 0:
            done_success_ids.add(test_id)

except KeyboardInterrupt:
    print("⚠️ Interrupted. CSV has already been saved after each completed query.")
    raise

except Exception as e:
    print(f"⚠️ Test run failed: {type(e).__name__}: {e}")
    print("CSV has already been saved after each completed query.")
    raise


# =========================
# Reload final CSV
# =========================
wide_loaded = _load_existing_csv(OUT_WIDE_PATH)
if not wide_loaded.empty:
    wide_df = wide_loaded

long_loaded = _load_existing_csv(OUT_LONG_PATH)
if not long_loaded.empty:
    long_df = long_loaded


# =========================
# Build quality summary
# =========================
quality_cols = [
    "result_count",
    "has_top5",
    "unique_store_count",
    "duplicate_count",
    "duplicate_rate",
    "evidence_hit_count",
    "evidence_coverage",
    "avg_evidence_count",
    "top1_score",
    "avg_score_top5",
    "score_gap_top1_top5",
    "top1_distance_km",
    "avg_distance_km_top5",
    "max_distance_km_top5",
    "avg_ce_score_top5",
    "latency_ms",
]

available_quality_cols = [c for c in quality_cols if c in wide_df.columns]

summary_records = []

summary = {
    "num_queries": len(wide_df),
    "num_errors": int(wide_df["error"].fillna("").astype(str).str.strip().ne("").sum()) if "error" in wide_df.columns else 0,
}

for col in available_quality_cols:
    numeric = pd.to_numeric(wide_df[col], errors="coerce")
    summary[f"mean_{col}"] = round(float(numeric.mean()), 6) if numeric.notna().any() else None
    summary[f"min_{col}"] = round(float(numeric.min()), 6) if numeric.notna().any() else None
    summary[f"max_{col}"] = round(float(numeric.max()), 6) if numeric.notna().any() else None

summary_records.append(summary)
quality_summary_df = pd.DataFrame(summary_records)

_atomic_write_csv(quality_summary_df, OUT_QUALITY_PATH)


# =========================
# Print final status
# =========================
error_count = int(wide_df["error"].fillna("").astype(str).str.strip().ne("").sum()) if "error" in wide_df.columns else 0

print("\n✅ DONE / RESUMABLE 50-QUERY TEST")
print(f"Saved wide CSV: {OUT_WIDE} - {len(wide_df)} rows")
print(f"Saved long CSV: {OUT_LONG} - {len(long_df)} rows")
print(f"Saved quality summary CSV: {OUT_QUALITY} - {len(quality_summary_df)} rows")
print(f"Error queries: {error_count}/{len(wide_df)}")

if error_count > 0:
    print("\n===== ERROR QUERIES =====")
    display(wide_df.loc[
        wide_df["error"].fillna("").astype(str).str.strip().ne(""),
        ["test_id", "query", "error", "error_trace"],
    ])

if error_count == len(wide_df) and len(wide_df) > 0:
    raise RuntimeError("Tất cả query đều lỗi. Dừng lại để kiểm tra hybrid_retrieve / Neo4j / Qdrant / LLM.")


# =========================
# Display Top 5 + quality metrics
# =========================
display_cols = [
    "test_id",
    "query",
    "user_lat",
    "user_lng",

    "result_count",
    "has_top5",
    "unique_store_count",
    "duplicate_count",
    "evidence_coverage",
    "avg_evidence_count",
    "top1_score",
    "avg_score_top5",
    "score_gap_top1_top5",
    "top1_distance_km",
    "avg_distance_km_top5",
    "avg_ce_score_top5",
    "latency_ms",
]

for rank in range(1, TOP_K_TEST + 1):
    display_cols.extend([
        f"top{rank}_name",
        f"top{rank}_distance_km",
        f"top{rank}_score",
        f"top{rank}_ce_score",
        f"top{rank}_rating",
        f"top{rank}_evidence_count",
        f"top{rank}_address",
    ])

display_cols.extend(["error", "saved_at"])
display_cols = [c for c in display_cols if c in wide_df.columns]

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

print("\n===== TOP 5 + RETRIEVAL QUALITY PROXY METRICS =====")
display(wide_df[display_cols].head(len(wide_df)))

print("\n===== QUALITY SUMMARY =====")
display(quality_summary_df)

Already successful queries: 0
Total queries in this run: 50
[01/50] T001 - Mình muốn ăn cơm gà ngon gần đây, rating ổn và có nhiều comment thật.
[02/50] T002 - Tìm quán phở bò ăn sáng giá sinh viên dưới 50k càng tốt.
[03/50] T003 - Có quán bún chả nào gần tôi không, ưu tiên quán được khen ngon trong comment.
[04/50] T004 - Mình muốn ăn bánh cuốn nóng, quán nào sạch sẽ và giá vừa phải?
[05/50] T005 - Gợi ý gà rán hoặc cơm gà cho bữa tối, giá không quá 100k, giao nhanh thì tốt.
[06/50] T006 - Quán cơm văn phòng nào phù hợp ăn trưa, phần ăn đầy đặn và không quá đắt?
[07/50] T007 - Tôi muốn tìm phở gà gần đây, ưu tiên quán có rating từ 4.5 trở lên.
[08/50] T008 - Có quán cơm rang hoặc phở xào nào ăn tối ổn không?
[09/50] T009 - Gợi ý quán bún hoặc phở giá rẻ, nhiều review thật, phù hợp sinh viên.
[10/50] T010 - Mình cần quán ăn gần nhất, món gì cũng được miễn rating ổn.
[11/50] T011 - Tìm quán cơm tấm hoặc cơm gà có comment khen ngon và giá không quá đắt.
[12/50] T012 - Quán nào có món phở

,test_id,query,user_lat,user_lng,result_count,has_top5,unique_store_count,duplicate_count,evidence_coverage,avg_evidence_count,top1_score,avg_score_top5,score_gap_top1_top5,top1_distance_km,avg_distance_km_top5,avg_ce_score_top5,latency_ms,top1_name,top1_distance_km,top1_score,top1_ce_score,top1_rating,top1_evidence_count,top1_address,top2_name,top2_distance_km,top2_score,top2_ce_score,top2_rating,top2_evidence_count,top2_address,top3_name,top3_distance_km,top3_score,top3_ce_score,top3_rating,top3_evidence_count,top3_address,top4_name,top4_distance_km,top4_score,top4_ce_score,top4_rating,top4_evidence_count,top4_address,top5_name,top5_distance_km,top5_score,top5_ce_score,top5_rating,top5_evidence_count,top5_address,error,saved_at
0,T001,"Mình muốn ăn cơm gà ngon gần đây, rating ổn và có nhiều comment thật.",21.005118,105.845592,5,1,5,0,0.0,0.0,0.362150,0.186379,0.246910,2.441,2.7856,0.46054,9712.39,Tiệm Cơm Gà - Trung Liệt,2.441,0.362150,1.0000,4.2,0,"98 Trung Liệt, Trung Liệt Đống Đa, Hà Nội",Cơm Bình Dân - Lạc Nghiệp,1.421,0.171515,0.5035,4.0,0,"241 Lạc Nghiệp, Phường Bạch Mai, Hà Nội","FoodHub - Mỳ Quảng, Cơm Gà Roti - Hoàng Ngọc Phách",3.366,0.165977,0.3220,4.7,0,"Số 110 Tập Thể C3 Hoàng Ngọc Phách, Phường Đống Đa, Hà Nội",Cơm Gà Xối Mắm Còn - Dương Văn Bé,3.337,0.117013,0.1713,4.7,0,"202 Dương Văn Bé, P. Vĩnh Tuy, Q. Hai Bà Trưng, Hà Nội",Guu Chicken - Cơm Gà & Gà Rán - Khương Hạ,3.363,0.115240,0.3059,4.7,0,"Số 63 Ngách 78 Ngõ 29 Khương Hạ, Phường Khương Đình, Thanh Xuân, Hà Nội",NaN,2026-05-28 10:24:00
1,T002,Tìm quán phở bò ăn sáng giá sinh viên dưới 50k càng tốt.,21.005540,105.843300,5,1,5,0,0.0,0.0,0.349409,0.232289,0.185599,3.683,2.1550,0.65514,15577.64,Phở bò - Cơm rang Duy Nhất - Huỳnh Thúc Kháng,3.683,0.349409,1.0000,4.7,0,"12a Ng. 16 - P. Huỳnh Thúc Kháng, Láng Hạ, Ba Đình, Hà Nội, Việt Nam",KCC - Phở - Cơm Rang - Nguyên Hồng,3.791,0.298632,0.8523,4.5,0,"105 D8 Nguyên Hồng, Thành Công, Ba Đình, Hà Nội",Quán Hà Nội Xưa - Bún Chả - Trường Chinh,0.760,0.185033,0.5429,3.9,0,"35 Ngõ 102, Trường Chinh, Phương Mai, Đống Đa, Hà Nội",Phở Bò Cơm Rang Duy Nhất - Giải Phóng,0.489,0.164559,0.4745,4.6,0,"16 Ngõ 205 Giải Phóng, phường Bạch Mai , Hà Nội","Phở Bò Nội - Bún Chả, Cháo & Cơm - Trần Hưng Đạo",2.052,0.163810,0.4060,4.8,0,"94A Trần Hưng Đạo, Phường Cửa Nam, Hà Nội",NaN,2026-05-28 10:24:15
2,T003,"Có quán bún chả nào gần tôi không, ưu tiên quán được khen ngon trong comment.",21.004520,105.843200,5,1,5,0,0.0,0.0,0.327571,0.200091,0.189630,1.317,2.1116,0.45908,12386.27,Hương Phệ - Bún Chả Ngon - Hồng Mai,1.317,0.327571,1.0000,4.9,0,"164 Hồng Mai, Quỳnh Lôi, Hai Bà Trưng, Hà Nội",Bún Chả Cô An - Hanoi Grilled Pork Noodles - Khâm Thiên,1.309,0.233223,0.4330,4.6,0,"Nghách 1/33 Ngõ 1 Khâm Thiên, Phường Đống Đa, Hà Nội",Bún Chả Xưa - Lĩnh Nam,2.715,0.154283,0.1833,4.2,0,"9 Ngõ 13 Lĩnh Nam, Phường Mai Động, Quận Hoàng Mai, Hà Nội","NaBi Bún Chả Quạt, Cơm Tấm & Cơm Văn Phòng - Tây Sơn",1.818,0.147437,0.3850,4.3,0,"55C Ngách 25 Ngõ 167 Tây Sơn, Phường Quang Trung, Quận Đống Đa, Hà Nội","Bún Chả Linh Anh - Bún Ngan Mọc, Bánh Giò Nóng & Trứng Vịt Lộn - Khương Đình",3.399,0.137941,0.2941,4.7,0,"Số 7 Ngõ 236 Khương Đình, Phường Hạ Đình, Quận Thanh Xuân, Hà Nội",NaN,2026-05-28 10:24:28
3,T004,"Mình muốn ăn bánh cuốn nóng, quán nào sạch sẽ và giá vừa phải?",21.003750,105.846400,5,1,5,0,0.0,0.0,0.318670,0.274812,0.122394,3.297,3.2764,0.84188,9274.82,Tùng Linh Quán - Bún Chả & Bánh Cuốn Nóng - Bùi Xương Trạch,3.297,0.318670,1.0000,5.0,0,"289 Bùi Xương Trạch, Phường Khương Đình, Thanh Xuân, Hà Nội","Bánh Cuốn Nóng, Bún Chả & Gà Tần - 156 Kim Giang",3.758,0.297881,0.9414,4.6,0,"156 đường Kim Giang, phường Định Công, Hà Nội",Bánh Cuốn Nóng & Bún Chả - Ngọc Hà,3.852,0.284347,0.8460,4.9,0,"124 Ngọc Hà, Phường Ngọc Hà, Quận Ba Đình, Hà Nội",Bánh Cuốn Nóng - Nguyễn Bỉnh Khiêm,1.419,0.276888,0.8188,4.5,0,"19 Nguyễn Bỉnh Khiêm, P. Nguyễn Du,Quận Hai Bà Trưng,Hà Nội","Bánh Cuốn Nóng, Bún Đậu Mẹt & Bún Chả - Vĩnh Hưng",4.056,0


===== QUALITY SUMMARY =====


,num_queries,num_errors,mean_result_count,min_result_count,max_result_count,mean_has_top5,min_has_top5,max_has_top5,mean_unique_store_count,min_unique_store_count,max_unique_store_count,mean_duplicate_count,min_duplicate_count,max_duplicate_count,mean_duplicate_rate,min_duplicate_rate,max_duplicate_rate,mean_evidence_hit_count,min_evidence_hit_count,max_evidence_hit_count,mean_evidence_coverage,min_evidence_coverage,max_evidence_coverage,mean_avg_evidence_count,min_avg_evidence_count,max_avg_evidence_count,mean_top1_score,min_top1_score,max_top1_score,mean_avg_score_top5,min_avg_score_top5,max_avg_score_top5,mean_score_gap_top1_top5,min_score_gap_top1_top5,max_score_gap_top1_top5,mean_top1_distance_km,min_top1_distance_km,max_top1_distance_km,mean_avg_distance_km_top5,min_avg_distance_km_top5,max_avg_distance_km_top5,mean_max_distance_km_top5,min_max_distance_km_top5,max_max_distance_km_top5,mean_avg_ce_score_top5,min_avg_ce_score_top5,max_avg_ce_score_top5,mean_latency_ms,min_latency_ms,max_latency_ms
0,50,0,5.0,5.0,5.0,1.0,1.0,1.0,5.0,5.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.329116,0.060633,0.411935,0.249712,0.158819,0.327791,0.139427,-0.156349,0.310639,1.58284,0.249,3.683,1.799264,0.4724,3.2764,2.91528,0.739,4.056,0.702603,0.36162,0.95232,11394.3182,6902.27,46163.42


In [39]:
restaurant_summary_df = summary.copy()

In [30]:
feedback_proc = feedback.copy()

In [88]:
def parse_list_like(x: Any) -> List[str]:
    if x is None:
        return []

    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]

    if isinstance(x, tuple) or isinstance(x, set):
        return [str(v).strip() for v in list(x) if str(v).strip()]

    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []

        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = json.loads(s.replace("'", '"'))
                if isinstance(parsed, list):
                    return [str(v).strip() for v in parsed if str(v).strip()]
            except Exception:
                pass

        if "|" in s:
            return [v.strip() for v in s.split("|") if v.strip()]

        if "," in s:
            return [v.strip() for v in s.split(",") if v.strip()]

        return [s]

    return [str(x).strip()] if str(x).strip() else []

In [100]:
# ============================================================
# FIX GRAPH_ONLY TOP-5 WITH USER LAT/LNG
# Chỉ dùng Neo4j graph, không fallback vector/text.
# ============================================================

import copy
from typing import Any, Dict, List, Optional


def _canon_store_key(x: Any) -> str:
    if x is None:
        return ""
    s = str(x).strip()
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]
    return s


def _dedupe_records_by_store_key(rows: List[dict], top_k: Optional[int] = None) -> List[dict]:
    seen = set()
    out = []

    for r in rows or []:
        sk = _canon_store_key(r.get("store_key"))
        if not sk or sk in seen:
            continue
        seen.add(sk)
        out.append(r)

        if top_k is not None and len(out) >= top_k:
            break

    return out


def _clean_intent(intent: dict, drop_keys: List[str] = None, updates: Dict[str, Any] = None) -> dict:
    x = copy.deepcopy(intent or {})
    drop_keys = drop_keys or []
    updates = updates or {}

    for k in drop_keys:
        x.pop(k, None)

    for k, v in updates.items():
        if v is None:
            x.pop(k, None)
        else:
            x[k] = v

    return x


def build_relaxed_graph_intents(intent: dict) -> List[tuple]:
    """
    Tạo các version intent nới dần.
    Vẫn là graph-only vì vẫn gọi graph_candidate_search trên Neo4j.
    """
    base = copy.deepcopy(intent or {})

    plans = []

    # 1. Strict như cũ
    plans.append(("strict", base))

    # 2. Bỏ entity_terms: thường làm query bị quá hẹp vì comment/ngon/giao nhanh
    plans.append((
        "drop_entity_terms",
        _clean_intent(base, drop_keys=["entity_terms"])
    ))

    # 3. Bỏ required_attributes: các attribute food_quality/cleanliness/speed đôi khi không có đủ edge
    plans.append((
        "drop_entity_terms_required_attributes",
        _clean_intent(base, drop_keys=["entity_terms", "required_attributes"])
    ))

    # 4. Bỏ min_rating: vì query nói rating ổn nhưng graph có thể thiếu rating hoặc rating field khác
    plans.append((
        "drop_entity_terms_attrs_min_rating",
        _clean_intent(base, drop_keys=["entity_terms", "required_attributes", "min_rating"])
    ))

    # 5. Mở rộng max distance rất lớn nhưng vẫn truyền user_lat/lng để tính distance
    # Lý do: graph_candidate_search mặc định dùng MAX_DISTANCE_KM nếu có user_lat/user_lng
    plans.append((
        "large_distance",
        _clean_intent(
            base,
            drop_keys=["entity_terms", "required_attributes", "min_rating"],
            updates={"max_distance_km": 999}
        )
    ))

    # 6. Giữ dish_name, bỏ category nếu category parse bị lệch
    plans.append((
        "dish_only_large_distance",
        {
            "dish_name": base.get("dish_name"),
            "geo_intent": base.get("geo_intent", "normal"),
            "max_distance_km": 999,
        }
    ))

    # 7. Giữ categories, bỏ dish nếu dish_family chưa upsert đủ
    plans.append((
        "category_only_large_distance",
        {
            "categories": base.get("categories") or [],
            "geo_intent": base.get("geo_intent", "normal"),
            "max_distance_km": 999,
        }
    ))

    # Loại plan rỗng vô nghĩa
    cleaned = []
    for name, it in plans:
        it = {k: v for k, v in (it or {}).items() if v not in [None, "", [], {}]}
        if it:
            cleaned.append((name, it))

    return cleaned


def _extract_graph_broad_terms(query: str, intent: dict) -> List[str]:
    terms = []

    # từ intent
    for key in ["dish_name", "district", "price_band"]:
        v = (intent or {}).get(key)
        if isinstance(v, str) and v.strip():
            terms.append(v.strip())

    for key in ["categories", "cuisines", "entity_terms"]:
        vals = (intent or {}).get(key) or []
        if isinstance(vals, str):
            vals = [vals]
        for v in vals:
            if str(v).strip():
                terms.append(str(v).strip())

    # từ query tiếng Việt phổ biến
    q = str(query).lower()
    keyword_phrases = [
        "cơm gà", "cơm rang", "cơm tấm", "cơm văn phòng", "cơm",
        "phở bò", "phở gà", "phở xào", "phở",
        "bún chả", "bún", "bánh cuốn",
        "gà rán", "gà", "mì", "mỳ",
    ]

    for kw in keyword_phrases:
        if kw in q:
            terms.append(kw)

    # dedupe giữ order
    seen = set()
    out = []
    for t in terms:
        t = str(t).strip().lower()
        if t and t not in seen:
            seen.add(t)
            out.append(t)

    return out


def graph_broad_keyword_search(
    query: str,
    intent: dict,
    top_k: int = 10,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[dict]:
    """
    Broad graph search:
    - Vẫn query Neo4j graph.
    - Match keyword trên Restaurant.name, categories, dish_families,
      extracted_entities, top_menu_items.
    - Không dùng vector/text.
    - Vẫn tính distance_km bằng user_lat/user_lng.
    """
    terms = _extract_graph_broad_terms(query, intent)

    has_user_location = user_lat is not None and user_lng is not None
    distance_expr = (
        "point.distance(point({latitude: $user_lat, longitude: $user_lng}), "
        "point({latitude: r.lat, longitude: r.lng})) / 1000.0"
        if has_user_location
        else "null"
    )

    q = f"""
    MATCH (r:Restaurant)
    OPTIONAL MATCH (r)-[:HAS_ATTRIBUTE]->(att:Attribute)
    OPTIONAL MATCH (r)-[:IN_AREA]->(a:Area)
    OPTIONAL MATCH (r)-[:HAS_CATEGORY]->(cat:Category)
    OPTIONAL MATCH (r)-[:HAS_CUISINE]->(cui:Cuisine)
    OPTIONAL MATCH (r)-[:SERVES_FAMILY]->(df:DishFamily)
    OPTIONAL MATCH (r)-[:HAS_EXTRACTED_ENTITY]->(ee:ExtractedEntity)
    OPTIONAL MATCH (r)-[:IN_COMMUNITY]->(:Community)-[:HAS_REPORT]->(rep:CommunityReport)

    WITH r, a, rep,
         collect(DISTINCT {{type: att.type, score: att.score}}) AS attributes,
         collect(DISTINCT cat.name) AS categories,
         collect(DISTINCT cui.name) AS cuisines,
         collect(DISTINCT df.name) AS dish_families,
         collect(DISTINCT {{name: ee.name, type: ee.type}}) AS extracted_entities,
         CASE WHEN r.lat IS NULL OR r.lng IS NULL THEN null ELSE {distance_expr} END AS distance_km

    WITH r, a, rep, attributes, categories, cuisines, dish_families, extracted_entities, distance_km,
         (
            reduce(score = 0, t IN $terms |
                score
                + CASE WHEN toLower(coalesce(r.name, "")) CONTAINS toLower(t) THEN 5 ELSE 0 END
                + CASE WHEN any(x IN categories WHERE toLower(coalesce(x, "")) CONTAINS toLower(t)) THEN 4 ELSE 0 END
                + CASE WHEN any(x IN dish_families WHERE toLower(coalesce(x, "")) CONTAINS toLower(t)) THEN 6 ELSE 0 END
                + CASE WHEN any(x IN r.top_menu_items WHERE toLower(coalesce(x, "")) CONTAINS toLower(t)) THEN 5 ELSE 0 END
                + CASE WHEN any(x IN extracted_entities WHERE toLower(coalesce(x.name, "")) CONTAINS toLower(t)) THEN 3 ELSE 0 END
            )
         ) AS graph_match_score

    WHERE graph_match_score > 0

    RETURN r.store_key AS store_key,
           r.name AS name,
           r.address AS address,
           a.name AS district,
           a.city AS city,
           coalesce(r.rating, r.gmaps_rating, r.foody_rating) AS rating,
           r.lat AS lat,
           r.lng AS lng,
           distance_km,
           r.price_band AS price_band,
           r.top_menu_items AS top_menu_items,
           r.menu_price_min AS menu_price_min,
           r.menu_price_max AS menu_price_max,
           r.menu_price_median AS menu_price_median,
           categories,
           cuisines,
           dish_families,
           extracted_entities,
           attributes,
           rep.summary AS community_report,
           graph_match_score

    ORDER BY graph_match_score DESC,
             CASE WHEN distance_km IS NULL THEN 1 ELSE 0 END,
             distance_km ASC,
             CASE WHEN coalesce(r.rating, r.gmaps_rating, r.foody_rating) IS NULL THEN 1 ELSE 0 END,
             coalesce(r.rating, r.gmaps_rating, r.foody_rating) DESC

    LIMIT $top_k
    """

    params = {
        "terms": terms,
        "top_k": int(top_k),
        "user_lat": user_lat,
        "user_lng": user_lng,
    }

    rows = neo4j_client.run(q, params)
    return [add_user_distance_to_record(r, user_lat, user_lng) for r in rows]


def debug_graph_only(
    query: str,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
    top_k: int = 5,
):
    """
    Debug xem graph rỗng ở bước nào.
    """
    intent = parse_intent(query)

    print("===== PARSED INTENT =====")
    print(intent)

    print("\n===== STRICT / RELAXED GRAPH SEARCH =====")
    for name, it in build_relaxed_graph_intents(intent):
        try:
            rows = graph_candidate_search(
                it,
                top_k=max(top_k * 3, 15),
                user_lat=user_lat,
                user_lng=user_lng,
            ) or []
            rows = _dedupe_records_by_store_key(rows, top_k=top_k)
            print(f"{name}: {len(rows)} -> {[r.get('store_key') for r in rows]}")
        except Exception as e:
            print(f"{name}: ERROR {type(e).__name__}: {e}")

    print("\n===== BROAD KEYWORD GRAPH SEARCH =====")
    try:
        broad = graph_broad_keyword_search(
            query=query,
            intent=intent,
            top_k=max(top_k * 3, 15),
            user_lat=user_lat,
            user_lng=user_lng,
        )
        broad = _dedupe_records_by_store_key(broad, top_k=top_k)
        print(f"broad: {len(broad)} -> {[r.get('store_key') for r in broad]}")
        for r in broad:
            print({
                "store_key": r.get("store_key"),
                "name": r.get("name"),
                "distance_km": r.get("distance_km"),
                "rating": r.get("rating"),
                "categories": r.get("categories"),
                "dish_families": r.get("dish_families"),
                "graph_match_score": r.get("graph_match_score"),
            })
    except Exception as e:
        print(f"broad: ERROR {type(e).__name__}: {e}")


def graph_candidate_top5_only_records(
    query: str,
    top_k: int = 5,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[dict]:
    """
    Fixed graph_only:
    - Có user_lat/user_lng.
    - Strict trước, relax dần.
    - Nếu vẫn thiếu thì dùng broad keyword graph search.
    - Nếu có seed thì subgraph expand bằng SIMILAR_TO.
    - Không fallback vector/text.
    """
    intent = parse_intent(query)

    fetch_k = max(top_k * 3, 15)
    candidates = []
    seen = set()

    def add_rows(rows, source_name):
        nonlocal candidates, seen
        for r in rows or []:
            sk = _canon_store_key(r.get("store_key"))
            if not sk or sk in seen:
                continue
            r = dict(r)
            r["graph_source"] = source_name
            seen.add(sk)
            candidates.append(r)
            if len(candidates) >= top_k:
                return

    # 1. Strict + relaxed graph_candidate_search
    for name, it in build_relaxed_graph_intents(intent):
        if len(candidates) >= top_k:
            break
        try:
            rows = graph_candidate_search(
                it,
                top_k=fetch_k,
                user_lat=user_lat,
                user_lng=user_lng,
            ) or []
            add_rows(rows, name)
        except Exception as e:
            print(f"⚠️ graph_candidate_search failed at {name}: {type(e).__name__}: {e}")

    # 2. Nếu có seed mà vẫn thiếu, expand trong graph bằng SIMILAR_TO
    if len(candidates) < top_k:
        seed_keys = [_canon_store_key(r.get("store_key")) for r in candidates]
        seed_keys = [x for x in seed_keys if x]

        if seed_keys:
            try:
                neighbors = subgraph_expand_candidates(
                    seed_store_keys=seed_keys,
                    max_neighbors=fetch_k,
                    user_lat=user_lat,
                    user_lng=user_lng,
                ) or []
                add_rows(neighbors, "subgraph_expand")
            except Exception as e:
                print(f"⚠️ subgraph_expand_candidates failed: {type(e).__name__}: {e}")

    # 3. Nếu vẫn thiếu hoặc ban đầu không có seed, broad graph keyword search
    if len(candidates) < top_k:
        try:
            broad = graph_broad_keyword_search(
                query=query,
                intent=intent,
                top_k=fetch_k,
                user_lat=user_lat,
                user_lng=user_lng,
            ) or []
            add_rows(broad, "broad_keyword_graph")
        except Exception as e:
            print(f"⚠️ graph_broad_keyword_search failed: {type(e).__name__}: {e}")

    candidates = _dedupe_records_by_store_key(candidates, top_k=top_k)

    # normalize nếu hàm _normalize_candidate đã có
    try:
        return [_normalize_candidate(c, i + 1) for i, c in enumerate(candidates)]
    except NameError:
        for i, c in enumerate(candidates, start=1):
            c["rank"] = i
        return candidates


def graph_candidate_top5_only(
    query: str,
    top_k: int = 5,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[str]:
    rows = graph_candidate_top5_only_records(
        query=query,
        top_k=top_k,
        user_lat=user_lat,
        user_lng=user_lng,
    )
    return [_canon_store_key(r.get("store_key")) for r in rows if _canon_store_key(r.get("store_key"))][:top_k]


# Update lại VARIANT_FUNCS phần graph_only
VARIANT_FUNCS["graph_only"] = graph_candidate_top5_only

if "VARIANT_DETAIL_FUNCS" in globals():
    VARIANT_DETAIL_FUNCS["graph_only"] = graph_candidate_top5_only_records


# ============================================================
# Test đúng query T001 có user_lat/user_lng
# ============================================================

qrow = {
    "test_id": "T001",
    "query": "Mình muốn ăn cơm gà ngon gần đây, rating ổn và có nhiều comment thật.",
    "user_lat": 21.005118,
    "user_lng": 105.845592,
}

debug_graph_only(
    qrow["query"],
    user_lat=qrow["user_lat"],
    user_lng=qrow["user_lng"],
    top_k=5,
)

print("\n===== FIXED graph_only =====")
print(
    VARIANT_FUNCS["graph_only"](
        qrow["query"],
        top_k=5,
        user_lat=qrow["user_lat"],
        user_lng=qrow["user_lng"],
    )
)

print("\n===== FIXED graph_only detail =====")
rows = graph_candidate_top5_only_records(
    qrow["query"],
    top_k=5,
    user_lat=qrow["user_lat"],
    user_lng=qrow["user_lng"],
)
for r in rows:
    print({
        "rank": r.get("rank"),
        "store_key": r.get("store_key"),
        "name": r.get("name"),
        "distance_km": r.get("distance_km"),
        "rating": r.get("rating"),
        "graph_source": r.get("graph_source"),
        "categories": r.get("categories"),
        "dish_families": r.get("dish_families"),
    })

===== PARSED INTENT =====
{'query_type': 'search', 'district': None, 'cuisines': [], 'categories': ['Cơm Gà'], 'dish_name': 'cơm gà', 'min_rating': 4, 'max_distance_km': None, 'price_band': None, 'geo_intent': 'nearby', 'entity_terms': ['comment thật', 'nhiều comment'], 'required_attributes': ['food_quality'], 'sentiment_pref': 'positive', 'top_k': 5}

===== STRICT / RELAXED GRAPH SEARCH =====
strict: 0 -> []
drop_entity_terms: 2 -> ['117708', '53514']
drop_entity_terms_required_attributes: 3 -> ['117708', '53514', '117760']
drop_entity_terms_attrs_min_rating: 3 -> ['117708', '53514', '117760']
large_distance: 3 -> ['117708', '53514', '117760']
dish_only_large_distance: 5 -> ['117526', '136086', '117708', '130643', '124125']
category_only_large_distance: 3 -> ['117708', '53514', '117760']

===== BROAD KEYWORD GRAPH SEARCH =====
broad: 5 -> ['7240', '10307', '11471', '130643', '25639']
{'store_key': '7240', 'name': 'Cơm Gà, Mì Cay & Mì Trộn Bếp Nhà Cám - Tôn Đức Thắng', 'distance_km': 2

In [99]:
# ============================================================
# 5 RETRIEVAL VARIANTS WITH USER LAT/LNG
# - vector_only
# - graph_only
# - text_unit_only
# - hybrid_no_ce
# - hybrid_with_ce
#
# Hybrid with CE bám đúng logic:
# hybrid_retrieve(query, top_k, use_cross_encoder=True, user_lat, user_lng)
# ============================================================

from typing import Any, Dict, List, Optional
import math
import traceback


# =========================
# Helpers
# =========================

def _safe_float(x: Any, default: Optional[float] = None) -> Optional[float]:
    if x is None:
        return default
    try:
        if pd.isna(x):
            return default
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        return default


def _get_candidate_id(c: Any) -> str:
    if isinstance(c, dict):
        return str(
            c.get("store_key")
            or c.get("restaurant_id")
            or c.get("id")
            or ""
        ).strip()
    return str(c).strip()


def _get_candidate_name(c: Dict[str, Any]) -> str:
    return (
        c.get("name")
        or c.get("restaurant_name")
        or c.get("store_name")
        or c.get("title")
        or ""
    )


def _normalize_candidate(c: Dict[str, Any], rank: int) -> Dict[str, Any]:
    evidence = c.get("evidence", [])
    if isinstance(evidence, str):
        evidence = [evidence]
    elif not isinstance(evidence, list):
        evidence = []

    return {
        "rank": rank,
        "store_key": _get_candidate_id(c),
        "name": _get_candidate_name(c),
        "address": c.get("address", ""),
        "district": c.get("district", ""),
        "city": c.get("city", ""),
        "lat": c.get("lat", c.get("latitude", "")),
        "lng": c.get("lng", c.get("longitude", "")),
        "distance_km": c.get("distance_km", c.get("distance", "")),
        "final_score": c.get("final_score", c.get("score", c.get("rrf_score", ""))),
        "ce_score": c.get("ce_score", ""),
        "final_score_before_ce": c.get("final_score_before_ce", ""),
        "rating": c.get("rating", c.get("gmaps_rating", c.get("foody_rating", ""))),
        "price_band": c.get("price_band", ""),
        "dish_families": c.get("dish_families", []),
        "categories": c.get("categories", []),
        "source_flags": c.get("source_flags", []),
        "community_id": c.get("community_id", ""),
        "community_report": c.get("community_report", ""),
        "evidence": evidence,
        "evidence_count": len([x for x in evidence if str(x).strip()]),
    }


def _unique_store_keys(items: List[Any], top_k: int = 5) -> List[str]:
    seen = set()
    out = []

    for item in items or []:
        sk = _get_candidate_id(item)
        if not sk or sk in seen:
            continue
        seen.add(sk)
        out.append(sk)
        if len(out) >= top_k:
            break

    return out


def _unique_candidate_records(items: List[Dict[str, Any]], top_k: int = 5) -> List[Dict[str, Any]]:
    seen = set()
    out = []

    for item in items or []:
        sk = _get_candidate_id(item)
        if not sk or sk in seen:
            continue
        seen.add(sk)
        out.append(item)
        if len(out) >= top_k:
            break

    return out


# ============================================================
# 1. VECTOR ONLY - nhận user_lat/user_lng để interface thống nhất
# ============================================================

def retrieve_vector_only_records(
    query: str,
    top_k: int = 5,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[Dict[str, Any]]:
    """
    Vector search theo Restaurant document.
    Có user_lat/user_lng trong signature để pipeline truyền thống nhất.
    Nếu vector_search_restaurants có hỗ trợ user_lat/user_lng thì truyền vào.
    Nếu không hỗ trợ thì gọi bản cũ.
    """
    fetch_k = max(top_k * 5, 30)

    try:
        rows = vector_search_restaurants(
            query,
            top_k=fetch_k,
            user_lat=user_lat,
            user_lng=user_lng,
        )
    except TypeError:
        rows = vector_search_restaurants(query, top_k=fetch_k)

    rows = rows or []

    if rows and not isinstance(rows[0], dict):
        rows = [{"store_key": str(x)} for x in rows]

    unique = _unique_candidate_records(rows, top_k=top_k)
    return [_normalize_candidate(c, i + 1) for i, c in enumerate(unique)]


def retrieve_vector_only(
    query: str,
    top_k: int = 5,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[str]:
    return _unique_store_keys(
        retrieve_vector_only_records(query, top_k, user_lat, user_lng),
        top_k=top_k,
    )


# ============================================================
# 2. GRAPH ONLY - chỉ graph, có user_lat/user_lng, không fallback vector/text
# ============================================================

def graph_candidate_top5_only_records(
    query: str,
    top_k: int = 5,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[Dict[str, Any]]:
    """
    Graph-only:
    1. parse_intent
    2. graph_candidate_search với user_lat/user_lng
    3. nếu thiếu top_k thì mở rộng subgraph bằng SIMILAR_TO
    Không fallback sang vector/text.
    """
    intent = parse_intent(query)

    fetch_k = max(top_k * 3, 15)

    direct = graph_candidate_search(
        intent,
        top_k=fetch_k,
        user_lat=user_lat,
        user_lng=user_lng,
    ) or []

    direct = _unique_candidate_records(direct, top_k=fetch_k)

    candidates = list(direct)

    if len(candidates) < top_k:
        seed_keys = [_get_candidate_id(c) for c in candidates]
        seed_keys = [x for x in seed_keys if x]

        neighbors = subgraph_expand_candidates(
            seed_store_keys=seed_keys,
            max_neighbors=max(top_k * 3, 15),
            user_lat=user_lat,
            user_lng=user_lng,
        ) or []

        seen = {_get_candidate_id(c) for c in candidates}

        for n in neighbors:
            sk = _get_candidate_id(n)
            if not sk or sk in seen:
                continue
            seen.add(sk)
            candidates.append(n)
            if len(candidates) >= top_k:
                break

    candidates = _unique_candidate_records(candidates, top_k=top_k)
    return [_normalize_candidate(c, i + 1) for i, c in enumerate(candidates)]


def graph_candidate_top5_only(
    query: str,
    top_k: int = 5,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[str]:
    return _unique_store_keys(
        graph_candidate_top5_only_records(query, top_k, user_lat, user_lng),
        top_k=top_k,
    )


# ============================================================
# 3. TEXT UNIT ONLY - top 5 quán unique, nhận user_lat/user_lng
# ============================================================

def retrieve_text_unit_top5_unique_records(
    query: str,
    top_k: int = 5,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[Dict[str, Any]]:
    """
    Text-unit vector search.
    Vì nhiều chunk có thể thuộc cùng store_key, lấy fetch_k lớn rồi dedupe store_key.
    """
    fetch_k = max(top_k * 10, 50)

    try:
        rows = vector_search_text_units(
            query,
            top_k=fetch_k,
            user_lat=user_lat,
            user_lng=user_lng,
        )
    except TypeError:
        rows = vector_search_text_units(query, top_k=fetch_k)

    rows = rows or []

    if rows and not isinstance(rows[0], dict):
        rows = [{"store_key": str(x)} for x in rows]

    unique = _unique_candidate_records(rows, top_k=top_k)
    return [_normalize_candidate(c, i + 1) for i, c in enumerate(unique)]


def retrieve_text_unit_top5_unique(
    query: str,
    top_k: int = 5,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[str]:
    return _unique_store_keys(
        retrieve_text_unit_top5_unique_records(query, top_k, user_lat, user_lng),
        top_k=top_k,
    )


# ============================================================
# 4. HYBRID NO CE - truyền user_lat/user_lng
# ============================================================

def retrieve_hybrid_no_ce_records(
    query: str,
    top_k: int = 5,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[Dict[str, Any]]:
    """
    Hybrid retrieval without cross-encoder.
    Có truyền user_lat/user_lng xuống hybrid_retrieve.
    """
    intent, ranked = hybrid_retrieve(
        query,
        top_k=top_k,
        use_cross_encoder=False,
        user_lat=user_lat,
        user_lng=user_lng,
    )

    ranked = ranked or []
    ranked = _unique_candidate_records(ranked, top_k=top_k)

    return [_normalize_candidate(c, i + 1) for i, c in enumerate(ranked)]


def retrieve_hybrid_no_ce(
    query: str,
    top_k: int = 5,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[str]:
    return _unique_store_keys(
        retrieve_hybrid_no_ce_records(query, top_k, user_lat, user_lng),
        top_k=top_k,
    )


# ============================================================
# 5. HYBRID WITH CE - bám y hệt logic file copy
# ============================================================

def retrieve_hybrid_with_ce_records(
    query: str,
    top_k: int = 5,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[Dict[str, Any]]:
    """
    Hybrid retrieval with cross-encoder.

    Logic giống file copy:
    intent, ranked = hybrid_retrieve(
        query,
        top_k=top_k,
        use_cross_encoder=True,
        user_lat=user_lat,
        user_lng=user_lng,
    )

    Sau đó normalize candidate để giữ:
    distance_km, final_score, ce_score, final_score_before_ce, rating, evidence...
    """
    intent, ranked = hybrid_retrieve(
        query,
        top_k=top_k,
        use_cross_encoder=True,
        user_lat=user_lat,
        user_lng=user_lng,
    )

    ranked = ranked or []
    ranked = _unique_candidate_records(ranked, top_k=top_k)

    return [_normalize_candidate(c, i + 1) for i, c in enumerate(ranked)]


def retrieve_hybrid_with_ce(
    query: str,
    top_k: int = 5,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[str]:
    return _unique_store_keys(
        retrieve_hybrid_with_ce_records(query, top_k, user_lat, user_lng),
        top_k=top_k,
    )


# ============================================================
# VARIANT FUNCS
# ============================================================

VARIANT_FUNCS = {
    "vector_only": retrieve_vector_only,
    "graph_only": graph_candidate_top5_only,
    "text_unit_only": retrieve_text_unit_top5_unique,
    "hybrid_no_ce": retrieve_hybrid_no_ce,
    "hybrid_with_ce": retrieve_hybrid_with_ce,
}

VARIANT_DETAIL_FUNCS = {
    "vector_only": retrieve_vector_only_records,
    "graph_only": graph_candidate_top5_only_records,
    "text_unit_only": retrieve_text_unit_top5_unique_records,
    "hybrid_no_ce": retrieve_hybrid_no_ce_records,
    "hybrid_with_ce": retrieve_hybrid_with_ce_records,
}


# ============================================================
# QUICK TEST WITH USER LAT/LNG
# ============================================================

qrow = {
    "test_id": "T001",
    "query": "Mình muốn ăn cơm gà ngon gần đây, rating ổn và có nhiều comment thật.",
    "user_lat": 21.005118,
    "user_lng": 105.845592,
}

print("===== STORE KEY ONLY =====")
for var in ["vector_only", "graph_only", "text_unit_only", "hybrid_no_ce", "hybrid_with_ce"]:
    try:
        out = VARIANT_FUNCS[var](
            qrow["query"],
            top_k=5,
            user_lat=qrow["user_lat"],
            user_lng=qrow["user_lng"],
        )
        print(var, out)
    except Exception as e:
        print(var, "ERROR:", type(e).__name__, e)


print("\n===== DETAIL RECORDS WITH DISTANCE / CE =====")
for var in ["vector_only", "graph_only", "text_unit_only", "hybrid_no_ce", "hybrid_with_ce"]:
    try:
        out = VARIANT_DETAIL_FUNCS[var](
            qrow["query"],
            top_k=5,
            user_lat=qrow["user_lat"],
            user_lng=qrow["user_lng"],
        )

        print(f"\n{var}")
        for x in out:
            print({
                "rank": x.get("rank"),
                "store_key": x.get("store_key"),
                "name": x.get("name"),
                "distance_km": x.get("distance_km"),
                "final_score": x.get("final_score"),
                "ce_score": x.get("ce_score"),
                "rating": x.get("rating"),
            })

    except Exception as e:
        print(var, "ERROR:", type(e).__name__, e)

===== STORE KEY ONLY =====
vector_only ['55835', '1255', '10307', '115272', '34875']
graph_only []
text_unit_only ['10307', '73467', '55835', '123151', '68175']
hybrid_no_ce ['123151', '10307', '55835', '128372', '68175']
hybrid_with_ce ['55835', '116044', '123151', '10307', '88782']

===== DETAIL RECORDS WITH DISTANCE / CE =====

vector_only
{'rank': 1, 'store_key': '55835', 'name': 'Tiệm Cơm Gà - Trung Liệt', 'distance_km': 2.696, 'final_score': '', 'ce_score': '', 'rating': 4.2}
{'rank': 2, 'store_key': '1255', 'name': 'Cơm Rang Dế Mèn', 'distance_km': 1.504, 'final_score': '', 'ce_score': '', 'rating': 4.6}
{'rank': 3, 'store_key': '10307', 'name': 'Cơm Gà Xối Mắm Còn - Dương Văn Bé', 'distance_km': 3.084, 'final_score': '', 'ce_score': '', 'rating': 4.7}
{'rank': 4, 'store_key': '115272', 'name': 'He Na Quán - Bánh Cuốn, Bún Chả & Gà Tần - Núi Trúc', 'distance_km': 3.536, 'final_score': '', 'ce_score': '', 'rating': 5.0}
{'rank': 5, 'store_key': '34875', 'name': 'Cherry Hồng - Phở

In [101]:
# ============================================================
# LLM JUDGE HELPERS FOR POOLED GROUND TRUTH
# ============================================================
# Provides:
# - build_judge_payload(query, store_key, pool_row=None)
# - judge_with_llm(payload)
#
# Requires:
# - get_llm()  # LLM function của bạn
#
# Optional dataframes used if available:
# - summary / restaurant_summary_df
# - menu_items
# - text_units / review_chunks / feedback_proc
# ============================================================

import os
import re
import json
import time
import math
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd


# =========================
# CONFIG
# =========================

JUDGE_MODE = "llm"  # "llm" hoặc "manual"

PROMPT_VERSION = "pooled_gt_restaurant_relevance_v2"
RUBRIC_VERSION = "relevance_grade_0_3_v2"

FORCE_REFRESH_JUDGE_LABELS = False

EVAL_OUTPUT_DIR = Path("eval_outputs")
EVAL_OUTPUT_DIR.mkdir(exist_ok=True)

OUT_LLM_LABEL_CACHE = EVAL_OUTPUT_DIR / "pooled_gt_llm_label_cache.json"


# =========================
# SAFE HELPERS
# =========================

def _safe_str(x: Any) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x).strip()


def canon_store_key(x: Any) -> str:
    s = _safe_str(x)
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]
    return s


def _safe_float(x: Any, default: Optional[float] = None) -> Optional[float]:
    if x is None:
        return default
    try:
        if pd.isna(x):
            return default
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        return default


def _safe_int(x: Any, default: int = 0) -> int:
    if x is None:
        return default
    try:
        if pd.isna(x):
            return default
    except Exception:
        pass
    try:
        return int(float(x))
    except Exception:
        return default


def json_dumps(obj: Any) -> str:
    """
    JSON dumps an toàn cho numpy / pandas / list / dict.
    """
    def default(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, pd.Timestamp):
            return str(o)
        return str(o)

    return json.dumps(obj, ensure_ascii=False, default=default)


def read_json_file(path: Path, default=None):
    if default is None:
        default = {}

    path = Path(path)

    if not path.exists() or path.stat().st_size == 0:
        return default

    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return default


def write_json_atomic(obj: Any, path: Path):
    path = Path(path)
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    tmp.replace(path)


def parse_list_like(x: Any) -> List[str]:
    """
    Chuyển list/string/json-string/pipe-string thành list[str].
    """
    if x is None:
        return []

    try:
        if pd.isna(x):
            return []
    except Exception:
        pass

    if isinstance(x, list):
        return [_safe_str(v) for v in x if _safe_str(v)]

    if isinstance(x, tuple) or isinstance(x, set):
        return [_safe_str(v) for v in list(x) if _safe_str(v)]

    s = _safe_str(x)
    if not s:
        return []

    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = json.loads(s.replace("'", '"'))
            if isinstance(parsed, list):
                return [_safe_str(v) for v in parsed if _safe_str(v)]
        except Exception:
            pass

    if "|" in s:
        return [v.strip() for v in s.split("|") if v.strip()]

    if "," in s:
        return [v.strip() for v in s.split(",") if v.strip()]

    return [s]


def _first_existing_col(df: pd.DataFrame, cols: List[str]) -> Optional[str]:
    for c in cols:
        if c in df.columns:
            return c
    return None


def _normalize_store_key_col(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "store_key" not in df.columns:
        for alt in ["store_id", "restaurant_id", "id"]:
            if alt in df.columns:
                df["store_key"] = df[alt]
                break

    if "store_key" in df.columns:
        df["store_key"] = df["store_key"].apply(canon_store_key)

    return df


# =========================
# METADATA HELPERS
# =========================

def _candidate_metadata_sources() -> List[pd.DataFrame]:
    """
    Ưu tiên các dataframe có metadata quán.
    """
    names = [
        "restaurant_summary_df",
        "summary",
        "restaurant_docs",
        "stores_df",
        "restaurants_df",
        "be_google_maps_unique",
        "foody_hust_places_from",
    ]

    dfs = []
    for name in names:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            df = globals()[name]
            if not df.empty:
                dfs.append(df)

    return dfs


def get_candidate_metadata(store_key: str) -> Dict[str, Any]:
    """
    Lấy metadata của quán từ các dataframe có sẵn.
    Nếu không tìm thấy thì vẫn trả store_key để LLM judge không crash.
    """
    sk = canon_store_key(store_key)

    fallback = {
        "store_key": sk,
        "name": "",
        "address": "",
        "district": "",
        "city": "",
        "lat": None,
        "lng": None,
        "rating": None,
        "price_band": "",
        "menu_price_min": None,
        "menu_price_max": None,
        "menu_price_median": None,
        "dish_families": [],
        "categories": [],
        "cuisines": [],
        "top_menu_items": [],
        "community_report": "",
    }

    for src in _candidate_metadata_sources():
        df = _normalize_store_key_col(src)

        if "store_key" not in df.columns:
            continue

        sub = df[df["store_key"].apply(canon_store_key) == sk]
        if sub.empty:
            continue

        r = sub.iloc[0]

        name_col = _first_existing_col(df, ["name", "store_name", "restaurant_name", "title"])
        address_col = _first_existing_col(df, ["address", "store_address", "formatted_address"])
        district_col = _first_existing_col(df, ["district", "area", "ward", "area_name"])
        city_col = _first_existing_col(df, ["city", "province"])
        lat_col = _first_existing_col(df, ["lat", "latitude", "store_lat"])
        lng_col = _first_existing_col(df, ["lng", "longitude", "store_lng", "lon"])
        rating_col = _first_existing_col(df, ["rating", "gmaps_rating", "foody_rating", "avg_rating"])
        price_band_col = _first_existing_col(df, ["price_band", "price_level"])
        min_price_col = _first_existing_col(df, ["menu_price_min", "min_price", "price_min"])
        max_price_col = _first_existing_col(df, ["menu_price_max", "max_price", "price_max"])
        median_price_col = _first_existing_col(df, ["menu_price_median", "median_price", "avg_price"])
        dish_col = _first_existing_col(df, ["dish_families", "dish_family"])
        cat_col = _first_existing_col(df, ["categories", "category_name", "category"])
        cuisine_col = _first_existing_col(df, ["cuisines", "cuisine"])
        top_menu_col = _first_existing_col(df, ["top_menu_items", "menu_items", "example_items"])
        report_col = _first_existing_col(df, ["community_report", "report", "summary_text"])

        return {
            "store_key": sk,
            "name": _safe_str(r.get(name_col, "")) if name_col else "",
            "address": _safe_str(r.get(address_col, "")) if address_col else "",
            "district": _safe_str(r.get(district_col, "")) if district_col else "",
            "city": _safe_str(r.get(city_col, "")) if city_col else "",
            "lat": _safe_float(r.get(lat_col), None) if lat_col else None,
            "lng": _safe_float(r.get(lng_col), None) if lng_col else None,
            "rating": _safe_float(r.get(rating_col), None) if rating_col else None,
            "price_band": _safe_str(r.get(price_band_col, "")) if price_band_col else "",
            "menu_price_min": _safe_float(r.get(min_price_col), None) if min_price_col else None,
            "menu_price_max": _safe_float(r.get(max_price_col), None) if max_price_col else None,
            "menu_price_median": _safe_float(r.get(median_price_col), None) if median_price_col else None,
            "dish_families": parse_list_like(r.get(dish_col, [])) if dish_col else [],
            "categories": parse_list_like(r.get(cat_col, [])) if cat_col else [],
            "cuisines": parse_list_like(r.get(cuisine_col, [])) if cuisine_col else [],
            "top_menu_items": parse_list_like(r.get(top_menu_col, [])) if top_menu_col else [],
            "community_report": _safe_str(r.get(report_col, "")) if report_col else "",
        }

    return fallback


# =========================
# MENU EVIDENCE
# =========================

def extract_menu_evidence(store_key: str, max_items: int = 10) -> List[Dict[str, Any]]:
    """
    Lấy evidence từ menu_items nếu có.
    Ưu tiên món có order_count cao.
    """
    if "menu_items" not in globals() or not isinstance(menu_items, pd.DataFrame) or menu_items.empty:
        return []

    df = _normalize_store_key_col(menu_items)

    if "store_key" not in df.columns:
        return []

    sk = canon_store_key(store_key)
    sub = df[df["store_key"].apply(canon_store_key) == sk].copy()

    if sub.empty:
        return []

    if "order_count" in sub.columns:
        sub["_sort_order"] = pd.to_numeric(sub["order_count"], errors="coerce").fillna(0)
        sub = sub.sort_values("_sort_order", ascending=False)
    elif "like_count" in sub.columns:
        sub["_sort_like"] = pd.to_numeric(sub["like_count"], errors="coerce").fillna(0)
        sub = sub.sort_values("_sort_like", ascending=False)

    evidence = []

    for idx, r in sub.head(max_items).iterrows():
        item_name = _safe_str(r.get("item_name", r.get("name", "")))
        if not item_name:
            continue

        menu_item_id = _safe_str(r.get("menu_item_id", r.get("id", idx)))

        evidence.append({
            "evidence_id": f"menu:{menu_item_id}",
            "source": "menu",
            "item_name": item_name,
            "category_name": _safe_str(r.get("category_name", "")),
            "price": _safe_float(r.get("price"), None),
            "old_price": _safe_float(r.get("old_price"), None),
            "order_count": _safe_int(r.get("order_count"), 0),
            "like_count": _safe_int(r.get("like_count"), 0),
            "dislike_count": _safe_int(r.get("dislike_count"), 0),
            "text": (
                f"Món: {item_name}; "
                f"category={_safe_str(r.get('category_name', ''))}; "
                f"price={_safe_str(r.get('price', ''))}; "
                f"order_count={_safe_str(r.get('order_count', ''))}; "
                f"like_count={_safe_str(r.get('like_count', ''))}"
            )
        })

    return evidence


# =========================
# REVIEW / TEXT UNIT EVIDENCE
# =========================

def _review_source_df() -> Optional[pd.DataFrame]:
    for name in ["text_units", "review_chunks", "feedback_proc", "feedback_df", "reviews_df"]:
        if name in globals() and isinstance(globals()[name], pd.DataFrame) and not globals()[name].empty:
            return globals()[name]
    return None


def _keyword_overlap_score(query: str, text: str) -> int:
    """
    Score đơn giản để chọn review evidence gần query.
    """
    q = str(query or "").lower()
    t = str(text or "").lower()

    keywords = [
        "cơm", "cơm gà", "gà", "gà rán", "phở", "phở bò", "phở gà",
        "bún", "bún chả", "bánh cuốn", "ngon", "rẻ", "sạch", "giao nhanh",
        "đóng gói", "comment", "review", "rating", "giá", "sinh viên",
        "gần", "không xa", "trưa", "tối", "sáng"
    ]

    score = 0
    for kw in keywords:
        if kw in q and kw in t:
            score += 2
        elif kw in t:
            score += 1

    return score


def extract_review_evidence(query: str, store_key: str, max_items: int = 6) -> List[Dict[str, Any]]:
    """
    Lấy evidence từ text_units / feedback_proc.
    Ưu tiên review/chunk có overlap với query.
    """
    src = _review_source_df()
    if src is None:
        return []

    df = _normalize_store_key_col(src)

    if "store_key" not in df.columns:
        return []

    sk = canon_store_key(store_key)
    sub = df[df["store_key"].apply(canon_store_key) == sk].copy()

    if sub.empty:
        return []

    text_col = _first_existing_col(sub, ["chunk_text", "feedback", "review", "comment", "content", "text"])
    if not text_col:
        return []

    id_col = _first_existing_col(sub, ["text_unit_id", "review_id", "feedback_id", "id"])
    rating_col = _first_existing_col(sub, ["rating", "user_rating", "score"])
    sentiment_col = _first_existing_col(sub, ["sentiment", "sentiment_label"])

    sub["_ev_text"] = sub[text_col].astype(str)
    sub["_overlap"] = sub["_ev_text"].apply(lambda x: _keyword_overlap_score(query, x))

    # Nếu có rating/sentiment thì vẫn giữ, nhưng sort chính theo overlap
    sub = sub.sort_values("_overlap", ascending=False)

    evidence = []

    for idx, r in sub.head(max_items).iterrows():
        text = _safe_str(r.get(text_col, ""))
        if not text:
            continue

        ev_id = _safe_str(r.get(id_col, idx)) if id_col else str(idx)

        evidence.append({
            "evidence_id": f"review:{ev_id}",
            "source": "review",
            "rating": _safe_float(r.get(rating_col), None) if rating_col else None,
            "sentiment": _safe_str(r.get(sentiment_col, "")) if sentiment_col else "",
            "text": text[:1200],
        })

    return evidence


# =========================
# PAYLOAD BUILDER
# =========================

def build_judge_payload(
    query: str,
    store_key: str,
    pool_row: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Tạo payload để LLM judge chấm candidate theo query.
    """
    sk = canon_store_key(store_key)
    meta = get_candidate_metadata(sk)

    evidence = []

    evidence.append({
        "evidence_id": f"metadata:{sk}",
        "source": "metadata",
        "text": {
            "store_key": meta.get("store_key"),
            "name": meta.get("name"),
            "address": meta.get("address"),
            "district": meta.get("district"),
            "city": meta.get("city"),
            "lat": meta.get("lat"),
            "lng": meta.get("lng"),
            "rating": meta.get("rating"),
            "price_band": meta.get("price_band"),
            "menu_price_min": meta.get("menu_price_min"),
            "menu_price_max": meta.get("menu_price_max"),
            "menu_price_median": meta.get("menu_price_median"),
            "dish_families": meta.get("dish_families"),
            "categories": meta.get("categories"),
            "cuisines": meta.get("cuisines"),
            "top_menu_items": meta.get("top_menu_items"),
            "community_report": meta.get("community_report"),
        }
    })

    evidence.extend(extract_menu_evidence(sk, max_items=10))
    evidence.extend(extract_review_evidence(query, sk, max_items=6))

    # Dedupe evidence
    seen = set()
    dedup_evidence = []

    for ev in evidence:
        ev_id = _safe_str(ev.get("evidence_id"))
        if not ev_id or ev_id in seen:
            continue
        seen.add(ev_id)
        dedup_evidence.append(ev)

    payload = {
        "prompt_version": PROMPT_VERSION,
        "rubric_version": RUBRIC_VERSION,
        "query": query,
        "candidate": meta,
        "retrieval_pool_info": pool_row or {},
        "evidence": dedup_evidence[:18],
        "instruction": (
            "Judge candidate restaurant relevance for the user query. "
            "Use only provided candidate metadata and evidence. "
            "Do not invent facts."
        )
    }

    return payload


# =========================
# LLM JUDGE
# =========================

LLM_LABEL_CACHE = read_json_file(OUT_LLM_LABEL_CACHE, default={})


def label_cache_key(payload: Dict[str, Any]) -> str:
    """
    Cache key ổn định theo query + store_key + evidence.
    """
    stable = {
        "prompt_version": PROMPT_VERSION,
        "rubric_version": RUBRIC_VERSION,
        "query": payload.get("query"),
        "store_key": payload.get("candidate", {}).get("store_key"),
        "candidate": payload.get("candidate"),
        "evidence": payload.get("evidence"),
    }

    blob = json_dumps(stable)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()


def extract_json_object(raw: str) -> Dict[str, Any]:
    """
    Parse JSON từ output LLM.
    """
    raw = str(raw or "").strip()
    raw = re.sub(r"```json|```", "", raw).strip()

    try:
        return json.loads(raw)
    except Exception:
        pass

    m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if not m:
        raise ValueError(f"No JSON object found in LLM output: {raw[:500]}")

    return json.loads(m.group(0))


def _clamp_relevance_grade(x: Any) -> int:
    return max(0, min(3, _safe_int(x, 0)))


def _clamp_evidence_support(x: Any) -> int:
    return max(0, min(2, _safe_int(x, 0)))


LABEL_SYSTEM_PROMPT = f"""
Bạn là giám khảo relevance cho hệ gợi ý quán ăn tiếng Việt.

Nhiệm vụ:
Chấm một quán ăn có phù hợp với query người dùng hay không.
Chỉ được dùng candidate metadata và evidence được cung cấp.
Không được tự bịa thông tin.

Rubric relevance_grade 0-3:
0 = Không phù hợp hoặc không có bằng chứng liên quan.
1 = Liên quan lỏng: có chút liên quan tới món/khu vực/giá nhưng thiếu ý chính.
2 = Phù hợp: đáp ứng phần lớn yêu cầu chính của query.
3 = Rất phù hợp: đáp ứng rõ món/nhu cầu chính, có evidence tốt về chất lượng/rating/giá/khoảng cách/comment tùy query.

Rubric evidence_support 0-2:
0 = Evidence yếu hoặc thiếu.
1 = Có một số evidence hỗ trợ.
2 = Evidence rõ, trực tiếp, đáng tin.

Các facet thường gặp:
- dish/menu: món người dùng muốn ăn.
- geo/distance: gần đây, không quá xa, gần nhất.
- rating/review: rating ổn, nhiều review/comment thật.
- price: giá sinh viên, dưới 50k, không quá 100k.
- quality: ngon, sạch sẽ, suất đầy đặn, giao nhanh, đóng gói tốt.
- negative avoidance: tránh vệ sinh kém, tránh bị chê.

Luật chấm:
- Nếu query hỏi món cụ thể mà quán không có món đó hoặc món rất xa yêu cầu: grade tối đa 1.
- Nếu không có evidence nào ngoài tên quán mà tên cũng không khớp query: grade 0.
- Nếu chỉ match vị trí nhưng không match món/nhu cầu chính: grade tối đa 1.
- Nếu match món rõ nhưng thiếu review/comment/rating evidence: thường grade 1-2.
- Nếu match món rõ + rating/review/menu/giá phù hợp: grade 2-3.

Trả về JSON hợp lệ duy nhất, không markdown:
{{
  "relevance_grade": 0,
  "evidence_support": 0,
  "matched_facets": [],
  "missing_or_weak_facets": [],
  "reasoning_short": "",
  "evidence_ids": [],
  "confidence": 0.0
}}

prompt_version={PROMPT_VERSION}
rubric_version={RUBRIC_VERSION}
""".strip()


def judge_with_llm(payload: Dict[str, Any]) -> Dict[str, Any]:
    """
    Gọi LLM để chấm relevance_grade 0-3.
    Có cache incremental: mất mạng chạy lại không chấm lại item cũ.
    """
    if JUDGE_MODE == "manual":
        return {
            "relevance_grade": "",
            "evidence_support": "",
            "matched_facets": [],
            "missing_or_weak_facets": [],
            "reasoning_short": "",
            "evidence_ids": [],
            "confidence": "",
            "_cache_hit": False,
        }

    if "get_llm" not in globals():
        raise NameError("Không tìm thấy get_llm(). Hãy load LLM trước hoặc set JUDGE_MODE='manual'.")

    key = label_cache_key(payload)

    if (not FORCE_REFRESH_JUDGE_LABELS) and key in LLM_LABEL_CACHE:
        cached = LLM_LABEL_CACHE[key].copy()
        cached["_cache_hit"] = True
        return cached

    llm = get_llm()

    resp = llm.invoke([
        {"role": "system", "content": LABEL_SYSTEM_PROMPT},
        {"role": "user", "content": json_dumps(payload)},
    ])

    raw = getattr(resp, "content", str(resp))

    parsed = extract_json_object(raw)

    parsed["relevance_grade"] = _clamp_relevance_grade(parsed.get("relevance_grade", 0))
    parsed["evidence_support"] = _clamp_evidence_support(parsed.get("evidence_support", 0))

    if not isinstance(parsed.get("matched_facets", []), list):
        parsed["matched_facets"] = [str(parsed.get("matched_facets"))]

    if not isinstance(parsed.get("missing_or_weak_facets", []), list):
        parsed["missing_or_weak_facets"] = [str(parsed.get("missing_or_weak_facets"))]

    if not isinstance(parsed.get("evidence_ids", []), list):
        parsed["evidence_ids"] = [str(parsed.get("evidence_ids"))]

    parsed["reasoning_short"] = _safe_str(parsed.get("reasoning_short", parsed.get("reasoning", "")))[:500]

    try:
        parsed["confidence"] = float(parsed.get("confidence", 0.0))
    except Exception:
        parsed["confidence"] = 0.0

    parsed["_cache_hit"] = False
    parsed["_judged_at"] = time.strftime("%Y-%m-%d %H:%M:%S")

    LLM_LABEL_CACHE[key] = parsed
    write_json_atomic(LLM_LABEL_CACHE, OUT_LLM_LABEL_CACHE)

    return parsed


# =========================
# QUICK SMOKE TEST
# =========================

print("✅ LLM judge helpers ready:")
print("- build_judge_payload(query, store_key, pool_row=None)")
print("- judge_with_llm(payload)")
print(f"- cache path: {OUT_LLM_LABEL_CACHE}")
print(f"- cache size: {len(LLM_LABEL_CACHE)}")

✅ LLM judge helpers ready:
- build_judge_payload(query, store_key, pool_row=None)
- judge_with_llm(payload)
- cache path: eval_outputs\pooled_gt_llm_label_cache.json
- cache size: 0


In [102]:
payload = build_judge_payload(
    query="Mình muốn ăn cơm gà ngon gần đây, rating ổn và có nhiều comment thật.",
    store_key="55835",
    pool_row={"variant": "hybrid_with_ce", "rank": 1}
)

label = judge_with_llm(payload)
label

C:\Users\FPT SHOP\AppData\Local\Temp\ipykernel_21928\1306086504.py:145: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(x):


{'relevance_grade': 2,
 'evidence_support': 2,
 'matched_facets': ['dish/menu', 'rating/review'],
 'missing_or_weak_facets': ['geo/distance', 'quality'],
 'reasoning_short': "Quán match rõ nhu cầu cơm gà với nhiều món cơm gà trong menu và rating 4.2 là khá ổn. Có nhiều comment/review thật được cung cấp. Tuy nhiên yếu tố 'ngon' không thật sự mạnh vì review lẫn lộn, chỉ có một review khen 'Khá ngon' trong khi nhiều review chê chất lượng, thiếu món/nước sốt, gà/cơm cháy hoặc gà còn sống. Không có thông tin khoảng cách tới người dùng nên chưa xác nhận được 'gần đây'.",
 'evidence_ids': ['metadata:55835',
  'menu:3931513',
  'menu:3931512',
  'menu:4283039',
  'review:tu_761c58de755a634d',
  'review:tu_91bb22aa1962a164',
  'review:tu_486348330a0d0b79',
  'review:tu_bb2b308efa0f3d6c'],
 'confidence': 0.86,
 '_cache_hit': False,
 '_judged_at': '2026-05-29 11:18:13'}

In [103]:
# ============================================================
# POOLED GROUND TRUTH EVALUATION FOR 5 RETRIEVAL VARIANTS
# ============================================================
# Output:
#   eval_outputs/pooled_gt_5variants_labels.csv
#   eval_outputs/pooled_gt_5variants_detail.csv
#   eval_outputs/pooled_gt_5variants_wide.csv
#   eval_outputs/pooled_gt_5variants_summary.csv
#   eval_outputs/pooled_gt_5variants_query_progress.csv
#
# Yêu cầu biến/hàm đã có trong notebook:
# - GRAPH_RAG_GEO_TEST_QUERIES
# - VARIANT_FUNCS
# - judge_with_llm(payload)
# - build_judge_payload(query, store_key, pool_row=None)
#
# Nếu có VARIANT_DETAIL_FUNCS thì dùng detail records.
# Nếu không có thì dùng VARIANT_FUNCS trả store_key.
# ============================================================

import os
import re
import json
import math
import time
import shutil
import traceback
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    display = print


# =========================
# 0. CONFIG
# =========================

TOP_K_EVAL = 5
POOL_TOP_K_PER_VARIANT = 5

JUDGE_MODE = "llm"          # "llm" hoặc "manual"
FORCE_REJUDGE = False       # True nếu muốn chấm lại toàn bộ bằng LLM
SMOKE_TEST_N = None         # ví dụ 3 để test nhanh, None để chạy đủ

EVAL_OUTPUT_DIR = Path("eval_outputs")
EVAL_OUTPUT_DIR.mkdir(exist_ok=True)

OUT_LABELS = EVAL_OUTPUT_DIR / "pooled_gt_5variants_labels.csv"
OUT_DETAIL = EVAL_OUTPUT_DIR / "pooled_gt_5variants_detail.csv"
OUT_WIDE = EVAL_OUTPUT_DIR / "pooled_gt_5variants_wide.csv"
OUT_SUMMARY = EVAL_OUTPUT_DIR / "pooled_gt_5variants_summary.csv"
OUT_PROGRESS = EVAL_OUTPUT_DIR / "pooled_gt_5variants_query_progress.csv"

VARIANTS = [
    "vector_only",
    "graph_only",
    "text_unit_only",
    "hybrid_no_ce",
    "hybrid_with_ce",
]
VARIANT_FUNCS = {
    "vector_only": retrieve_vector_only,
    "graph_only": graph_candidate_top5_only,
    "text_unit_only": retrieve_text_unit_top5_unique,
    "hybrid_no_ce": retrieve_hybrid_no_ce,
    "hybrid_with_ce": retrieve_hybrid_with_ce,
}
GRAPH_RAG_GEO_TEST_QUERIES = [
    {"test_id": "T001", "query": "Mình muốn ăn cơm gà ngon gần đây, rating ổn và có nhiều comment thật.", "user_lat": 21.005118, "user_lng": 105.845592},
    {"test_id": "T002", "query": "Tìm quán phở bò ăn sáng giá sinh viên dưới 50k càng tốt.", "user_lat": 21.00554, "user_lng": 105.84330},
    {"test_id": "T003", "query": "Có quán bún chả nào gần tôi không, ưu tiên quán được khen ngon trong comment.", "user_lat": 21.00452, "user_lng": 105.84320},
    {"test_id": "T004", "query": "Mình muốn ăn bánh cuốn nóng, quán nào sạch sẽ và giá vừa phải?", "user_lat": 21.00375, "user_lng": 105.84640},
    {"test_id": "T005", "query": "Gợi ý gà rán hoặc cơm gà cho bữa tối, giá không quá 100k, giao nhanh thì tốt.", "user_lat": 21.00491, "user_lng": 105.84820},

    {"test_id": "T006", "query": "Quán cơm văn phòng nào phù hợp ăn trưa, phần ăn đầy đặn và không quá đắt?", "user_lat": 21.00546, "user_lng": 105.84370},
    {"test_id": "T007", "query": "Tôi muốn tìm phở gà gần đây, ưu tiên quán có rating từ 4.5 trở lên.", "user_lat": 21.00697, "user_lng": 105.84590},
    {"test_id": "T008", "query": "Có quán cơm rang hoặc phở xào nào ăn tối ổn không?", "user_lat": 21.00816, "user_lng": 105.84610},
    {"test_id": "T009", "query": "Gợi ý quán bún hoặc phở giá rẻ, nhiều review thật, phù hợp sinh viên.", "user_lat": 21.00574, "user_lng": 105.84420},
    {"test_id": "T010", "query": "Mình cần quán ăn gần nhất, món gì cũng được miễn rating ổn.", "user_lat": 21.00539, "user_lng": 105.84340},

    {"test_id": "T011", "query": "Tìm quán cơm tấm hoặc cơm gà có comment khen ngon và giá không quá đắt.", "user_lat": 21.00120, "user_lng": 105.85200},
    {"test_id": "T012", "query": "Quán nào có món phở hoặc bún, phù hợp ăn tối một mình?", "user_lat": 20.99970, "user_lng": 105.84820},
    {"test_id": "T013", "query": "Muốn đặt cơm trưa, ưu tiên quán giao nhanh và có nhiều comment tích cực.", "user_lat": 21.00660, "user_lng": 105.84510},
    {"test_id": "T014", "query": "Gợi ý quán bún chả giá khoảng 50-100k, không quá xa.", "user_lat": 21.00620, "user_lng": 105.83150},
    {"test_id": "T015", "query": "Tôi muốn ăn phở bò, quán nào có nhiều review và rating tốt?", "user_lat": 21.01320, "user_lng": 105.83590},

    {"test_id": "T016", "query": "Có quán cơm nào phù hợp sinh viên, giá dưới 50k không?", "user_lat": 21.01020, "user_lng": 105.83980},
    {"test_id": "T017", "query": "Gợi ý quán gà rán đang mở cửa buổi tối, rating ổn.", "user_lat": 21.01080, "user_lng": 105.83170},
    {"test_id": "T018", "query": "Tìm quán bánh cuốn, quán nào sạch sẽ và comment ổn?", "user_lat": 21.00580, "user_lng": 105.84420},
    {"test_id": "T019", "query": "Mình muốn tìm quán cơm hoặc phở có giá hợp lý trong bán kính vài km.", "user_lat": 21.01650, "user_lng": 105.84360},
    {"test_id": "T020", "query": "Quán nào có món bún hoặc phở, nhiều đánh giá, phù hợp ăn trưa?", "user_lat": 21.01560, "user_lng": 105.85530},

    {"test_id": "T021", "query": "Tôi muốn tránh quán bị chê vệ sinh, tìm quán cơm gà có comment tốt.", "user_lat": 21.005118, "user_lng": 105.845592},
    {"test_id": "T022", "query": "Tìm quán giao nhanh, món cơm hoặc bún đều được, rating trên 4.", "user_lat": 21.00343, "user_lng": 105.84710},
    {"test_id": "T023", "query": "Có quán phở nào giá mềm, ăn sáng được không?", "user_lat": 21.00290, "user_lng": 105.83650},
    {"test_id": "T024", "query": "Gợi ý quán cơm rang phần ăn nhiều, phù hợp ăn trưa.", "user_lat": 21.00870, "user_lng": 105.83680},
    {"test_id": "T025", "query": "Mình muốn ăn bún chả, quán nào rating cao và có comment thật?", "user_lat": 21.01390, "user_lng": 105.84740},

    {"test_id": "T026", "query": "Tìm quán cơm giá không quá 70k, đang mở cửa giờ tối.", "user_lat": 21.00940, "user_lng": 105.85640},
    {"test_id": "T027", "query": "Có quán gà nào không quá xa, review ổn không?", "user_lat": 20.99890, "user_lng": 105.85510},
    {"test_id": "T028", "query": "Gợi ý quán phở hoặc cơm giá rẻ và giao nhanh.", "user_lat": 20.99260, "user_lng": 105.86220},
    {"test_id": "T029", "query": "Mình muốn tìm quán bún/phở/cơm có rating tốt quanh khu này.", "user_lat": 20.99490, "user_lng": 105.86720},
    {"test_id": "T030", "query": "Tìm quán cơm hoặc gà rán giá tầm trung, comment không tệ.", "user_lat": 20.98980, "user_lng": 105.85090},

    {"test_id": "T031", "query": "Quán nào có cơm/phở, nhiều món trong menu, hợp đi nhóm nhỏ?", "user_lat": 21.00190, "user_lng": 105.83240},
    {"test_id": "T032", "query": "Tôi cần ăn nhanh, quán cơm hoặc phở nào gần và ổn nhất?", "user_lat": 21.00060, "user_lng": 105.83920},
    {"test_id": "T033", "query": "Gợi ý quán bún chả hoặc bánh cuốn, ưu tiên quán không bị chê đồ nguội.", "user_lat": 21.00375, "user_lng": 105.84640},
    {"test_id": "T034", "query": "Có quán cơm gà nào ăn tối, giá khoảng 50-100k không?", "user_lat": 21.00641, "user_lng": 105.84470},
    {"test_id": "T035", "query": "Tìm quán phở xào hoặc cơm rang, quán nào comment khen nhiều?", "user_lat": 21.00816, "user_lng": 105.84610},

    {"test_id": "T036", "query": "Tôi muốn quán ăn trưa gần đây, ưu tiên rẻ, rating ổn và không quá xa.", "user_lat": 21.005118, "user_lng": 105.845592},
    {"test_id": "T037", "query": "Mình muốn đặt món lúc 22h, còn quán cơm/phở/gà nào mở cửa không?", "user_lat": 21.005118, "user_lng": 105.845592},
    {"test_id": "T038", "query": "Tìm quán có comment nói đóng gói tốt, giao nhanh, món cơm hoặc gà.", "user_lat": 21.00697, "user_lng": 105.84590},
    {"test_id": "T039", "query": "Gợi ý quán có rating cao nhất trong nhóm cơm, phở, bún chả.", "user_lat": 21.005118, "user_lng": 105.845592},
    {"test_id": "T040", "query": "Quán nào có nhiều review nhất, món ăn chính là cơm hoặc bún?", "user_lat": 21.00491, "user_lng": 105.84820},

    {"test_id": "T041", "query": "Tìm quán phở mở cửa sáng sớm, phù hợp ăn sáng.", "user_lat": 21.01160, "user_lng": 105.85020},
    {"test_id": "T042", "query": "Có quán cơm giá sinh viên và có comment về suất ăn nhiều không?", "user_lat": 20.99680, "user_lng": 105.84320},
    {"test_id": "T043", "query": "Mình muốn bún hoặc phở rating tốt, không quá xa.", "user_lat": 21.01940, "user_lng": 105.84980},
    {"test_id": "T044", "query": "Tìm quán bánh cuốn hoặc bún chả đang mở cửa buổi trưa.", "user_lat": 21.00120, "user_lng": 105.85200},
    {"test_id": "T045", "query": "Gợi ý quán cơm giá không quá đắt, có review thật để tin tưởng.", "user_lat": 21.01180, "user_lng": 105.82680},

    {"test_id": "T046", "query": "Mình cần quán gần nhất có món phở bò hoặc phở gà.", "user_lat": 21.00870, "user_lng": 105.83680},
    {"test_id": "T047", "query": "Tìm quán cơm hoặc gà rán có ảnh, rating ổn.", "user_lat": 21.00413, "user_lng": 105.84700},
    {"test_id": "T048", "query": "Quán nào phù hợp đặt cho nhóm 3 người, menu đa dạng, có cơm và bún/phở?", "user_lat": 21.005118, "user_lng": 105.845592},
    {"test_id": "T049", "query": "Tìm quán có comment tốt về phục vụ, món cơm hoặc bánh cuốn.", "user_lat": 21.00637, "user_lng": 105.84470},
    {"test_id": "T050", "query": "Mình muốn quán đáng tin nhất quanh đây: rating cao, nhiều review, có comment thật, món cơm/phở/bún đều được.", "user_lat": 21.005118, "user_lng": 105.845592},
]


# =========================
# 1. SAFE HELPERS
# =========================

def _safe_str(x: Any) -> str:
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x).strip()


def canon_store_key(x: Any) -> str:
    s = _safe_str(x)
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]
    return s


def _safe_float(x: Any, default: Optional[float] = None) -> Optional[float]:
    if x is None:
        return default
    try:
        if pd.isna(x):
            return default
    except Exception:
        pass
    try:
        return float(x)
    except Exception:
        return default


def _safe_int(x: Any, default: int = 0) -> int:
    try:
        if x is None or pd.isna(x):
            return default
    except Exception:
        pass
    try:
        return int(float(x))
    except Exception:
        return default


def _json_dumps(obj: Any) -> str:
    def default(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        return str(o)
    return json.dumps(obj, ensure_ascii=False, default=default)


def _json_loads_safe(x: Any, default=None):
    if default is None:
        default = {}
    if x is None:
        return default
    try:
        if pd.isna(x):
            return default
    except Exception:
        pass
    try:
        return json.loads(str(x))
    except Exception:
        return default


def _split_pipe(x: Any) -> List[str]:
    s = _safe_str(x)
    if not s:
        return []
    return [canon_store_key(v) for v in s.split("|") if canon_store_key(v)]


def _join_pipe(xs: List[Any]) -> str:
    return "|".join([canon_store_key(x) for x in xs if canon_store_key(x)])


def _atomic_write_csv(df: pd.DataFrame, path: Path, retries: int = 5) -> None:
    path = Path(path)

    if df is None or (df.empty and len(df.columns) == 0):
        print(f"⚠️ Skip writing empty no-column CSV: {path}")
        return

    tmp_path = Path(str(path) + ".tmp")
    df.to_csv(tmp_path, index=False, encoding="utf-8-sig")

    last_err = None
    for attempt in range(retries):
        try:
            tmp_path.replace(path)
            return
        except PermissionError as e:
            last_err = e
            print(f"⚠️ CSV locked retry {attempt + 1}/{retries}: {path}")
            time.sleep(1.0)

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    backup_path = path.with_name(f"{path.stem}_{timestamp}{path.suffix}")
    shutil.copyfile(tmp_path, backup_path)

    try:
        tmp_path.unlink(missing_ok=True)
    except Exception:
        pass

    print(f"⚠️ Could not overwrite locked CSV: {path}")
    print(f"✅ Saved backup instead: {backup_path}")
    if last_err:
        print(f"Last error: {last_err}")


def _load_csv_safe(path: Path) -> pd.DataFrame:
    path = Path(path)

    if not path.exists():
        return pd.DataFrame()

    if path.stat().st_size == 0:
        print(f"⚠️ Empty CSV found, deleting: {path}")
        path.unlink(missing_ok=True)
        return pd.DataFrame()

    try:
        df = pd.read_csv(path)
        print(f"✅ Loaded existing CSV: {path} - {len(df)} rows")
        return df
    except pd.errors.EmptyDataError:
        print(f"⚠️ CSV has no columns, deleting: {path}")
        path.unlink(missing_ok=True)
        return pd.DataFrame()
    except Exception as e:
        print(f"⚠️ Could not load {path}: {type(e).__name__}: {e}")
        return pd.DataFrame()


def _get_candidate_id(c: Any) -> str:
    if isinstance(c, dict):
        return canon_store_key(
            c.get("store_key")
            or c.get("restaurant_id")
            or c.get("id")
            or ""
        )
    return canon_store_key(c)


def _get_candidate_name(c: Dict[str, Any]) -> str:
    if not isinstance(c, dict):
        return ""
    return (
        c.get("name")
        or c.get("restaurant_name")
        or c.get("store_name")
        or c.get("title")
        or ""
    )


def _normalize_retrieval_record(c: Any, rank: int, variant: str) -> Dict[str, Any]:
    if not isinstance(c, dict):
        c = {"store_key": canon_store_key(c)}

    evidence = c.get("evidence", [])
    if isinstance(evidence, str):
        evidence = [evidence]
    elif not isinstance(evidence, list):
        evidence = []

    return {
        "variant": variant,
        "rank": rank,
        "store_key": _get_candidate_id(c),
        "name": _get_candidate_name(c),
        "address": c.get("address", ""),
        "district": c.get("district", ""),
        "city": c.get("city", ""),
        "lat": c.get("lat", c.get("latitude", "")),
        "lng": c.get("lng", c.get("longitude", "")),
        "distance_km": c.get("distance_km", c.get("distance", "")),
        "final_score": c.get("final_score", c.get("score", c.get("rrf_score", ""))),
        "ce_score": c.get("ce_score", ""),
        "final_score_before_ce": c.get("final_score_before_ce", ""),
        "rating": c.get("rating", c.get("gmaps_rating", c.get("foody_rating", ""))),
        "price_band": c.get("price_band", ""),
        "dish_families": c.get("dish_families", []),
        "categories": c.get("categories", []),
        "source_flags": c.get("source_flags", []),
        "community_id": c.get("community_id", ""),
        "community_report": c.get("community_report", ""),
        "evidence": evidence,
        "evidence_count": len([x for x in evidence if str(x).strip()]),
    }


def _dedupe_keep_order(records: List[Dict[str, Any]], key: str = "store_key") -> List[Dict[str, Any]]:
    seen = set()
    out = []

    for r in records or []:
        k = canon_store_key(r.get(key))
        if not k or k in seen:
            continue
        seen.add(k)
        out.append(r)

    return out


# =========================
# 2. RETRIEVAL WRAPPER
# =========================

def retrieve_variant_records(
    variant: str,
    query: str,
    top_k: int,
    user_lat: Optional[float] = None,
    user_lng: Optional[float] = None,
) -> List[Dict[str, Any]]:
    """
    Trả về list record cho một variant.
    Không dedupe giữa các variant.
    Chỉ lấy top_k của chính variant đó.
    """
    if "VARIANT_DETAIL_FUNCS" in globals() and isinstance(VARIANT_DETAIL_FUNCS, dict) and variant in VARIANT_DETAIL_FUNCS:
        fn = VARIANT_DETAIL_FUNCS[variant]
        try:
            rows = fn(query, top_k=top_k, user_lat=user_lat, user_lng=user_lng)
        except TypeError:
            rows = fn(query, top_k=top_k)
    else:
        fn = VARIANT_FUNCS[variant]
        try:
            rows = fn(query, top_k=top_k, user_lat=user_lat, user_lng=user_lng)
        except TypeError:
            rows = fn(query, top_k=top_k)

    rows = rows or []

    normalized = []
    for i, item in enumerate(rows[:top_k], start=1):
        normalized.append(_normalize_retrieval_record(item, rank=i, variant=variant))

    # Trong từng phương diện, bỏ trùng store_key nếu hàm trả trùng chunk/text_unit.
    # Không ảnh hưởng chuyện trùng giữa các phương diện.
    normalized = _dedupe_keep_order(normalized, key="store_key")[:top_k]

    # Rank lại sau dedupe
    for i, r in enumerate(normalized, start=1):
        r["rank"] = i

    return normalized


def retrieve_all_variants_for_query(qrow: Dict[str, Any]) -> Dict[str, List[Dict[str, Any]]]:
    query = qrow["query"]
    user_lat = qrow.get("user_lat")
    user_lng = qrow.get("user_lng")

    out = {}

    for variant in VARIANTS:
        try:
            out[variant] = retrieve_variant_records(
                variant=variant,
                query=query,
                top_k=POOL_TOP_K_PER_VARIANT,
                user_lat=user_lat,
                user_lng=user_lng,
            )
        except Exception as e:
            print(f"⚠️ Retrieval failed: test_id={qrow.get('test_id')}, variant={variant}, error={type(e).__name__}: {e}")
            out[variant] = []

    return out


# =========================
# 3. JUDGE / GROUND TRUTH
# =========================

def _label_key(test_id: str, query: str, store_key: str) -> str:
    return f"{test_id}::{canon_store_key(store_key)}"


def _existing_label_map(labels_df: pd.DataFrame) -> Dict[str, Dict[str, Any]]:
    if labels_df is None or labels_df.empty:
        return {}

    required = {"test_id", "query", "store_key"}
    if not required.issubset(set(labels_df.columns)):
        return {}

    mp = {}
    for _, r in labels_df.iterrows():
        test_id = _safe_str(r.get("test_id"))
        query = _safe_str(r.get("query"))
        sk = canon_store_key(r.get("store_key"))
        if not test_id or not sk:
            continue
        mp[_label_key(test_id, query, sk)] = r.to_dict()

    return mp


def judge_one_candidate(
    test_id: str,
    query: str,
    store_key: str,
    pool_row: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Chấm 1 candidate.
    Nếu JUDGE_MODE='manual' thì grade để trống cho bạn điền tay.
    """
    if JUDGE_MODE == "manual":
        return {
            "relevance_grade": "",
            "evidence_support": "",
            "matched_facets": [],
            "missing_or_weak_facets": [],
            "reasoning_short": "",
            "confidence": "",
            "judge_error": "",
        }

    try:
        payload = build_judge_payload(
            query=query,
            store_key=store_key,
            pool_row=pool_row,
        )
        label = judge_with_llm(payload)

        return {
            "relevance_grade": _safe_int(label.get("relevance_grade"), 0),
            "evidence_support": _safe_int(label.get("evidence_support"), 0),
            "matched_facets": label.get("matched_facets", []),
            "missing_or_weak_facets": label.get("missing_or_weak_facets", []),
            "reasoning_short": label.get("reasoning_short", label.get("reasoning", "")),
            "confidence": label.get("confidence", ""),
            "judge_error": "",
        }

    except Exception as e:
        return {
            "relevance_grade": 0,
            "evidence_support": 0,
            "matched_facets": [],
            "missing_or_weak_facets": [],
            "reasoning_short": "",
            "confidence": "",
            "judge_error": f"{type(e).__name__}: {e}",
        }


def build_ground_truth_topk_for_query(
    qrow: Dict[str, Any],
    variant_records: Dict[str, List[Dict[str, Any]]],
    existing_labels: Dict[str, Dict[str, Any]],
    labels_rows_accum: List[Dict[str, Any]],
) -> Tuple[List[str], List[Dict[str, Any]]]:
    """
    Pool candidate từ 5 phương diện, dedupe store_key để judge 1 lần.
    Sau đó sort theo relevance_grade desc để chọn ground_truth_topk.
    """
    test_id = _safe_str(qrow["test_id"])
    query = _safe_str(qrow["query"])
    user_lat = qrow.get("user_lat")
    user_lng = qrow.get("user_lng")

    # Pool all candidates from all variants
    pooled_by_store = {}

    for variant, rows in variant_records.items():
        for r in rows:
            sk = canon_store_key(r.get("store_key"))
            if not sk:
                continue

            if sk not in pooled_by_store:
                pooled_by_store[sk] = {
                    "test_id": test_id,
                    "query": query,
                    "user_lat": user_lat,
                    "user_lng": user_lng,
                    "store_key": sk,
                    "name": r.get("name", ""),
                    "address": r.get("address", ""),
                    "rating": r.get("rating", ""),
                    "distance_km": r.get("distance_km", ""),
                    "candidate_sources": [],
                    "best_rank": 999999,
                    "source_ranks": {},
                    "raw_candidate_json": r,
                }

            pooled_by_store[sk]["candidate_sources"].append(variant)
            pooled_by_store[sk]["source_ranks"][variant] = int(r.get("rank") or 999999)
            pooled_by_store[sk]["best_rank"] = min(
                pooled_by_store[sk]["best_rank"],
                int(r.get("rank") or 999999),
            )

    pooled = list(pooled_by_store.values())

    judged_rows = []

    for pool_row in pooled:
        sk = canon_store_key(pool_row["store_key"])
        key = _label_key(test_id, query, sk)

        use_existing = (not FORCE_REJUDGE) and (key in existing_labels)

        if use_existing:
            old = existing_labels[key]
            judged = {
                **pool_row,
                "relevance_grade": _safe_int(old.get("relevance_grade"), 0),
                "evidence_support": _safe_int(old.get("evidence_support"), 0),
                "matched_facets": _json_loads_safe(old.get("matched_facets_json"), []),
                "missing_or_weak_facets": _json_loads_safe(old.get("missing_or_weak_facets_json"), []),
                "reasoning_short": old.get("reasoning_short", ""),
                "confidence": old.get("confidence", ""),
                "judge_error": old.get("judge_error", ""),
                "label_source": "cache_csv",
            }
        else:
            label = judge_one_candidate(
                test_id=test_id,
                query=query,
                store_key=sk,
                pool_row=pool_row,
            )

            judged = {
                **pool_row,
                **label,
                "label_source": JUDGE_MODE,
            }

        judged["candidate_sources"] = sorted(set(judged.get("candidate_sources", [])))
        judged["candidate_sources_str"] = "|".join(judged["candidate_sources"])
        judged["source_ranks_json"] = _json_dumps(judged.get("source_ranks", {}))
        judged["matched_facets_json"] = _json_dumps(judged.get("matched_facets", []))
        judged["missing_or_weak_facets_json"] = _json_dumps(judged.get("missing_or_weak_facets", []))
        judged["raw_candidate_json"] = _json_dumps(judged.get("raw_candidate_json", {}))

        judged_rows.append(judged)

        # append vào accum nếu là label mới hoặc force rejudge
        if (not use_existing) or FORCE_REJUDGE:
            labels_rows_accum.append({
                "test_id": judged["test_id"],
                "query": judged["query"],
                "user_lat": judged["user_lat"],
                "user_lng": judged["user_lng"],
                "store_key": judged["store_key"],
                "name": judged["name"],
                "address": judged["address"],
                "rating": judged["rating"],
                "distance_km": judged["distance_km"],
                "candidate_sources_str": judged["candidate_sources_str"],
                "source_ranks_json": judged["source_ranks_json"],
                "best_rank": judged["best_rank"],
                "relevance_grade": judged["relevance_grade"],
                "evidence_support": judged["evidence_support"],
                "matched_facets_json": judged["matched_facets_json"],
                "missing_or_weak_facets_json": judged["missing_or_weak_facets_json"],
                "reasoning_short": judged["reasoning_short"],
                "confidence": judged["confidence"],
                "judge_error": judged["judge_error"],
                "label_source": judged["label_source"],
                "raw_candidate_json": judged["raw_candidate_json"],
                "judged_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            })

    # chọn ground truth top K:
    # ưu tiên relevance_grade cao hơn, evidence_support cao hơn,
    # candidate xuất hiện rank cao hơn trong retrieval.
    judged_sorted = sorted(
        judged_rows,
        key=lambda r: (
            -_safe_int(r.get("relevance_grade"), 0),
            -_safe_int(r.get("evidence_support"), 0),
            _safe_int(r.get("best_rank"), 999999),
            _safe_float(r.get("distance_km"), 999999.0) if _safe_float(r.get("distance_km"), None) is not None else 999999.0,
            canon_store_key(r.get("store_key")),
        )
    )

    ground_truth_rows = judged_sorted[:TOP_K_EVAL]
    ground_truth_topk = [canon_store_key(r.get("store_key")) for r in ground_truth_rows if canon_store_key(r.get("store_key"))]

    return ground_truth_topk, judged_rows


# =========================
# 4. METRICS
# =========================

def precision_at_k(retrieved: List[str], relevant: set, k: int) -> float:
    ret = retrieved[:k]
    if k <= 0:
        return 0.0
    return sum(1 for x in ret if x in relevant) / k


def recall_at_k(retrieved: List[str], relevant: set, k: int) -> float:
    if not relevant:
        return 0.0
    ret = retrieved[:k]
    return sum(1 for x in ret if x in relevant) / len(relevant)


def mrr_at_k(retrieved: List[str], relevant: set, k: int) -> float:
    for i, x in enumerate(retrieved[:k], start=1):
        if x in relevant:
            return 1.0 / i
    return 0.0


def ndcg_binary_at_k(retrieved: List[str], relevant: set, k: int) -> float:
    dcg = 0.0
    for i, x in enumerate(retrieved[:k]):
        if x in relevant:
            dcg += 1.0 / math.log2(i + 2)

    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal_hits))

    return dcg / idcg if idcg > 0 else 0.0


def graded_ndcg_at_k(retrieved: List[str], grade_map: Dict[str, int], k: int) -> float:
    dcg = 0.0
    for i, x in enumerate(retrieved[:k]):
        rel = max(0, int(grade_map.get(x, 0)))
        gain = (2 ** rel) - 1
        dcg += gain / math.log2(i + 2)

    ideal_grades = sorted([max(0, int(v)) for v in grade_map.values()], reverse=True)[:k]
    idcg = sum(((2 ** rel) - 1) / math.log2(i + 2) for i, rel in enumerate(ideal_grades))

    return dcg / idcg if idcg > 0 else 0.0


# =========================
# 5. LOAD EXISTING CHECKPOINTS
# =========================

queries_to_run = GRAPH_RAG_GEO_TEST_QUERIES
if SMOKE_TEST_N is not None:
    queries_to_run = GRAPH_RAG_GEO_TEST_QUERIES[:int(SMOKE_TEST_N)]

labels_df_old = _load_csv_safe(OUT_LABELS)
detail_df_old = _load_csv_safe(OUT_DETAIL)
wide_df_old = _load_csv_safe(OUT_WIDE)

existing_labels = _existing_label_map(labels_df_old)

completed_test_ids = set()
if not wide_df_old.empty and "test_id" in wide_df_old.columns:
    completed_test_ids = set(wide_df_old["test_id"].astype(str).tolist())

print(f"Existing judged labels: {len(existing_labels)}")
print(f"Existing completed queries: {len(completed_test_ids)}")
print(f"Total queries configured: {len(queries_to_run)}")


# =========================
# 6. RUN EVALUATION WITH RESUME
# =========================

all_label_rows = []
if not labels_df_old.empty:
    all_label_rows = labels_df_old.to_dict("records")

detail_rows = []
if not detail_df_old.empty:
    detail_rows = detail_df_old.to_dict("records")

wide_rows = []
if not wide_df_old.empty:
    wide_rows = wide_df_old.to_dict("records")

progress_rows = _load_csv_safe(OUT_PROGRESS).to_dict("records") if OUT_PROGRESS.exists() else []

try:
    for idx, qrow in enumerate(queries_to_run, start=1):
        test_id = _safe_str(qrow["test_id"])
        query = _safe_str(qrow["query"])
        user_lat = qrow.get("user_lat")
        user_lng = qrow.get("user_lng")

        if test_id in completed_test_ids and not FORCE_REJUDGE:
            print(f"[{idx:02d}/{len(queries_to_run)}] {test_id} - SKIP existing completed query")
            continue

        print(f"\n[{idx:02d}/{len(queries_to_run)}] {test_id} - {query[:100]}")

        t0 = time.time()
        query_error = ""
        query_error_trace = ""

        # Xóa record cũ của test_id nếu chạy lại
        detail_rows = [r for r in detail_rows if _safe_str(r.get("test_id")) != test_id]
        wide_rows = [r for r in wide_rows if _safe_str(r.get("test_id")) != test_id]

        try:
            # 1. Retrieve 5 variants
            variant_records = retrieve_all_variants_for_query(qrow)

            # 2. Judge pooled candidates + build ground_truth_topk
            new_label_rows_for_this_query = []

            ground_truth_topk, judged_rows = build_ground_truth_topk_for_query(
                qrow=qrow,
                variant_records=variant_records,
                existing_labels=existing_labels,
                labels_rows_accum=new_label_rows_for_this_query,
            )

            # update label storage
            if new_label_rows_for_this_query:
                # Nếu FORCE_REJUDGE thì bỏ labels cũ của test_id
                if FORCE_REJUDGE:
                    all_label_rows = [r for r in all_label_rows if _safe_str(r.get("test_id")) != test_id]

                all_label_rows.extend(new_label_rows_for_this_query)

                labels_df_now = pd.DataFrame(all_label_rows)
                _atomic_write_csv(labels_df_now, OUT_LABELS)

                # Refresh existing label map để query sau dùng được
                existing_labels = _existing_label_map(labels_df_now)

            # grade map từ judged_rows
            grade_map = {
                canon_store_key(r.get("store_key")): _safe_int(r.get("relevance_grade"), 0)
                for r in judged_rows
                if canon_store_key(r.get("store_key"))
            }

            ground_truth_set = set(ground_truth_topk)
            ground_truth_str = _join_pipe(ground_truth_topk)

            # optional readable ground truth with grade
            gt_readable_parts = []
            judged_by_sk = {canon_store_key(r.get("store_key")): r for r in judged_rows}
            for sk in ground_truth_topk:
                jr = judged_by_sk.get(sk, {})
                gt_readable_parts.append(
                    f"{sk}:{_safe_int(jr.get('relevance_grade'), 0)}:{jr.get('name', '')}"
                )
            ground_truth_readable = " | ".join(gt_readable_parts)

            # 3. Build detail rows per variant
            for variant in VARIANTS:
                records = variant_records.get(variant, [])
                retrieved_topk = [canon_store_key(r.get("store_key")) for r in records if canon_store_key(r.get("store_key"))][:TOP_K_EVAL]

                hits_topk = [sk for sk in retrieved_topk if sk in ground_truth_set]
                hit_top = hits_topk[0] if hits_topk else ""

                row = {
                    "test_id": test_id,
                    "query": query,
                    "user_lat": user_lat,
                    "user_lng": user_lng,
                    "variant": variant,

                    # cột chính bạn cần
                    "retrieved_topk": _join_pipe(retrieved_topk),
                    "ground_truth_topk": ground_truth_str,
                    "ground_truth_readable": ground_truth_readable,
                    "hits_topk": _join_pipe(hits_topk),
                    "hit_top": hit_top,

                    # metric binary theo ground_truth_topk
                    "precision@5": precision_at_k(retrieved_topk, ground_truth_set, TOP_K_EVAL),
                    "recall@5": recall_at_k(retrieved_topk, ground_truth_set, TOP_K_EVAL),
                    "mrr@5": mrr_at_k(retrieved_topk, ground_truth_set, TOP_K_EVAL),
                    "ndcg@5": ndcg_binary_at_k(retrieved_topk, ground_truth_set, TOP_K_EVAL),

                    # metric graded theo relevance_grade 0-3
                    "graded_ndcg@5": graded_ndcg_at_k(retrieved_topk, grade_map, TOP_K_EVAL),

                    "num_retrieved": len(retrieved_topk),
                    "num_relevant": len(ground_truth_topk),
                    "num_hits": len(hits_topk),

                    # debug
                    "retrieved_detail_json": _json_dumps(records),
                    "judged_candidates_json": _json_dumps(judged_rows),
                    "latency_ms": round((time.time() - t0) * 1000, 2),
                    "error": "",
                    "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                }

                detail_rows.append(row)

            # 4. Build one wide row per query
            wide_row = {
                "test_id": test_id,
                "query": query,
                "user_lat": user_lat,
                "user_lng": user_lng,
                "ground_truth_topk": ground_truth_str,
                "ground_truth_readable": ground_truth_readable,
                "num_ground_truth": len(ground_truth_topk),
                "num_judged_candidates": len(judged_rows),
                "latency_ms": round((time.time() - t0) * 1000, 2),
                "error": "",
                "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            }

            for variant in VARIANTS:
                records = variant_records.get(variant, [])
                retrieved_topk = [canon_store_key(r.get("store_key")) for r in records if canon_store_key(r.get("store_key"))][:TOP_K_EVAL]
                hits_topk = [sk for sk in retrieved_topk if sk in ground_truth_set]

                wide_row[f"{variant}_retrieved_topk"] = _join_pipe(retrieved_topk)
                wide_row[f"{variant}_hits_topk"] = _join_pipe(hits_topk)
                wide_row[f"{variant}_hit_top"] = hits_topk[0] if hits_topk else ""
                wide_row[f"{variant}_precision@5"] = precision_at_k(retrieved_topk, ground_truth_set, TOP_K_EVAL)
                wide_row[f"{variant}_recall@5"] = recall_at_k(retrieved_topk, ground_truth_set, TOP_K_EVAL)
                wide_row[f"{variant}_mrr@5"] = mrr_at_k(retrieved_topk, ground_truth_set, TOP_K_EVAL)
                wide_row[f"{variant}_ndcg@5"] = ndcg_binary_at_k(retrieved_topk, ground_truth_set, TOP_K_EVAL)
                wide_row[f"{variant}_graded_ndcg@5"] = graded_ndcg_at_k(retrieved_topk, grade_map, TOP_K_EVAL)
                wide_row[f"{variant}_num_retrieved"] = len(retrieved_topk)
                wide_row[f"{variant}_num_hits"] = len(hits_topk)

            wide_rows.append(wide_row)

            completed_test_ids.add(test_id)

            progress_rows.append({
                "test_id": test_id,
                "query": query,
                "status": "success",
                "num_judged_candidates": len(judged_rows),
                "ground_truth_topk": ground_truth_str,
                "latency_ms": round((time.time() - t0) * 1000, 2),
                "error": "",
                "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            })

        except Exception as e:
            query_error = f"{type(e).__name__}: {e}"
            query_error_trace = traceback.format_exc(limit=5)
            print(f"⚠️ Query failed: {test_id}: {query_error}")

            progress_rows.append({
                "test_id": test_id,
                "query": query,
                "status": "error",
                "num_judged_candidates": 0,
                "ground_truth_topk": "",
                "latency_ms": round((time.time() - t0) * 1000, 2),
                "error": query_error,
                "error_trace": query_error_trace,
                "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            })

        # 5. Checkpoint sau mỗi query
        detail_df_now = pd.DataFrame(detail_rows)
        wide_df_now = pd.DataFrame(wide_rows)
        progress_df_now = pd.DataFrame(progress_rows)

        _atomic_write_csv(detail_df_now, OUT_DETAIL)
        _atomic_write_csv(wide_df_now, OUT_WIDE)
        _atomic_write_csv(progress_df_now, OUT_PROGRESS)

        print(f"✅ Checkpoint saved: {test_id}")

except KeyboardInterrupt:
    print("⚠️ Interrupted. Existing query checkpoints already saved.")
    raise


# =========================
# 7. FINAL SUMMARY
# =========================

detail_df = _load_csv_safe(OUT_DETAIL)
wide_df = _load_csv_safe(OUT_WIDE)
labels_df = _load_csv_safe(OUT_LABELS)

summary_rows = []

if not detail_df.empty:
    for variant in VARIANTS:
        sub = detail_df[detail_df["variant"] == variant].copy()

        if sub.empty:
            summary_rows.append({
                "variant": variant,
                "num_queries": 0,
                "mean_precision@5": None,
                "mean_recall@5": None,
                "mean_mrr@5": None,
                "mean_ndcg@5": None,
                "mean_graded_ndcg@5": None,
                "mean_num_retrieved": None,
                "mean_num_hits": None,
            })
            continue

        summary_rows.append({
            "variant": variant,
            "num_queries": len(sub),
            "mean_precision@5": round(float(pd.to_numeric(sub["precision@5"], errors="coerce").mean()), 6),
            "mean_recall@5": round(float(pd.to_numeric(sub["recall@5"], errors="coerce").mean()), 6),
            "mean_mrr@5": round(float(pd.to_numeric(sub["mrr@5"], errors="coerce").mean()), 6),
            "mean_ndcg@5": round(float(pd.to_numeric(sub["ndcg@5"], errors="coerce").mean()), 6),
            "mean_graded_ndcg@5": round(float(pd.to_numeric(sub["graded_ndcg@5"], errors="coerce").mean()), 6),
            "mean_num_retrieved": round(float(pd.to_numeric(sub["num_retrieved"], errors="coerce").mean()), 6),
            "mean_num_hits": round(float(pd.to_numeric(sub["num_hits"], errors="coerce").mean()), 6),
        })

summary_df = pd.DataFrame(summary_rows)
_atomic_write_csv(summary_df, OUT_SUMMARY)


# =========================
# 8. DISPLAY
# =========================

print("\n✅ DONE - POOLED GROUND TRUTH EVALUATION")
print(f"Labels CSV : {OUT_LABELS} - {len(labels_df)} rows")
print(f"Detail CSV : {OUT_DETAIL} - {len(detail_df)} rows")
print(f"Wide CSV   : {OUT_WIDE} - {len(wide_df)} rows")
print(f"Summary CSV: {OUT_SUMMARY} - {len(summary_df)} rows")

display_cols = [
    "test_id",
    "query",
    "variant",
    "retrieved_topk",
    "ground_truth_topk",
    "hits_topk",
    "hit_top",
    "precision@5",
    "recall@5",
    "mrr@5",
    "ndcg@5",
    "graded_ndcg@5",
    "num_retrieved",
    "num_relevant",
    "num_hits",
    "error",
]

display_cols = [c for c in display_cols if c in detail_df.columns]

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 160)

print("\n===== DETAIL RESULT =====")
display(detail_df[display_cols].head(30))

print("\n===== WIDE RESULT =====")
display(wide_df.head(10))

print("\n===== SUMMARY =====")
display(summary_df)

Existing judged labels: 0
Existing completed queries: 0
Total queries configured: 50

[01/50] T001 - Mình muốn ăn cơm gà ngon gần đây, rating ổn và có nhiều comment thật.


C:\Users\FPT SHOP\AppData\Local\Temp\ipykernel_21928\1306086504.py:145: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(x):
C:\Users\FPT SHOP\AppData\Local\Temp\ipykernel_21928\1306086504.py:145: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(x):
C:\Users\FPT SHOP\AppData\Local\Temp\ipykernel_21928\1306086504.py:145: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(x):
C:\Users\FPT SHOP\AppData\Local\Temp\ipykernel_21928\1306086504.py:145: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future t

✅ Checkpoint saved: T001

[02/50] T002 - Tìm quán phở bò ăn sáng giá sinh viên dưới 50k càng tốt.


C:\Users\FPT SHOP\AppData\Local\Temp\ipykernel_21928\1306086504.py:145: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(x):
C:\Users\FPT SHOP\AppData\Local\Temp\ipykernel_21928\1306086504.py:145: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(x):
C:\Users\FPT SHOP\AppData\Local\Temp\ipykernel_21928\1306086504.py:145: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if pd.isna(x):
C:\Users\FPT SHOP\AppData\Local\Temp\ipykernel_21928\1306086504.py:145: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future t

✅ Checkpoint saved: T002

[03/50] T003 - Có quán bún chả nào gần tôi không, ưu tiên quán được khen ngon trong comment.


...

,test_id,query,variant,retrieved_topk,ground_truth_topk,hits_topk,hit_top,precision@5,recall@5,mrr@5,ndcg@5,graded_ndcg@5,num_retrieved,num_relevant,num_hits,error
0,T001,"Mình muốn ăn cơm gà ngon gần đây, rating ổn và có nhiều comment thật.",vector_only,55835|1255|10307|115272|34875,10307|123151|128372|68175|117708,10307,10307.0,0.2,0.2,0.333333,0.169580,0.482423,5,5,1,NaN
1,T001,"Mình muốn ăn cơm gà ngon gần đây, rating ổn và có nhiều comment thật.",graph_only,117708|53514|117760|117526|136086,10307|123151|128372|68175|117708,117708,117708.0,0.2,0.2,1.000000,0.339160,0.463308,5,5,1,NaN
2,T001,"Mình muốn ăn cơm gà ngon gần đây, rating ổn và có nhiều comment thật.",text_unit_only,10307|73467|55835|123151|68175,10307|123151|128372|68175|117708,10307|123151|68175,10307.0,0.6,0.6,1.000000,0.616434,0.778011,5,5,3,NaN
3,T001,"Mình muốn ăn cơm gà ngon gần đây, rating ổn và có nhiều comment thật.",hybrid_no_ce,123151|10307|55835|128372|68175,10307|123151|128372|68175|117708,123151|10307|128372|68175,123151.0,0.8,0.8,1.000000,0.830420,0.976294,5,5,4,NaN
4,T001,"Mình muốn ăn cơm gà ngon gần đây, rating ổn và có nhiều comment thật.",hybrid_with_ce,55835|116044|123151|10307|88782,10307|123151|128372|68175|117708,123151|10307,123151.0,0.4,0.4,0.333333,0.315648,0.658297,5,5,2,NaN
5,T002,Tìm quán phở bò ăn sáng giá sinh viên dưới 50k càng tốt.,vector_only,130307|21224|13942|41433|49400,13942|80920|2282|130307|98870,130307|13942,130307.0,0.4,0.4,1.000000,0.508740,0.579862,5,5,2,NaN
6,T002,Tìm quán phở bò ăn sáng giá sinh viên dưới 50k càng tốt.,graph_only,98870|18796|41433|111742|111500,13942|80920|2282|130307|98870,98870,98870.0,0.2,0.2,1.000000,0.339160,0.464715,5,5,1,NaN
7,T002,Tìm quán phở bò ăn sáng giá sinh viên dưới 50k càng tốt.,text_unit_only,25452|21413|130307|80920|8200,13942|80920|2282|130307|98870,130307|80920,130307.0,0.4,0.4,0.333333,0.315648,0.493295,5,5,2,NaN
8,T002,Tìm quán phở bò ăn sáng giá sinh viên dưới 50k càng tốt.,hybrid_no_ce,8200|12952|130307|60874|2282,13942|80920|2282|130307|98870,130307|2282,130307.0,0.4,0.4,0.333333,0.300785,0.598349,5,5,2,NaN
9,T002,Tìm quán phở bò ăn sáng giá sinh viên dưới 50k càng tốt.,hybrid_with_ce,8200|12952|59967|8314|130307,13942|80920|2282|130307|98870,130307,130307.0,0.2,0.2,0.200000,0.131205,0.509260,5,5,1,NaN



===== WIDE RESULT =====


,test_id,query,user_lat,user_lng,ground_truth_topk,ground_truth_readable,num_ground_truth,num_judged_candidates,latency_ms,error,saved_at,vector_only_retrieved_topk,vector_only_hits_topk,vector_only_hit_top,vector_only_precision@5,vector_only_recall@5,vector_only_mrr@5,vector_only_ndcg@5,vector_only_graded_ndcg@5,vector_only_num_retrieved,vector_only_num_hits,graph_only_retrieved_topk,graph_only_hits_topk,graph_only_hit_top,graph_only_precision@5,graph_only_recall@5,graph_only_mrr@5,graph_only_ndcg@5,graph_only_graded_ndcg@5,graph_only_num_retrieved,graph_only_num_hits,text_unit_only_retrieved_topk,text_unit_only_hits_topk,text_unit_only_hit_top,text_unit_only_precision@5,text_unit_only_recall@5,text_unit_only_mrr@5,text_unit_only_ndcg@5,text_unit_only_graded_ndcg@5,text_unit_only_num_retrieved,text_unit_only_num_hits,hybrid_no_ce_retrieved_topk,hybrid_no_ce_hits_topk,hybrid_no_ce_hit_top,hybrid_no_ce_precision@5,hybrid_no_ce_recall@5,hybrid_no_ce_mrr@5,hybrid_no_ce_ndcg@5,hybrid_no_ce_graded_ndcg@5,hybrid_no_ce_num_retrieved,hybrid_no_ce_num_hits,hybrid_with_ce_retrieved_topk,hybrid_with_ce_hits_topk,hybrid_with_ce_hit_top,hybrid_with_ce_precision@5,hybrid_with_ce_recall@5,hybrid_with_ce_mrr@5,hybrid_with_ce_ndcg@5,hybrid_with_ce_graded_ndcg@5,hybrid_with_ce_num_retrieved,hybrid_with_ce_num_hits
0,T001,"Mình muốn ăn cơm gà ngon gần đây, rating ổn và có nhiều comment thật.",21.005118,105.845592,10307|123151|128372|68175|117708,"10307:3:Cơm Gà Xối Mắm Còn - Dương Văn Bé | 123151:3:FoodHub - Mỳ Quảng, Cơm Gà Roti - Hoàng Ngọc Phách | 128372:3:Cơm BomPi - Gà & Xá Xíu - Trường Chinh | ...",5,16,217333.30,NaN,2026-05-29 11:32:58,55835|1255|10307|115272|34875,10307,10307.0,0.2,0.2,0.333333,0.169580,0.482423,5,1,117708|53514|117760|117526|136086,117708,117708.0,0.2,0.2,1.00,0.339160,0.463308,5,1,10307|73467|55835|123151|68175,10307|123151|68175,10307.0,0.6,0.6,1.000000,0.616434,0.778011,5,3,123151|10307|55835|128372|68175,123151|10307|128372|68175,123151.0,0.8,0.8,1.000000,0.830420,0.976294,5,4,55835|116044|123151|10307|88782,123151|10307,123151.0,0.4,0.4,0.333333,0.315648,0.658297,5,2
1,T002,Tìm quán phở bò ăn sáng giá sinh viên dưới 50k càng tốt.,21.005540,105.843300,13942|80920|2282|130307|98870,13942:3:Phở Bò & Cơm Rang - Nguyễn Trãi | 80920:3:Cồ Cử Phở Bò - Cơm Rang - Kim Giang | 2282:3:Phở Bò Cơm Rang Nhân Hòa - Nguyễn Huy Tưởng | 130307:2:Phở Bò...,5,18,259502.27,NaN,2026-05-29 11:37:17,130307|21224|13942|41433|49400,130307|13942,130307.0,0.4,0.4,1.000000,0.508740,0.579862,5,2,98870|18796|41433|111742|111500,98870,98870.0,0.2,0.2,1.00,0.339160,0.464715,5,1,25452|21413|130307|80920|8200,130307|80920,130307.0,0.4,0.4,0.333333,0.315648,0.493295,5,2,8200|12952|130307|60874|2282,130307|2282,130307.0,0.4,0.4,0.333333,0.300785,0.598349,5,2,8200|12952|59967|8314|130307,130307,130307.0,0.2,0.2,0.200000,0.131205,0.509260,5,1
2,T003,"Có quán bún chả nào gần tôi không, ưu tiên quán được khen ngon trong comment.",21.004520,105.843200,11020|97742|17700|49400|89400,"11020:3:Bún Chả Cô An - Hanoi Grilled Pork Noodles - Khâm Thiên | 97742:3:NaBi Bún Chả Quạt, Cơm Tấm & Cơm Văn Phòng - Tây Sơn | 17700:3:Bún Chả Xưa - Lĩnh ...",5,14,177509.28,NaN,2026-05-29 11:40:15,40014|11020|76776|25452|7160,11020,11020.0,0.2,0.2,0.500000,0.213986,0.731220,5,1,97742|17700|134353|89400|50258,97742|17700|89400,97742.0,0.6,0.6,1.00,0.699215,0.828123,5,3,17700|49400|89400|61570|18796,17700|49400|89400,17700.0,0.6,0.6,1.000000,0.722727,1.000000,5,3,17700|11020|7160|89400|76776,17700|11020|89400,17700.0,0.6,0.6,1.000000,0.699215,0.903097,5,3,11020|97742|89400|17700|88436,11020|97742|89400|17700,11020.0,0.8,0.8,1.000000,0.868795,1.000000,5,4
3,T004,"Mình muốn ăn bánh cuốn nóng, quán nào sạch sẽ và giá vừa phải?",21.003750,105.846400,49400|20000|20601|78300|88436,"49400:3:Tùng Linh Quán - Bún Chả & Bánh Cuốn Nóng - Bùi Xương Trạch | 20000:3:Bánh Cuốn Nóng - Nguyễn Bỉnh Khiêm | 20601:3:Bánh Cuốn Nóng, Bún Chả & Gà Tần ...",5,15,179680.67,NaN,2026-05-29 11:43:15,1279|3


===== SUMMARY =====


,variant,num_queries,mean_precision@5,mean_recall@5,mean_mrr@5,mean_ndcg@5,mean_graded_ndcg@5,mean_num_retrieved,mean_num_hits
0,vector_only,50,0.308,0.308,0.670667,0.352056,0.640876,5.0,1.54
1,graph_only,50,0.372,0.372,0.805000,0.455753,0.741196,4.8,1.86
2,text_unit_only,50,0.332,0.332,0.682333,0.380912,0.653250,5.0,1.66
3,hybrid_no_ce,50,0.404,0.404,0.746333,0.442259,0.764814,5.0,2.02
4,hybrid_with_ce,50,0.432,0.432,0.789000,0.484171,0.791735,5.0,2.16
